# Forward-scattering suppression sweep: colour and directional-scattering analysis

This notebook analyses all calculated cases in the forward-scattering
suppression sweep.

The objectives are:

1. verify the completeness and numerical consistency of the sweep;
2. quantify colour performance using CIELAB and the normalized
   Rösch-MacAdam metrics;
3. quantify forward-scattering suppression using directional,
   band-integrated, and spectral metrics;
4. identify trade-offs between colour quality and directional scattering;
5. identify Pareto-optimal and application-relevant designs;
6. determine which physical parameters control the observed performance.

No case is excluded silently. Every exclusion or warning is retained in
an auditable table.

Code cell 1: Imports and paths

In [ ]:
# ============================================================
# IMPORTS, STYLE, AND PATHS
# ============================================================

from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

from IPython.display import display

from bragg_onion._notebook_tools.io import load_sweep_outputs
from bragg_onion._notebook_tools.colour import (
    add_display_normalized_colour_columns,
)

# ------------------------------------------------------------
# Matplotlib configuration
# ------------------------------------------------------------

mpl.rcdefaults()

STYLE_PATH = Path(
    "/mnt/c/Users/SchwarzN/"
    "OneDrive - Université de Fribourg/"
    "Institution/Other/Scripts_and_HowTo/"
    "science_paper.mplstyle"
)

if STYLE_PATH.exists():
    plt.style.use(STYLE_PATH)
    print("Loaded plotting style:", STYLE_PATH)
else:
    warnings.warn(
        "The paper plotting style was not found. "
        "Matplotlib defaults will be used."
    )

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600

# ------------------------------------------------------------
# Sweep output directory
# ------------------------------------------------------------

# Native WSL version of:
#
# C:\Users\SchwarzN\OneDrive - Université de Fribourg\
# Institution\P1_BraggSpericalPigments\Simulation\
# MIE_Computation\Scattnlay_BraggOnion\
# outputs\forward_suppression_sweep

OUTPUT_DIR = Path(
    "/mnt/c/Users/SchwarzN/"
    "OneDrive - Université de Fribourg/"
    "Institution/P1_BraggSpericalPigments/"
    "Simulation/MIE_Computation/"
    "Scattnlay_BraggOnion/outputs/"
    "forward_suppression_sweep"
)

FIGURE_DIR = OUTPUT_DIR / "analysis_figures"
TABLE_DIR = OUTPUT_DIR / "analysis_tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PNG_DPI = 600

print("OUTPUT_DIR :", OUTPUT_DIR)
print("Exists     :", OUTPUT_DIR.exists())
print("FIGURE_DIR :", FIGURE_DIR)
print("TABLE_DIR  :", TABLE_DIR)

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        "The forward-suppression sweep directory was not found:\n"
        f"{OUTPUT_DIR}\n\n"
        "Check the spelling and confirm that the OneDrive directory "
        "is accessible from WSL."
    )

Code cell 2: Inspect everything stored in the directory

This cell is intentionally independent of the project loader. It gives us an inventory of all result and metadata files.

In [ ]:
# ============================================================
# INVENTORY OF THE SWEEP OUTPUT DIRECTORY
# ============================================================

def human_readable_size(number_of_bytes):
    """Return a compact human-readable file size."""
    number_of_bytes = float(number_of_bytes)

    for unit in ["B", "kB", "MB", "GB", "TB"]:
        if number_of_bytes < 1024.0 or unit == "TB":
            return f"{number_of_bytes:.2f} {unit}"
        number_of_bytes /= 1024.0


file_records = []

for path in sorted(OUTPUT_DIR.rglob("*")):
    if not path.is_file():
        continue

    relative_path = path.relative_to(OUTPUT_DIR)

    file_records.append(
        {
            "relative_path": str(relative_path),
            "name": path.name,
            "suffix": path.suffix.lower(),
            "parent": str(relative_path.parent),
            "size_bytes": path.stat().st_size,
            "size": human_readable_size(path.stat().st_size),
        }
    )

df_file_inventory = pd.DataFrame(file_records)

if df_file_inventory.empty:
    raise FileNotFoundError(
        f"No files were found in {OUTPUT_DIR}"
    )

print("Number of files:", len(df_file_inventory))
print(
    "Total size:",
    human_readable_size(df_file_inventory["size_bytes"].sum()),
)

print("\nFiles by extension:")
display(
    df_file_inventory
    .groupby("suffix", dropna=False)
    .agg(
        number_of_files=("relative_path", "size"),
        total_bytes=("size_bytes", "sum"),
    )
    .sort_values("total_bytes", ascending=False)
)

print("\nComplete file inventory:")
display(
    df_file_inventory[
        [
            "relative_path",
            "suffix",
            "size",
        ]
    ]
)

Code cell 3: Load the combined sweep

Your previous notebooks pass a Windows path to load_sweep_outputs, but here we use the explicit WSL path to avoid ambiguity

In [ ]:
# ============================================================
# LOAD THE COMPLETE FORWARD-SUPPRESSION SWEEP
# ============================================================

SUMMARY_CANDIDATES = [
    OUTPUT_DIR
    / "forward_suppression_summary.parquet",

    OUTPUT_DIR
    / "forward_suppression_summary.pkl",

    OUTPUT_DIR
    / "forward_suppression_summary.csv",
]

summary_path = next(
    (
        path
        for path in SUMMARY_CANDIDATES
        if path.exists()
    ),
    None,
)

if summary_path is None:
    raise FileNotFoundError(
        "No combined forward-suppression summary was found.\n"
        "Checked:\n"
        + "\n".join(
            f"  - {path}"
            for path in SUMMARY_CANDIDATES
        )
    )


# ------------------------------------------------------------
# Load the preferred available format
# ------------------------------------------------------------

if summary_path.suffix == ".parquet":

    df_all = pd.read_parquet(
        summary_path
    )

    USE_PARQUET = True

elif summary_path.suffix == ".pkl":

    df_all = pd.read_pickle(
        summary_path
    )

    USE_PARQUET = False

elif summary_path.suffix == ".csv":

    df_all = pd.read_csv(
        summary_path
    )

    USE_PARQUET = False

else:
    raise ValueError(
        f"Unsupported summary format: {summary_path}"
    )


OUTPUT_DIR_LOADED = OUTPUT_DIR


# ------------------------------------------------------------
# Validate the loaded result
# ------------------------------------------------------------

if df_all.empty:
    raise ValueError(
        "The loaded forward-suppression summary is empty."
    )

if df_all.columns.duplicated().any():

    duplicated_columns = (
        df_all.columns[
            df_all.columns.duplicated()
        ]
        .tolist()
    )

    raise ValueError(
        "The loaded dataframe contains duplicated columns: "
        f"{duplicated_columns}"
    )


# Preserve an untouched copy.
df_raw = df_all.copy(deep=True)


# ------------------------------------------------------------
# Report the loaded dataset
# ------------------------------------------------------------

print("Loaded forward-suppression sweep")
print("--------------------------------")
print("File              :", summary_path)
print("Format            :", summary_path.suffix)
print("Number of rows    :", len(df_raw))
print("Number of columns :", len(df_raw.columns))
print("Memory usage      :", human_readable_size(
    df_raw.memory_usage(
        index=True,
        deep=True,
    ).sum()
))

print("\nFirst rows:")
display(df_raw.head())

In [ ]:
# ============================================================
# BASIC SWEEP-COMPLETENESS CHECK
# ============================================================

print(
    "Collection directions:",
    sorted(
        df_raw[
            "collection_direction"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if "collection_direction" in df_raw.columns
    else "column unavailable",
)

print(
    "Reference wavelengths:",
    sorted(
        pd.to_numeric(
            df_raw[
                "reference_design_wavelength_nm"
            ],
            errors="coerce",
        )
        .dropna()
        .unique()
        .tolist()
    )
    if (
        "reference_design_wavelength_nm"
        in df_raw.columns
    )
    else "column unavailable",
)

print(
    "Case families:",
    sorted(
        df_raw[
            "case_family"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if "case_family" in df_raw.columns
    else "column unavailable",
)

print(
    "Scenarios:",
    sorted(
        df_raw[
            "scenario"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if "scenario" in df_raw.columns
    else "column unavailable",
)

Code cell 4: Column schema and candidate metric detection

This is the most important first diagnostic. It identifies likely colour, forward, backward, band, spectral, warning, and geometry columns automatically.

In [ ]:
# ============================================================
# AUDIT THE AVAILABLE COLUMNS
#
# Array-safe version:
# Some columns contain wavelength-dependent or angular arrays.
# Such arrays cannot be passed directly to Series.nunique().
# ============================================================

def classify_column(column_name):
    """
    Assign broad scientific categories based on the column name.
    """

    name = str(column_name).lower()
    categories = []

    keyword_groups = {
        "colour": [
            "eta_c",
            "eta_l",
            "eta_y",
            "cielab",
            "hue",
            "chroma",
            "hex",
            "colour",
            "color",
            "xyz",
            "lab",
            "d_opt",
        ],
        "forward": [
            "forward",
            "fwd",
        ],
        "backward": [
            "backward",
            "back",
            "bwd",
        ],
        "directionality": [
            "asymmetry",
            "direction",
            "forward",
            "backward",
            "side",
            "g_",
            "_g",
        ],
        "scattering": [
            "qsca",
            "qext",
            "qabs",
            "qbk",
            "qpr",
            "scatt",
            "albedo",
            "cross_section",
            "collected",
        ],
        "spectral": [
            "wavelength",
            "peak",
            "band",
            "spectrum",
            "spectral",
            "fwhm",
        ],
        "geometry": [
            "diameter",
            "radius",
            "layer",
            "shell",
            "thickness",
            "core",
            "outer",
        ],
        "material": [
            "n_a",
            "n_b",
            "index",
            "medium",
            "extinction",
            "absorber",
            "material",
        ],
        "quality_control": [
            "warning",
            "error",
            "valid",
            "passiv",
            "converg",
            "failed",
            "status",
        ],
    }

    for category, keywords in (
        keyword_groups.items()
    ):
        if any(
            keyword in name
            for keyword in keywords
        ):
            categories.append(category)

    if not categories:
        categories.append("other")

    return ", ".join(categories)


def value_type_name(value):
    """
    Return a concise description of one stored value.
    """

    if isinstance(value, np.ndarray):
        return (
            f"ndarray"
            f"[shape={value.shape}, "
            f"dtype={value.dtype}]"
        )

    if isinstance(value, list):
        return f"list[length={len(value)}]"

    if isinstance(value, tuple):
        return f"tuple[length={len(value)}]"

    if isinstance(value, dict):
        return f"dict[length={len(value)}]"

    return type(value).__name__


def make_hashable_summary(value):
    """
    Convert scalar or array-like values into hashable summaries.

    The conversion is used only for schema inspection. It does not
    modify the original dataframe.
    """

    if isinstance(value, np.ndarray):

        array = np.asarray(value)

        if array.size == 0:
            return (
                "ndarray",
                tuple(array.shape),
                str(array.dtype),
                "empty",
            )

        if np.issubdtype(
            array.dtype,
            np.number,
        ):

            finite = array[
                np.isfinite(array)
            ]

            if finite.size == 0:

                return (
                    "ndarray",
                    tuple(array.shape),
                    str(array.dtype),
                    "no_finite_values",
                )

            return (
                "ndarray",
                tuple(array.shape),
                str(array.dtype),
                float(np.nanmin(array)),
                float(np.nanmax(array)),
                float(np.nanmean(array)),
            )

        return (
            "ndarray",
            tuple(array.shape),
            str(array.dtype),
            repr(array.ravel()[:3].tolist()),
        )

    if isinstance(value, list):

        return (
            "list",
            len(value),
            repr(value[:3]),
        )

    if isinstance(value, tuple):

        return (
            "tuple",
            len(value),
            repr(value[:3]),
        )

    if isinstance(value, dict):

        return (
            "dict",
            tuple(
                sorted(
                    map(str, value.keys())
                )
            ),
        )

    try:
        hash(value)
        return value

    except TypeError:
        return repr(value)


def safe_number_unique(series):
    """
    Count unique scalar values or unique array summaries without
    raising an exception for unhashable types.
    """

    if series.empty:
        return 0

    try:
        return series.nunique(dropna=True)

    except TypeError:

        hashable_summaries = (
            series
            .apply(make_hashable_summary)
            .dropna()
        )

        return hashable_summaries.nunique(dropna=True)

In [ ]:
# ============================================================
# IDENTIFY ARRAY-VALUED COLUMNS
# ============================================================

array_column_records = []

for column in df_raw.columns:

    series = df_raw[column]

    array_values = [
        value
        for value in series
        if isinstance(
            value,
            np.ndarray,
        )
    ]

    if not array_values:
        continue

    shapes = sorted(
        {
            tuple(
                np.asarray(
                    value
                ).shape
            )
            for value in array_values
        }
    )

    dtypes = sorted(
        {
            str(
                np.asarray(
                    value
                ).dtype
            )
            for value in array_values
        }
    )

    lengths = sorted(
        {
            int(
                np.asarray(
                    value
                ).size
            )
            for value in array_values
        }
    )

    array_column_records.append(
        {
            "column": column,
            "number_of_array_rows": (
                len(array_values)
            ),
            "shapes": shapes,
            "dtypes": dtypes,
            "array_sizes": lengths,
        }
    )


df_array_columns = pd.DataFrame(
    array_column_records
)

print(
    "Number of array-valued columns:",
    len(df_array_columns),
)

display(df_array_columns)


df_array_columns.to_csv(
    TABLE_DIR
    / "array_valued_columns.csv",
    index=False,
)

Code cell 5: Print scientifically relevant column groups

In [ ]:
# ============================================================
# SCIENTIFICALLY RELEVANT COLUMNS AND DIRECTION VALUES
# ============================================================

def columns_containing(
    *keywords,
):
    keywords = [
        keyword.lower()
        for keyword in keywords
    ]

    return [
        column
        for column in df_raw.columns
        if any(
            keyword
            in str(column).lower()
            for keyword in keywords
        )
    ]


column_groups = {
    "colour": columns_containing(
        "eta_c",
        "eta_l",
        "eta_y",
        "hue",
        "chroma",
        "hex",
        "colour",
        "color",
        "d_opt",
    ),
    "collection": columns_containing(
        "collected",
        "collection",
        "fraction",
    ),
    "scattering": columns_containing(
        "qsca",
        "qext",
        "qabs",
        "qbk",
        "qpr",
        "albedo",
        "asymmetry",
        "g",
    ),
    "geometry": columns_containing(
        "diameter",
        "radius",
        "layer",
        "shell",
        "thickness",
        "core",
    ),
    "spectral_arrays": columns_containing(
        "wavelength",
        "spectrum",
        "theta",
        "angle",
    ),
    "quality_control": columns_containing(
        "warning",
        "error",
        "valid",
        "passiv",
        "status",
        "failed",
    ),
}


for group_name, columns in (
    column_groups.items()
):

    print(
        f"\n{group_name.upper()}"
    )

    print(
        "-" * len(group_name)
    )

    if columns:

        for column in columns:
            print(column)

    else:

        print(
            "No matching columns found."
        )


# ------------------------------------------------------------
# Direction is stored as categorical values, not column names.
# ------------------------------------------------------------

if "collection_direction" in df_raw.columns:

    print(
        "\nCOLLECTION DIRECTION VALUES"
    )

    print(
        "---------------------------"
    )

    print(
        sorted(
            df_raw[
                "collection_direction"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
    )

    display(
        df_raw[
            "collection_direction"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "collection_direction"
        )
        .reset_index(
            name="number_of_rows"
        )
    )

else:

    print(
        "\nWarning: collection_direction "
        "column was not found."
    )

Code cell 6: Detect parameters and dimensions of the sweep

This helps separate input parameters from result metrics.

In [ ]:
# ============================================================
# DETECT LIKELY SWEEP PARAMETERS
#
# Array-valued columns are excluded because they contain spectra
# or angularly resolved quantities rather than scalar parameters.
# ============================================================

known_result_patterns = [
    "eta_",
    "qsca",
    "qext",
    "qabs",
    "qbk",
    "qpr",
    "collected",
    "forward",
    "backward",
    "albedo",
    "asymmetry",
    "hue",
    "hex",
    "peak",
    "maximum",
    "mean",
    "rms",
    "warning",
    "error",
    "spectrum",
    "wavelengths",
    "theta",
    "angles",
]


def contains_unhashable_values(
    series,
    maximum_values_to_check=50,
):
    """
    Return True if the series contains arrays, lists, dictionaries,
    sets, or other unhashable objects.
    """

    checked = 0

    for value in series:

        if value is None:
            continue

        if (
            isinstance(value, float)
            and np.isnan(value)
        ):
            continue

        if isinstance(
            value,
            (
                np.ndarray,
                list,
                dict,
                set,
            ),
        ):
            return True

        try:
            hash(value)

        except TypeError:
            return True

        checked += 1

        if checked >= maximum_values_to_check:
            break

    return False


def safe_unique_scalar_values(
    series,
):
    """
    Return unique scalar values.

    Array-valued or otherwise unhashable columns return None.
    """

    if contains_unhashable_values(series):
        return None

    try:

        return series.dropna().unique()

    except TypeError:

        return None


likely_parameter_columns = []
excluded_array_columns = []

for column in df_raw.columns:

    name = str(column).lower()
    series = df_raw[column]

    unique_values = (
        safe_unique_scalar_values(
            series
        )
    )

    if unique_values is None:

        excluded_array_columns.append(
            column
        )

        continue

    number_unique = len(
        unique_values
    )

    if number_unique <= 1:
        continue

    if any(
        pattern in name
        for pattern in known_result_patterns
    ):
        continue

    # A parameter usually has considerably fewer unique values
    # than the number of calculated result rows.
    maximum_parameter_values = max(
        100,
        int(
            0.10
            * len(df_raw)
        ),
    )

    if number_unique <= maximum_parameter_values:

        likely_parameter_columns.append(
            column
        )


# ============================================================
# BUILD PARAMETER SUMMARY
# ============================================================

parameter_summary_records = []

for column in likely_parameter_columns:

    series = df_raw[column]

    unique_values = (
        safe_unique_scalar_values(
            series
        )
    )

    if unique_values is None:
        continue

    number_unique = len(
        unique_values
    )

    if number_unique <= 20:

        try:

            values_preview = sorted(
                unique_values.tolist()
            )

        except TypeError:

            values_preview = (
                unique_values.tolist()
            )

    else:

        values_preview = (
            unique_values[:20]
            .tolist()
        )

        values_preview.append(
            "..."
        )

    parameter_summary_records.append(
        {
            "parameter": column,
            "dtype": str(
                series.dtype
            ),
            "number_of_values": (
                number_unique
            ),
            "values_preview": (
                values_preview
            ),
        }
    )


df_parameter_summary = pd.DataFrame(
    parameter_summary_records
)


if df_parameter_summary.empty:

    print(
        "No likely scalar sweep parameters "
        "were detected automatically."
    )

else:

    df_parameter_summary = (
        df_parameter_summary
        .sort_values(
            [
                "number_of_values",
                "parameter",
            ]
        )
        .reset_index(drop=True)
    )

    print(
        "Likely scalar sweep parameters:"
    )

    display(
        df_parameter_summary
    )


print(
    "\nArray-valued columns excluded from "
    "parameter detection:"
)

for column in excluded_array_columns:
    print("  ", column)


parameter_summary_path = (
    TABLE_DIR
    / "likely_sweep_parameters.csv"
)

df_parameter_summary.to_csv(
    parameter_summary_path,
    index=False,
)

print(
    "\nSaved:",
    parameter_summary_path,
)

In [ ]:
# ============================================================
# EXPLICIT SUMMARY OF THE FORWARD-SUPPRESSION SWEEP
# ============================================================

expected_parameter_columns = [
    "reference_design_wavelength_nm",
    "case_family",
    "scenario",
    "absorber_location",
    "absorber_k",
    "core_thickness_factor",
    "n_shells",
    "outer_shell_thickness_nm",
    "outer_shell_n",
    "collection_direction",
]

explicit_parameter_records = []

for column in expected_parameter_columns:

    if column not in df_raw.columns:
        continue

    series = df_raw[column]

    unique_values = (
        safe_unique_scalar_values(
            series
        )
    )

    if unique_values is None:

        values = [
            "unhashable values"
        ]

    else:

        try:

            values = sorted(
                unique_values.tolist()
            )

        except TypeError:

            values = (
                unique_values.tolist()
            )

    explicit_parameter_records.append(
        {
            "parameter": column,
            "number_of_values": (
                len(unique_values)
                if unique_values
                is not None
                else np.nan
            ),
            "values": values,
        }
    )


df_explicit_parameter_summary = (
    pd.DataFrame(
        explicit_parameter_records
    )
)

display(
    df_explicit_parameter_summary
)

Code cell 7: Basic duplicate and missing-data audit

In [ ]:
# ============================================================
# ARRAY-SAFE DATA-INTEGRITY AUDIT
# ============================================================

# ------------------------------------------------------------
# Select scalar columns that uniquely describe one result row
# ------------------------------------------------------------

candidate_identifier_columns = [
    "task_index",
    "task_hash",
    "reference_design_wavelength_nm",
    "case_family",
    "scenario",
    "absorber_location",
    "absorber_k",
    "core_thickness_factor",
    "n_shells",
    "outer_shell_thickness_nm",
    "outer_shell_n",
    "collection_direction",
]

identifier_columns = [
    column
    for column in candidate_identifier_columns
    if column in df_raw.columns
]


if not identifier_columns:
    raise KeyError(
        "No suitable scalar identifier columns were found."
    )


print(
    "Columns used for duplicate detection:"
)

for column in identifier_columns:
    print("  ", column)


# ------------------------------------------------------------
# Duplicate checks
# ------------------------------------------------------------

duplicate_identifier_mask = (
    df_raw.duplicated(
        subset=identifier_columns,
        keep=False,
    )
)

number_duplicated_rows = int(
    duplicate_identifier_mask.sum()
)

number_duplicated_groups = int(
    df_raw.loc[
        duplicate_identifier_mask,
        identifier_columns,
    ]
    .drop_duplicates()
    .shape[0]
)


# A task should usually have exactly two compact result rows:
# one backward and one forward.
task_group_columns = [
    column
    for column in [
        "task_index",
        "task_hash",
    ]
    if column in df_raw.columns
]

if task_group_columns:

    df_rows_per_task = (
        df_raw
        .groupby(
            task_group_columns,
            dropna=False,
        )
        .size()
        .rename(
            "rows_per_task"
        )
        .reset_index()
    )

else:

    df_rows_per_task = pd.DataFrame()


# ------------------------------------------------------------
# Missingness audit
# ------------------------------------------------------------

columns_fully_missing = [
    column
    for column in df_raw.columns
    if df_raw[column].isna().all()
]

rows_with_missing_values = int(
    df_raw.isna().any(axis=1).sum()
)


audit_records = [
    {
        "check": "number_of_rows",
        "value": len(df_raw),
    },
    {
        "check": "number_of_columns",
        "value": len(df_raw.columns),
    },
    {
        "check": (
            "rows_in_duplicated_identifier_groups"
        ),
        "value": number_duplicated_rows,
    },
    {
        "check": (
            "duplicated_identifier_groups"
        ),
        "value": number_duplicated_groups,
    },
    {
        "check": (
            "columns_containing_only_missing_values"
        ),
        "value": len(
            columns_fully_missing
        ),
    },
    {
        "check": (
            "rows_containing_at_least_one_missing_value"
        ),
        "value": rows_with_missing_values,
    },
]


df_basic_audit = pd.DataFrame(
    audit_records
)

display(
    df_basic_audit
)


# ------------------------------------------------------------
# Inspect duplicated identifiers
# ------------------------------------------------------------

if number_duplicated_rows > 0:

    print(
        "\nDuplicated identifier combinations:"
    )

    display(
        df_raw.loc[
            duplicate_identifier_mask,
            identifier_columns,
        ]
        .sort_values(
            identifier_columns
        )
    )

else:

    print(
        "\nNo duplicated result identifiers were found."
    )


# ------------------------------------------------------------
# Verify rows per physical task
# ------------------------------------------------------------

if not df_rows_per_task.empty:

    print(
        "\nNumber of compact result rows per physical task:"
    )

    display(
        df_rows_per_task[
            "rows_per_task"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "rows_per_task"
        )
        .reset_index(
            name="number_of_tasks"
        )
    )

    unexpected_task_sizes = (
        df_rows_per_task.loc[
            df_rows_per_task[
                "rows_per_task"
            ]
            != 2
        ]
    )

    if unexpected_task_sizes.empty:

        print(
            "Every physical task has exactly two rows: "
            "one backward and one forward result."
        )

    else:

        print(
            "Warning: some tasks do not have exactly "
            "two compact result rows."
        )

        display(
            unexpected_task_sizes
        )


# ------------------------------------------------------------
# Columns with missing values
# ------------------------------------------------------------

missingness_records = []

for column in df_raw.columns:

    series = df_raw[column]

    number_missing = int(
        series.isna().sum()
    )

    if number_missing == 0:
        continue

    missingness_records.append(
        {
            "column": column,
            "dtype": str(
                series.dtype
            ),
            "missing": number_missing,
            "missing_fraction": (
                number_missing
                / len(series)
            ),
        }
    )


df_missingness = pd.DataFrame(
    missingness_records
)


if df_missingness.empty:

    print(
        "\nNo missing values were found."
    )

else:

    df_missingness = (
        df_missingness
        .sort_values(
            "missing_fraction",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    print(
        "\nColumns containing missing values:"
    )

    display(
        df_missingness.style.format(
            {
                "missing_fraction": (
                    "{:.2%}"
                ),
            }
        )
    )


# ------------------------------------------------------------
# Save audit tables
# ------------------------------------------------------------

df_basic_audit.to_csv(
    TABLE_DIR
    / "basic_integrity_audit.csv",
    index=False,
)

df_rows_per_task.to_csv(
    TABLE_DIR
    / "rows_per_task.csv",
    index=False,
)

df_missingness.to_csv(
    TABLE_DIR
    / "missingness_audit.csv",
    index=False,
)

print(
    "\nSaved integrity-audit tables to:",
    TABLE_DIR,
)

Code cell 8: Inspect possible forward-suppression metrics numerically

In [ ]:
# ============================================================
# PAIR FORWARD AND BACKWARD RESULTS
# AND CALCULATE PROVISIONAL SCALAR METRICS
#
# Important:
# The compact summary stores forward and backward results as
# separate rows identified by collection_direction.
#
# This cell converts them into one row per physical case and
# calculates:
#
#   - backward/forward peak-efficiency ratio
#   - directional contrast in dB
#   - forward suppression relative to the nonabsorbing reference
#   - backward-scattering retention relative to the reference
#   - changes in backward-colour CIELAB coordinates
#
# Peak-based metrics are useful for initial screening but are not
# final spectral figures because forward and backward maxima can
# occur at different wavelengths. Spectrally matched metrics will
# later be calculated from the full checkpoints.
# ============================================================

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# SETTINGS
# ============================================================

DIRECTION_COLUMN = "collection_direction"

DIRECTION_VALUES_REQUIRED = {
    "backward",
    "forward",
}

# Small positive number used only to prevent division by zero.
# The resulting ratios should still be inspected for cases with
# negligibly small absolute scattering.
RATIO_EPSILON = 1e-15


# ============================================================
# VALIDATE DIRECTION ENCODING
# ============================================================

if DIRECTION_COLUMN not in df_raw.columns:
    raise KeyError(
        f"Required column {DIRECTION_COLUMN!r} was not found."
    )

direction_values = set(
    df_raw[
        DIRECTION_COLUMN
    ]
    .dropna()
    .astype(str)
    .str.lower()
    .unique()
)

missing_directions = (
    DIRECTION_VALUES_REQUIRED
    - direction_values
)

if missing_directions:
    raise ValueError(
        "The compact summary does not contain all required "
        "collection directions.\n"
        f"Available: {sorted(direction_values)}\n"
        f"Missing: {sorted(missing_directions)}"
    )

print(
    "Collection directions:",
    sorted(direction_values),
)


# Standardize direction names.
df_directional = df_raw.copy()

df_directional[
    DIRECTION_COLUMN
] = (
    df_directional[
        DIRECTION_COLUMN
    ]
    .astype(str)
    .str.lower()
)


# ============================================================
# DEFINE PHYSICAL-CASE IDENTIFIERS
# ============================================================

# task_hash should uniquely identify one physical case.
# task_index is retained for traceability but is not required
# for matching if task_hash is available.

if "task_hash" in df_directional.columns:

    case_identifier_columns = [
        "task_hash",
    ]

    if "task_index" in df_directional.columns:
        case_identifier_columns.insert(
            0,
            "task_index",
        )

else:

    # Fallback if task_hash is ever absent.
    fallback_identifiers = [
        "reference_design_wavelength_nm",
        "case_family",
        "scenario",
        "absorber_location",
        "absorber_k",
        "core_thickness_factor",
        "n_shells",
        "outer_shell_thickness_nm",
        "outer_shell_n",
    ]

    case_identifier_columns = [
        column
        for column in fallback_identifiers
        if column in df_directional.columns
    ]

    if not case_identifier_columns:
        raise KeyError(
            "No physical-case identifier columns were found."
        )


print(
    "Physical-case identifiers:",
    case_identifier_columns,
)


# ============================================================
# VERIFY ONE ROW PER DIRECTION AND PHYSICAL CASE
# ============================================================

direction_count_table = (
    df_directional
    .groupby(
        case_identifier_columns,
        dropna=False,
    )[DIRECTION_COLUMN]
    .agg(
        number_of_rows="size",
        number_of_directions="nunique",
    )
    .reset_index()
)

invalid_direction_groups = (
    direction_count_table.loc[
        (
            direction_count_table[
                "number_of_rows"
            ]
            != 2
        )
        | (
            direction_count_table[
                "number_of_directions"
            ]
            != 2
        )
    ]
)

if not invalid_direction_groups.empty:

    print(
        "Warning: some physical cases do not contain exactly "
        "one forward and one backward row."
    )

    display(
        invalid_direction_groups
    )

    raise ValueError(
        "Resolve incomplete or duplicated direction pairs before "
        "continuing with the analysis."
    )

print(
    "All physical cases contain exactly one forward and "
    "one backward result row."
)


# ============================================================
# SELECT SCALAR RESULT COLUMNS FOR PAIRING
# ============================================================

candidate_directional_result_columns = [
    # Collected scattering
    "q_collected_geom_max",
    "q_collected_geom_peak_wavelength_nm",
    "Ccollected_max_m2",
    "fraction_collected_max",
    "fraction_collected_mean",

    # Colour coordinates
    "L",
    "a",
    "b",
    "C",
    "hue_deg",
    "eta_C",
    "eta_L",
    "eta_Y",
    "X",
    "Y",
    "Z",
    "hex",
    "sRGB_r_clipped",
    "sRGB_g_clipped",
    "sRGB_b_clipped",
    "sRGB_was_clipped",

    # Scalar spectral summaries
    "spectrum_peak_wavelength_nm",
    "spectrum_peak_value",
    "spectrum_integral",
    "spectrum_centroid_nm",

    # Solver summaries
    "Qsca_max",
    "Qsca_peak_wavelength_nm",
    "Qext_max",
    "Qabs_max",
]

directional_result_columns = [
    column
    for column in candidate_directional_result_columns
    if column in df_directional.columns
]

if not directional_result_columns:
    raise KeyError(
        "No scalar directional result columns were found."
    )

print(
    "\nDirectional scalar columns to pair:"
)

for column in directional_result_columns:
    print("  ", column)


# ============================================================
# PIVOT FORWARD AND BACKWARD RESULTS
# ============================================================

directional_wide = (
    df_directional[
        case_identifier_columns
        + [DIRECTION_COLUMN]
        + directional_result_columns
    ]
    .set_index(
        case_identifier_columns
        + [DIRECTION_COLUMN]
    )
    .unstack(
        DIRECTION_COLUMN
    )
)


# Flatten the two-level column index.
directional_wide.columns = [
    f"{result_column}_{direction}"
    for result_column, direction
    in directional_wide.columns
]

directional_wide = (
    directional_wide
    .reset_index()
)


# ============================================================
# ADD CASE METADATA ONCE PER PHYSICAL CASE
# ============================================================

metadata_candidates = [
    "reference_design_wavelength_nm",
    "reference_id",
    "case_family",
    "scenario",
    "absorber_location",
    "absorber_k",
    "core_thickness_factor",
    "n_shells",
    "outer_shell_thickness_nm",
    "outer_shell_n",
    "n_A",
    "n_B",
    "outer_layer",
    "reference_n_shells",
    "reference_diameter_nm",
    "reference_core_thickness_factor",
    "original_reference_k",
    "original_reference_absorber_location",
    "predicted_diameter_nm",
    "predicted_diameter_difference_nm",
    "reference_absorber_removed",
]

metadata_columns = [
    column
    for column in metadata_candidates
    if column in df_directional.columns
    and column not in case_identifier_columns
]

df_case_metadata = (
    df_directional[
        case_identifier_columns
        + metadata_columns
    ]
    .drop_duplicates(
        subset=case_identifier_columns
    )
)


df_paired = directional_wide.merge(
    df_case_metadata,
    on=case_identifier_columns,
    how="left",
    validate="one_to_one",
)


print(
    "\nPaired dataframe shape:",
    df_paired.shape,
)

print(
    "Physical cases:",
    len(df_paired),
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_ratio(
    numerator,
    denominator,
    epsilon=RATIO_EPSILON,
):
    """
    Calculate a nonnegative ratio while avoiding division by zero.
    """

    numerator = pd.to_numeric(
        numerator,
        errors="coerce",
    )

    denominator = pd.to_numeric(
        denominator,
        errors="coerce",
    )

    ratio = (
        numerator
        / (
            denominator
            + epsilon
        )
    )

    ratio.loc[
        (
            numerator < 0.0
        )
        | (
            denominator < 0.0
        )
    ] = np.nan

    return ratio


def circular_difference_deg(
    values_deg,
    reference_deg,
):
    """
    Wrapped circular difference in the interval [-180, 180).
    """

    return (
        (
            np.asarray(
                values_deg,
                dtype=float,
            )
            - np.asarray(
                reference_deg,
                dtype=float,
            )
            + 180.0
        )
        % 360.0
        - 180.0
    )


# ============================================================
# DIRECTIONAL SCATTERING METRICS
# ============================================================

required_peak_columns = {
    "q_collected_geom_max_backward",
    "q_collected_geom_max_forward",
}

if required_peak_columns.issubset(
    df_paired.columns
):

    q_back_peak = pd.to_numeric(
        df_paired[
            "q_collected_geom_max_backward"
        ],
        errors="coerce",
    )

    q_forward_peak = pd.to_numeric(
        df_paired[
            "q_collected_geom_max_forward"
        ],
        errors="coerce",
    )

    # Ratio above one indicates stronger backward than forward
    # collected peak efficiency.
    df_paired[
        "backward_to_forward_peak_ratio"
    ] = safe_ratio(
        q_back_peak,
        q_forward_peak,
    )

    # Positive values indicate backward-dominant peak scattering.
    df_paired[
        "backward_to_forward_peak_db"
    ] = (
        10.0
        * np.log10(
            df_paired[
                "backward_to_forward_peak_ratio"
            ]
        )
    )

    # Fraction of the two hemispherical peak metrics associated
    # with backward collection.
    #
    # This remains provisional because the two maxima can occur at
    # different wavelengths.
    df_paired[
        "backward_peak_share"
    ] = safe_ratio(
        q_back_peak,
        q_back_peak + q_forward_peak,
    )

else:

    print(
        "Warning: q_collected_geom_max was unavailable in one or "
        "both collection directions."
    )


# ============================================================
# IDENTIFY THE NEW NONABSORBING REFERENCE CASES
# ============================================================

reference_mask = pd.Series(
    False,
    index=df_paired.index,
)

if "case_family" in df_paired.columns:

    reference_mask |= (
        df_paired["case_family"]
        .astype(str)
        .str.lower()
        .eq("reference")
    )

if (
    "absorber_location"
    in df_paired.columns
    and "absorber_k"
    in df_paired.columns
):

    reference_mask |= (
        df_paired[
            "absorber_location"
        ]
        .astype(str)
        .str.lower()
        .eq("none")
        & np.isclose(
            pd.to_numeric(
                df_paired[
                    "absorber_k"
                ],
                errors="coerce",
            ),
            0.0,
        )
    )


df_reference_cases = (
    df_paired.loc[
        reference_mask
    ]
    .copy()
)


if "reference_design_wavelength_nm" not in df_reference_cases.columns:
    raise KeyError(
        "reference_design_wavelength_nm is required to pair each "
        "absorbing case with its nonabsorbing reference."
    )


reference_counts = (
    df_reference_cases[
        "reference_design_wavelength_nm"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    "\nNonabsorbing reference cases per design wavelength:"
)

display(
    reference_counts
    .rename_axis(
        "reference_design_wavelength_nm"
    )
    .reset_index(
        name="number_of_reference_cases"
    )
)


if not (
    reference_counts == 1
).all():

    display(
        df_reference_cases[
            [
                column
                for column in [
                    "reference_design_wavelength_nm",
                    "task_index",
                    "task_hash",
                    "case_family",
                    "scenario",
                    "absorber_location",
                    "absorber_k",
                ]
                if column
                in df_reference_cases.columns
            ]
        ]
    )

    raise ValueError(
        "Expected exactly one new nonabsorbing reference for each "
        "design wavelength."
    )


# ============================================================
# MERGE REFERENCE METRICS INTO EVERY CASE
# ============================================================

reference_metric_candidates = [
    "q_collected_geom_max_backward",
    "q_collected_geom_max_forward",
    "spectrum_integral_backward",
    "spectrum_integral_forward",
    "L_backward",
    "a_backward",
    "b_backward",
    "C_backward",
    "hue_deg_backward",
    "eta_C_backward",
    "eta_L_backward",
    "eta_Y_backward",
    "hex_backward",
]

reference_metric_columns = [
    column
    for column in reference_metric_candidates
    if column in df_reference_cases.columns
]


df_reference_metrics = (
    df_reference_cases[
        [
            "reference_design_wavelength_nm",
        ]
        + reference_metric_columns
    ]
    .copy()
)


df_reference_metrics = (
    df_reference_metrics.rename(
        columns={
            column: (
                f"{column}_reference"
            )
            for column
            in reference_metric_columns
        }
    )
)


df_paired = df_paired.merge(
    df_reference_metrics,
    on="reference_design_wavelength_nm",
    how="left",
    validate="many_to_one",
)


# ============================================================
# FORWARD SUPPRESSION AND BACKWARD RETENTION
# ============================================================

if {
    "q_collected_geom_max_forward",
    "q_collected_geom_max_forward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "forward_peak_retention"
    ] = safe_ratio(
        df_paired[
            "q_collected_geom_max_forward"
        ],
        df_paired[
            "q_collected_geom_max_forward_reference"
        ],
    )

    df_paired[
        "forward_peak_suppression_fraction"
    ] = (
        1.0
        - df_paired[
            "forward_peak_retention"
        ]
    )

    # Positive dB values mean that forward peak scattering has
    # been suppressed relative to the nonabsorbing reference.
    df_paired[
        "forward_peak_suppression_db"
    ] = (
        -10.0
        * np.log10(
            df_paired[
                "forward_peak_retention"
            ]
        )
    )


if {
    "q_collected_geom_max_backward",
    "q_collected_geom_max_backward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "backward_peak_retention"
    ] = safe_ratio(
        df_paired[
            "q_collected_geom_max_backward"
        ],
        df_paired[
            "q_collected_geom_max_backward_reference"
        ],
    )

    df_paired[
        "backward_peak_change_fraction"
    ] = (
        df_paired[
            "backward_peak_retention"
        ]
        - 1.0
    )


# Spectral-integral metrics are preferable to independent maxima
# if spectrum_integral exists in the compact result table.
if {
    "spectrum_integral_forward",
    "spectrum_integral_forward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "forward_integral_retention"
    ] = safe_ratio(
        df_paired[
            "spectrum_integral_forward"
        ],
        df_paired[
            "spectrum_integral_forward_reference"
        ],
    )

    df_paired[
        "forward_integral_suppression_fraction"
    ] = (
        1.0
        - df_paired[
            "forward_integral_retention"
        ]
    )

    df_paired[
        "forward_integral_suppression_db"
    ] = (
        -10.0
        * np.log10(
            df_paired[
                "forward_integral_retention"
            ]
        )
    )


if {
    "spectrum_integral_backward",
    "spectrum_integral_backward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "backward_integral_retention"
    ] = safe_ratio(
        df_paired[
            "spectrum_integral_backward"
        ],
        df_paired[
            "spectrum_integral_backward_reference"
        ],
    )

    df_paired[
        "backward_to_forward_integral_ratio"
    ] = safe_ratio(
        df_paired[
            "spectrum_integral_backward"
        ],
        df_paired[
            "spectrum_integral_forward"
        ],
    )

    df_paired[
        "backward_to_forward_integral_db"
    ] = (
        10.0
        * np.log10(
            df_paired[
                "backward_to_forward_integral_ratio"
            ]
        )
    )


# ============================================================
# BACKWARD-COLOUR PERFORMANCE RELATIVE TO REFERENCE
# ============================================================

if {
    "C_backward",
    "C_backward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "delta_C_backward"
    ] = (
        pd.to_numeric(
            df_paired[
                "C_backward"
            ],
            errors="coerce",
        )
        - pd.to_numeric(
            df_paired[
                "C_backward_reference"
            ],
            errors="coerce",
        )
    )


if {
    "L_backward",
    "L_backward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "delta_L_backward"
    ] = (
        pd.to_numeric(
            df_paired[
                "L_backward"
            ],
            errors="coerce",
        )
        - pd.to_numeric(
            df_paired[
                "L_backward_reference"
            ],
            errors="coerce",
        )
    )


if {
    "hue_deg_backward",
    "hue_deg_backward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "delta_hue_backward_deg"
    ] = circular_difference_deg(
        df_paired[
            "hue_deg_backward"
        ],
        df_paired[
            "hue_deg_backward_reference"
        ],
    )


if {
    "L_backward",
    "a_backward",
    "b_backward",
    "L_backward_reference",
    "a_backward_reference",
    "b_backward_reference",
}.issubset(
    df_paired.columns
):

    df_paired[
        "delta_E_ab_backward"
    ] = np.sqrt(
        (
            pd.to_numeric(
                df_paired[
                    "L_backward"
                ],
                errors="coerce",
            )
            - pd.to_numeric(
                df_paired[
                    "L_backward_reference"
                ],
                errors="coerce",
            )
        )
        ** 2
        + (
            pd.to_numeric(
                df_paired[
                    "a_backward"
                ],
                errors="coerce",
            )
            - pd.to_numeric(
                df_paired[
                    "a_backward_reference"
                ],
                errors="coerce",
            )
        )
        ** 2
        + (
            pd.to_numeric(
                df_paired[
                    "b_backward"
                ],
                errors="coerce",
            )
            - pd.to_numeric(
                df_paired[
                    "b_backward_reference"
                ],
                errors="coerce",
            )
        )
        ** 2
    )


# Existing normalized colour objective.
if {
    "eta_C_backward",
    "eta_L_backward",
}.issubset(
    df_paired.columns
):

    eta_C_capped = pd.to_numeric(
        df_paired[
            "eta_C_backward"
        ],
        errors="coerce",
    ).clip(
        upper=1.0
    )

    eta_L_capped = pd.to_numeric(
        df_paired[
            "eta_L_backward"
        ],
        errors="coerce",
    ).clip(
        upper=1.0
    )

    df_paired[
        "d_opt_colour_backward"
    ] = np.hypot(
        1.0 - eta_C_capped,
        1.0 - eta_L_capped,
    )


# ============================================================
# REPORT AVAILABLE DERIVED METRICS
# ============================================================

derived_metric_candidates = [
    "backward_to_forward_peak_ratio",
    "backward_to_forward_peak_db",
    "backward_peak_share",
    "forward_peak_retention",
    "forward_peak_suppression_fraction",
    "forward_peak_suppression_db",
    "backward_peak_retention",
    "backward_peak_change_fraction",
    "forward_integral_retention",
    "forward_integral_suppression_fraction",
    "forward_integral_suppression_db",
    "backward_integral_retention",
    "backward_to_forward_integral_ratio",
    "backward_to_forward_integral_db",
    "delta_C_backward",
    "delta_L_backward",
    "delta_hue_backward_deg",
    "delta_E_ab_backward",
    "d_opt_colour_backward",
]

derived_metric_columns = [
    column
    for column in derived_metric_candidates
    if column in df_paired.columns
]


derived_summary_records = []

for column in derived_metric_columns:

    values = pd.to_numeric(
        df_paired[column],
        errors="coerce",
    )

    finite_values = values.loc[
        np.isfinite(values)
    ]

    if finite_values.empty:
        continue

    derived_summary_records.append(
        {
            "metric": column,
            "finite_values": len(
                finite_values
            ),
            "minimum": float(
                finite_values.min()
            ),
            "median": float(
                finite_values.median()
            ),
            "mean": float(
                finite_values.mean()
            ),
            "maximum": float(
                finite_values.max()
            ),
        }
    )


df_derived_metric_summary = pd.DataFrame(
    derived_summary_records
)


print(
    "\nDerived scalar metrics:"
)

display(
    df_derived_metric_summary.style.format(
        {
            "minimum": "{:.5g}",
            "median": "{:.5g}",
            "mean": "{:.5g}",
            "maximum": "{:.5g}",
        }
    )
)


# ============================================================
# DISPLAY THE NEW NONABSORBING REFERENCES
# ============================================================

reference_display_columns = [
    column
    for column in [
        "reference_design_wavelength_nm",
        "n_A",
        "n_B",
        "reference_n_shells",
        "reference_diameter_nm",
        "q_collected_geom_max_backward",
        "q_collected_geom_max_forward",
        "spectrum_integral_backward",
        "spectrum_integral_forward",
        "L_backward",
        "C_backward",
        "hue_deg_backward",
        "eta_C_backward",
        "eta_L_backward",
        "d_opt_colour_backward",
        "hex_backward",
    ]
    if column in df_reference_cases.columns
    or column in df_paired.columns
]


print(
    "\nNew nonabsorbing reference cases:"
)

display(
    df_paired.loc[
        reference_mask,
        reference_display_columns,
    ].sort_values(
        "reference_design_wavelength_nm"
    )
)


# ============================================================
# SAVE THE PAIRED ANALYSIS TABLE
# ============================================================

PAIRED_PARQUET_PATH = (
    TABLE_DIR
    / "forward_backward_paired_cases.parquet"
)

PAIRED_CSV_PATH = (
    TABLE_DIR
    / "forward_backward_paired_cases.csv"
)

df_paired.to_parquet(
    PAIRED_PARQUET_PATH,
    index=False,
)

df_paired.to_csv(
    PAIRED_CSV_PATH,
    index=False,
)

df_derived_metric_summary.to_csv(
    TABLE_DIR
    / "derived_scalar_metric_summary.csv",
    index=False,
)


print(
    "\nSaved paired analysis table:"
)

print(
    "  ",
    PAIRED_PARQUET_PATH,
)

print(
    "  ",
    PAIRED_CSV_PATH,
)

In [ ]:
print(df_paired.shape)
print(df_paired.columns.tolist())
display(df_derived_metric_summary)

In [ ]:
# ============================================================
# REFERENCE-RELATIVE DIRECTIONALITY METRICS
#
# Primary quantities:
#
#   Directional ratio:
#       R_BF = Integral(Q_back) / Integral(Q_forward)
#
#   Directionality improvement:
#       G_BF = R_BF / R_BF,reference
#
#   Directionality improvement in dB:
#       Delta S_BF = 10 log10(G_BF)
#
# Positive Delta S_BF means that the backward-to-forward balance
# improved relative to the nonabsorbing reference.
#
# This does not imply that scattering became backward dominant.
# ============================================================

required_columns = {
    "reference_design_wavelength_nm",
    "spectrum_integral_backward",
    "spectrum_integral_forward",
    "backward_integral_retention",
    "forward_integral_retention",
}

missing_columns = (
    required_columns
    - set(df_paired.columns)
)

if missing_columns:
    raise KeyError(
        "Missing columns required for integrated directionality "
        f"analysis: {sorted(missing_columns)}"
    )


# ------------------------------------------------------------
# Directional ratio for every physical case
# ------------------------------------------------------------

df_paired[
    "backward_to_forward_integral_ratio"
] = safe_ratio(
    df_paired[
        "spectrum_integral_backward"
    ],
    df_paired[
        "spectrum_integral_forward"
    ],
)


# ------------------------------------------------------------
# Extract one ratio for each nonabsorbing reference
# ------------------------------------------------------------

reference_ratio_table = (
    df_paired.loc[
        reference_mask,
        [
            "reference_design_wavelength_nm",
            "backward_to_forward_integral_ratio",
        ],
    ]
    .rename(
        columns={
            "backward_to_forward_integral_ratio":
                "backward_to_forward_integral_ratio_reference",
        }
    )
    .copy()
)

reference_ratio_counts = (
    reference_ratio_table[
        "reference_design_wavelength_nm"
    ]
    .value_counts()
)

if not (
    reference_ratio_counts == 1
).all():
    raise ValueError(
        "Expected exactly one nonabsorbing directional-ratio "
        "reference for each design wavelength."
    )


# Remove an old merged reference column if this cell is rerun.
column_to_replace = (
    "backward_to_forward_integral_ratio_reference"
)

if column_to_replace in df_paired.columns:
    df_paired = df_paired.drop(
        columns=[
            column_to_replace,
        ]
    )


df_paired = df_paired.merge(
    reference_ratio_table,
    on="reference_design_wavelength_nm",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Improvement relative to the nonabsorbing reference
# ------------------------------------------------------------

df_paired[
    "directionality_improvement_factor"
] = safe_ratio(
    df_paired[
        "backward_to_forward_integral_ratio"
    ],
    df_paired[
        "backward_to_forward_integral_ratio_reference"
    ],
)

df_paired[
    "directionality_improvement_percent"
] = (
    100.0
    * (
        df_paired[
            "directionality_improvement_factor"
        ]
        - 1.0
    )
)

df_paired[
    "directionality_improvement_db"
] = (
    10.0
    * np.log10(
        df_paired[
            "directionality_improvement_factor"
        ]
    )
)


# ------------------------------------------------------------
# Directionality change can also be calculated directly from
# retentions:
#
#    G_BF = backward retention / forward retention
#
# This should agree numerically with the ratio-based definition.
# ------------------------------------------------------------

df_paired[
    "directionality_improvement_from_retentions"
] = safe_ratio(
    df_paired[
        "backward_integral_retention"
    ],
    df_paired[
        "forward_integral_retention"
    ],
)

df_paired[
    "directionality_improvement_consistency_error"
] = (
    df_paired[
        "directionality_improvement_factor"
    ]
    - df_paired[
        "directionality_improvement_from_retentions"
    ]
)


maximum_consistency_error = float(
    np.nanmax(
        np.abs(
            df_paired[
                "directionality_improvement_consistency_error"
            ]
        )
    )
)

print(
    "Maximum consistency difference:",
    maximum_consistency_error,
)

if maximum_consistency_error > 1e-8:
    warnings.warn(
        "The two directionality-improvement definitions differ "
        "by more than expected. Inspect the reference pairing."
    )


# ------------------------------------------------------------
# Convenient percentage metrics
# ------------------------------------------------------------

df_paired[
    "forward_integral_suppression_percent"
] = (
    100.0
    * (
        1.0
        - df_paired[
            "forward_integral_retention"
        ]
    )
)

df_paired[
    "backward_integral_retention_percent"
] = (
    100.0
    * df_paired[
        "backward_integral_retention"
    ]
)

df_paired[
    "backward_integral_loss_percent"
] = (
    100.0
    * (
        1.0
        - df_paired[
            "backward_integral_retention"
        ]
    )
)


# ------------------------------------------------------------
# Inspect reference directionality
# ------------------------------------------------------------

reference_directionality_columns = [
    "reference_design_wavelength_nm",
    "spectrum_integral_backward",
    "spectrum_integral_forward",
    "backward_to_forward_integral_ratio",
    "backward_to_forward_integral_db",
    "L_backward",
    "C_backward",
    "hue_deg_backward",
    "eta_C_backward",
    "eta_L_backward",
    "d_opt_colour_backward",
]

reference_directionality_columns = [
    column
    for column in reference_directionality_columns
    if column in df_paired.columns
]

print(
    "\nNonabsorbing reference directionality:"
)

display(
    df_paired.loc[
        reference_mask,
        reference_directionality_columns,
    ]
    .sort_values(
        "reference_design_wavelength_nm"
    )
    .style.format(
        {
            "spectrum_integral_backward": "{:.4g}",
            "spectrum_integral_forward": "{:.4g}",
            "backward_to_forward_integral_ratio": "{:.4f}",
            "backward_to_forward_integral_db": "{:.2f}",
            "L_backward": "{:.2f}",
            "C_backward": "{:.2f}",
            "hue_deg_backward": "{:.2f}",
            "eta_C_backward": "{:.3f}",
            "eta_L_backward": "{:.3f}",
            "d_opt_colour_backward": "{:.3f}",
        }
    )
)

In [ ]:
# ============================================================
# INTERPRETABLE PERFORMANCE CONSTRAINTS
#
# These are analysis thresholds, not universal perceptual limits.
# Adjust them after inspecting the trade-off distributions.
# ============================================================

MIN_FORWARD_SUPPRESSION_PERCENT = 10.0
MIN_BACKWARD_RETENTION_PERCENT = 80.0

MAX_ABS_DELTA_C = 10.0
MAX_ABS_DELTA_L = 10.0
MAX_ABS_DELTA_HUE_DEG = 10.0

# Optional colour-solid performance constraints.
MIN_ETA_C = 0.25
MIN_ETA_L = 0.60


# ------------------------------------------------------------
# Primary flags
# ------------------------------------------------------------

df_paired[
    "improves_directionality"
] = (
    df_paired[
        "directionality_improvement_factor"
    ]
    > 1.0
)

df_paired[
    "suppresses_forward_integral"
] = (
    df_paired[
        "forward_integral_suppression_percent"
    ]
    >= MIN_FORWARD_SUPPRESSION_PERCENT
)

df_paired[
    "retains_backward_integral"
] = (
    df_paired[
        "backward_integral_retention_percent"
    ]
    >= MIN_BACKWARD_RETENTION_PERCENT
)

df_paired[
    "preserves_chroma"
] = (
    df_paired[
        "delta_C_backward"
    ].abs()
    <= MAX_ABS_DELTA_C
)

df_paired[
    "preserves_lightness"
] = (
    df_paired[
        "delta_L_backward"
    ].abs()
    <= MAX_ABS_DELTA_L
)

df_paired[
    "preserves_hue"
] = (
    df_paired[
        "delta_hue_backward_deg"
    ].abs()
    <= MAX_ABS_DELTA_HUE_DEG
)


# ------------------------------------------------------------
# Absolute colour-performance flags
# ------------------------------------------------------------

df_paired[
    "acceptable_eta_C"
] = (
    pd.to_numeric(
        df_paired[
            "eta_C_backward"
        ],
        errors="coerce",
    )
    >= MIN_ETA_C
)

df_paired[
    "acceptable_eta_L"
] = (
    pd.to_numeric(
        df_paired[
            "eta_L_backward"
        ],
        errors="coerce",
    )
    >= MIN_ETA_L
)


# ------------------------------------------------------------
# Combined transparent selection
# ------------------------------------------------------------

df_paired[
    "passes_primary_tradeoff_constraints"
] = (
    df_paired[
        "improves_directionality"
    ]
    & df_paired[
        "suppresses_forward_integral"
    ]
    & df_paired[
        "retains_backward_integral"
    ]
    & df_paired[
        "preserves_chroma"
    ]
    & df_paired[
        "preserves_lightness"
    ]
    & df_paired[
        "preserves_hue"
    ]
)


constraint_summary = pd.DataFrame(
    {
        "criterion": [
            "Improves backward/forward balance",
            (
                f"Forward suppression >= "
                f"{MIN_FORWARD_SUPPRESSION_PERCENT:.0f}%"
            ),
            (
                f"Backward retention >= "
                f"{MIN_BACKWARD_RETENTION_PERCENT:.0f}%"
            ),
            (
                f"|Delta C*| <= "
                f"{MAX_ABS_DELTA_C:.0f}"
            ),
            (
                f"|Delta L*| <= "
                f"{MAX_ABS_DELTA_L:.0f}"
            ),
            (
                f"|Delta hue| <= "
                f"{MAX_ABS_DELTA_HUE_DEG:.0f} degrees"
            ),
            "Passes all primary constraints",
        ],
        "number_of_cases": [
            int(
                df_paired[
                    "improves_directionality"
                ].sum()
            ),
            int(
                df_paired[
                    "suppresses_forward_integral"
                ].sum()
            ),
            int(
                df_paired[
                    "retains_backward_integral"
                ].sum()
            ),
            int(
                df_paired[
                    "preserves_chroma"
                ].sum()
            ),
            int(
                df_paired[
                    "preserves_lightness"
                ].sum()
            ),
            int(
                df_paired[
                    "preserves_hue"
                ].sum()
            ),
            int(
                df_paired[
                    "passes_primary_tradeoff_constraints"
                ].sum()
            ),
        ],
    }
)

constraint_summary[
    "fraction_of_cases"
] = (
    constraint_summary[
        "number_of_cases"
    ]
    / len(df_paired)
)

display(
    constraint_summary.style.format(
        {
            "fraction_of_cases": "{:.1%}",
        }
    )
)

Cell 1: Add balanced-directionality and forward/backward hue metrics

In [ ]:
# ============================================================
# BALANCED FORWARD/BACKWARD SCATTERING AND HUE METRICS
#
# The objective is not complete suppression of forward scattering.
# Instead, favourable cases should have:
#
#   1. comparable integrated forward and backward scattering;
#   2. similar forward and backward hues;
#   3. substantial retained backward scattering;
#   4. acceptable backward chroma and lightness.
# ============================================================

import numpy as np
import pandas as pd


def circular_difference_deg(
    values_deg,
    reference_deg,
):
    """
    Circular difference in the interval [-180, 180).
    """

    return (
        (
            np.asarray(
                values_deg,
                dtype=float,
            )
            - np.asarray(
                reference_deg,
                dtype=float,
            )
            + 180.0
        )
        % 360.0
        - 180.0
    )


# ------------------------------------------------------------
# Verify required columns
# ------------------------------------------------------------

required_balance_columns = {
    "spectrum_integral_backward",
    "spectrum_integral_forward",
    "hue_deg_backward",
    "hue_deg_forward",
    "backward_integral_retention",
}

missing_balance_columns = (
    required_balance_columns
    - set(df_paired.columns)
)

if missing_balance_columns:

    raise KeyError(
        "Missing columns required for the balanced-scattering "
        f"analysis: {sorted(missing_balance_columns)}"
    )


# ------------------------------------------------------------
# Integrated forward/backward scattering balance
# ------------------------------------------------------------

df_paired[
    "backward_to_forward_integral_ratio"
] = safe_ratio(
    df_paired[
        "spectrum_integral_backward"
    ],
    df_paired[
        "spectrum_integral_forward"
    ],
)


# Signed ratio in dB:
#
#   0 dB  -> equal integrated scattering
#   <0 dB -> forward-dominant
#   >0 dB -> backward-dominant
df_paired[
    "backward_forward_integral_db"
] = (
    10.0
    * np.log10(
        df_paired[
            "backward_to_forward_integral_ratio"
        ]
    )
)


# Absolute distance from equal forward/backward scattering.
# Smaller is better.
df_paired[
    "integral_directional_imbalance_db"
] = (
    df_paired[
        "backward_forward_integral_db"
    ].abs()
)


# Symmetric bounded balance score:
#
#   1.0 -> equal forward and backward integrals
#   0.0 -> strongly imbalanced
#
# min(B/F, F/B) avoids assigning preference to either direction.
df_paired[
    "integral_directional_balance_score"
] = np.minimum(
    df_paired[
        "backward_to_forward_integral_ratio"
    ],
    safe_ratio(
        df_paired[
            "spectrum_integral_forward"
        ],
        df_paired[
            "spectrum_integral_backward"
        ],
    ),
)


# ------------------------------------------------------------
# Forward/backward colour agreement
# ------------------------------------------------------------

df_paired[
    "forward_minus_backward_hue_deg"
] = (
    circular_difference_deg(
        df_paired[
            "hue_deg_forward"
        ],
        df_paired[
            "hue_deg_backward"
        ],
    )
)

df_paired[
    "absolute_forward_backward_hue_difference_deg"
] = (
    df_paired[
        "forward_minus_backward_hue_deg"
    ].abs()
)


# Optional complete CIELAB difference between directions.
if {
    "L_forward",
    "a_forward",
    "b_forward",
    "L_backward",
    "a_backward",
    "b_backward",
}.issubset(
    df_paired.columns
):

    df_paired[
        "delta_E_ab_forward_backward"
    ] = np.sqrt(
        (
            pd.to_numeric(
                df_paired["L_forward"],
                errors="coerce",
            )
            - pd.to_numeric(
                df_paired["L_backward"],
                errors="coerce",
            )
        )
        ** 2
        + (
            pd.to_numeric(
                df_paired["a_forward"],
                errors="coerce",
            )
            - pd.to_numeric(
                df_paired["a_backward"],
                errors="coerce",
            )
        )
        ** 2
        + (
            pd.to_numeric(
                df_paired["b_forward"],
                errors="coerce",
            )
            - pd.to_numeric(
                df_paired["b_backward"],
                errors="coerce",
            )
        )
        ** 2
    )


# ------------------------------------------------------------
# Percentage form of backward retention
# ------------------------------------------------------------

df_paired[
    "backward_integral_retention_percent"
] = (
    100.0
    * pd.to_numeric(
        df_paired[
            "backward_integral_retention"
        ],
        errors="coerce",
    )
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

balance_metric_columns = [
    "backward_to_forward_integral_ratio",
    "backward_forward_integral_db",
    "integral_directional_imbalance_db",
    "integral_directional_balance_score",
    "absolute_forward_backward_hue_difference_deg",
    "delta_E_ab_forward_backward",
    "backward_integral_retention_percent",
]

balance_metric_columns = [
    column
    for column in balance_metric_columns
    if column in df_paired.columns
]

balance_summary_records = []

for column in balance_metric_columns:

    values = pd.to_numeric(
        df_paired[column],
        errors="coerce",
    )

    values = values.loc[
        np.isfinite(values)
    ]

    if values.empty:
        continue

    balance_summary_records.append(
        {
            "metric": column,
            "minimum": values.min(),
            "median": values.median(),
            "mean": values.mean(),
            "maximum": values.max(),
        }
    )


df_balance_summary = pd.DataFrame(
    balance_summary_records
)

display(
    df_balance_summary.style.format(
        {
            "minimum": "{:.4g}",
            "median": "{:.4g}",
            "mean": "{:.4g}",
            "maximum": "{:.4g}",
        }
    )
)

Cell 2: Define meaningful constraint categories

The previous requirement of at least 10%10\%10% forward suppression is no longer central. Replace it with ranges of directional comparability.

In [ ]:
# ============================================================
# SOFTENED, REFERENCE-RELATIVE SCREENING CRITERIA
#
# These criteria are chosen to identify improvement within the
# calculated design space. They are operational screening limits,
# not universal perceptual thresholds.
# ============================================================

MIN_DIRECTIONAL_IMPROVEMENT_DB = 1.0

MAX_FORWARD_BACKWARD_HUE_DIFFERENCE_DEG = 60.0

MIN_BACKWARD_RETENTION_PERCENT = 70.0

MAX_ABS_DELTA_C_BACKWARD = 10.0
MAX_ABS_DELTA_L_BACKWARD = 10.0
MAX_ABS_DELTA_HUE_BACKWARD_DEG = 15.0


# ------------------------------------------------------------
# Exclude nonabsorbing references from absorber optimization
# ------------------------------------------------------------

df_paired[
    "is_nonabsorbing_reference"
] = (
    df_paired[
        "case_family"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("reference")
)


# ------------------------------------------------------------
# Criteria
# ------------------------------------------------------------

df_paired[
    "improves_directionality_by_required_amount"
] = (
    pd.to_numeric(
        df_paired[
            "directionality_improvement_db"
        ],
        errors="coerce",
    )
    >= MIN_DIRECTIONAL_IMPROVEMENT_DB
)

df_paired[
    "similar_forward_backward_hue"
] = (
    pd.to_numeric(
        df_paired[
            "absolute_forward_backward_hue_difference_deg"
        ],
        errors="coerce",
    )
    <= MAX_FORWARD_BACKWARD_HUE_DIFFERENCE_DEG
)

df_paired[
    "sufficient_backward_retention"
] = (
    pd.to_numeric(
        df_paired[
            "backward_integral_retention_percent"
        ],
        errors="coerce",
    )
    >= MIN_BACKWARD_RETENTION_PERCENT
)

df_paired[
    "preserves_backward_chroma"
] = (
    pd.to_numeric(
        df_paired[
            "delta_C_backward"
        ],
        errors="coerce",
    ).abs()
    <= MAX_ABS_DELTA_C_BACKWARD
)

df_paired[
    "preserves_backward_lightness"
] = (
    pd.to_numeric(
        df_paired[
            "delta_L_backward"
        ],
        errors="coerce",
    ).abs()
    <= MAX_ABS_DELTA_L_BACKWARD
)

df_paired[
    "preserves_backward_hue"
] = (
    pd.to_numeric(
        df_paired[
            "delta_hue_backward_deg"
        ],
        errors="coerce",
    ).abs()
    <= MAX_ABS_DELTA_HUE_BACKWARD_DEG
)


df_paired[
    "passes_softened_tradeoff_constraints"
] = (
    ~df_paired[
        "is_nonabsorbing_reference"
    ]
    & df_paired[
        "improves_directionality_by_required_amount"
    ]
    & df_paired[
        "similar_forward_backward_hue"
    ]
    & df_paired[
        "sufficient_backward_retention"
    ]
    & df_paired[
        "preserves_backward_chroma"
    ]
    & df_paired[
        "preserves_backward_lightness"
    ]
    & df_paired[
        "preserves_backward_hue"
    ]
)


soft_constraint_definitions = {
    (
        f"Directionality improvement >= "
        f"{MIN_DIRECTIONAL_IMPROVEMENT_DB:.1f} dB"
    ): "improves_directionality_by_required_amount",

    (
        f"Forward/backward hue difference <= "
        f"{MAX_FORWARD_BACKWARD_HUE_DIFFERENCE_DEG:.0f} degrees"
    ): "similar_forward_backward_hue",

    (
        f"Backward retention >= "
        f"{MIN_BACKWARD_RETENTION_PERCENT:.0f}%"
    ): "sufficient_backward_retention",

    (
        f"|Delta C backward| <= "
        f"{MAX_ABS_DELTA_C_BACKWARD:.0f}"
    ): "preserves_backward_chroma",

    (
        f"|Delta L backward| <= "
        f"{MAX_ABS_DELTA_L_BACKWARD:.0f}"
    ): "preserves_backward_lightness",

    (
        f"|Delta hue backward| <= "
        f"{MAX_ABS_DELTA_HUE_BACKWARD_DEG:.0f} degrees"
    ): "preserves_backward_hue",

    "Passes all softened trade-off constraints":
        "passes_softened_tradeoff_constraints",
}


soft_constraint_records = []

for criterion, column in (
    soft_constraint_definitions.items()
):

    number_of_cases = int(
        df_paired[column].sum()
    )

    soft_constraint_records.append(
        {
            "criterion": criterion,
            "number_of_cases": number_of_cases,
            "fraction_of_cases": (
                number_of_cases
                / len(df_paired)
            ),
        }
    )


df_soft_constraint_summary = pd.DataFrame(
    soft_constraint_records
)

display(
    df_soft_constraint_summary.style.format(
        {
            "fraction_of_cases": "{:.1%}",
        }
    )
)

In [ ]:
# ============================================================
# PLOTTING-DATA DIAGNOSTIC
# ============================================================

df_paired[
    "case_family_clean"
] = (
    df_paired[
        "case_family"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        " ",
        "_",
        regex=False,
    )
    .str.replace(
        "-",
        "_",
        regex=False,
    )
)


print("Case-family values:")

display(
    df_paired[
        "case_family_clean"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "case_family_clean"
    )
    .reset_index(
        name="number_of_cases"
    )
)


plot_columns = [
    "backward_forward_integral_db",
    "directionality_improvement_db",
    "backward_integral_retention_percent",
    "absolute_forward_backward_hue_difference_deg",
    "delta_C_backward",
    "delta_L_backward",
]

finite_records = []

for wavelength_nm in sorted(
    df_paired[
        "reference_design_wavelength_nm"
    ]
    .dropna()
    .astype(int)
    .unique()
):

    wavelength_mask = np.isclose(
        pd.to_numeric(
            df_paired[
                "reference_design_wavelength_nm"
            ],
            errors="coerce",
        ),
        wavelength_nm,
    )

    row = {
        "reference_design_wavelength_nm":
            wavelength_nm,
        "total_cases": int(
            wavelength_mask.sum()
        ),
    }

    for column in plot_columns:

        values = pd.to_numeric(
            df_paired.loc[
                wavelength_mask,
                column,
            ],
            errors="coerce",
        )

        row[
            f"finite_{column}"
        ] = int(
            np.isfinite(values).sum()
        )

    finite_records.append(row)


df_plot_diagnostic = pd.DataFrame(
    finite_records
)

display(df_plot_diagnostic)

3. Complete corrected overview figure

This version deliberately avoids fragile categorical filtering. It converts all plotting columns to numeric, plots every finite case, and uses normalized case-family labels.

The horizontal axis is now directionality improvement relative to the nonabsorbing reference, rather than the absolute backward-to-forward ratio. Thus:

0 dB0\ \mathrm{dB}0 dB: no change relative to the reference;
positive: the backward-to-forward balance improved;
negative: the balance worsened.

In [ ]:
# ============================================================
# OVERVIEW: DIRECTIONALITY IMPROVEMENT VERSUS COLOUR PERFORMANCE
#
# Horizontal coordinate:
#   improvement of integrated backward/forward balance relative
#   to the corresponding nonabsorbing reference
#
#   0 dB  -> unchanged directional balance
#   >0 dB -> improved backward/forward balance
#   <0 dB -> worsened backward/forward balance
#
# Rows:
#   380, 580, and 760 nm reference designs
#
# Columns:
#   backward-integral retention
#   forward/backward hue difference
#   backward-colour change
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

REFERENCE_WAVELENGTHS = sorted(
    pd.to_numeric(
        df_paired[
            "reference_design_wavelength_nm"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

CASE_FAMILY_MARKERS = {
    "reference": "X",
    "low_index": "o",
    "all_layers": "s",
    "outer_shell": "^",
    "core": "D",
}

CASE_FAMILY_LABELS = {
    "reference": "Nonabsorbing reference",
    "low_index": "Low-index domains",
    "all_layers": "All layers",
    "outer_shell": "Additional outer shell",
    "core": "Core",
}

OVERVIEW_WIDTH_CM = 17.5
OVERVIEW_HEIGHT_CM = 12.5

SAVE_DIRECTIONAL_OVERVIEW = True

DIRECTIONAL_OVERVIEW_BASENAME = (
    "directionality_improvement_colour_overview"
)


# ------------------------------------------------------------
# Prepare a clean plotting dataframe
# ------------------------------------------------------------

df_overview = df_paired.copy()

df_overview[
    "case_family_clean"
] = (
    df_overview[
        "case_family"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        " ",
        "_",
        regex=False,
    )
    .str.replace(
        "-",
        "_",
        regex=False,
    )
)


numeric_plot_columns = [
    "reference_design_wavelength_nm",
    "absorber_k",
    "directionality_improvement_db",
    "backward_integral_retention_percent",
    "absolute_forward_backward_hue_difference_deg",
    "delta_C_backward",
]

for column in numeric_plot_columns:

    df_overview[column] = pd.to_numeric(
        df_overview[column],
        errors="coerce",
    )


required_overview_columns = {
    "reference_design_wavelength_nm",
    "case_family_clean",
    "absorber_k",
    "directionality_improvement_db",
    "backward_integral_retention_percent",
    "absolute_forward_backward_hue_difference_deg",
    "delta_C_backward",
}

missing_overview_columns = (
    required_overview_columns
    - set(df_overview.columns)
)

if missing_overview_columns:

    raise KeyError(
        "Missing overview columns: "
        f"{sorted(missing_overview_columns)}"
    )


# ------------------------------------------------------------
# Verify finite values before generating axes
# ------------------------------------------------------------

complete_plot_mask = (
    np.isfinite(
        df_overview[
            "directionality_improvement_db"
        ]
    )
    & np.isfinite(
        df_overview[
            "backward_integral_retention_percent"
        ]
    )
    & np.isfinite(
        df_overview[
            "absolute_forward_backward_hue_difference_deg"
        ]
    )
    & np.isfinite(
        df_overview[
            "delta_C_backward"
        ]
    )
)

print(
    "Cases with complete overview data:",
    int(complete_plot_mask.sum()),
    "of",
    len(df_overview),
)

if complete_plot_mask.sum() == 0:

    raise ValueError(
        "No finite rows are available for the overview plot."
    )


# ------------------------------------------------------------
# Extinction-coefficient colour normalization
# ------------------------------------------------------------

positive_k = df_overview.loc[
    np.isfinite(
        df_overview["absorber_k"]
    )
    & (
        df_overview["absorber_k"]
        > 0.0
    ),
    "absorber_k",
]

if positive_k.empty:

    k_norm = None

else:

    k_norm = mpl.colors.LogNorm(
        vmin=float(
            positive_k.min()
        ),
        vmax=float(
            positive_k.max()
        ),
    )

k_cmap = plt.get_cmap(
    "viridis"
)


# ------------------------------------------------------------
# Create axes
# ------------------------------------------------------------

cm_to_inches = 1.0 / 2.54

fig, axes = plt.subplots(
    nrows=len(
        REFERENCE_WAVELENGTHS
    ),
    ncols=3,
    figsize=(
        OVERVIEW_WIDTH_CM
        * cm_to_inches,
        OVERVIEW_HEIGHT_CM
        * cm_to_inches,
    ),
    sharex=True,
    constrained_layout=True,
)

axes = np.asarray(
    axes,
    dtype=object,
).reshape(
    len(
        REFERENCE_WAVELENGTHS
    ),
    3,
)


y_definitions = [
    {
        "column": (
            "backward_integral_retention_percent"
        ),
        "ylabel": (
            "Backward-integral\nretention [%]"
        ),
        "reference_value": 100.0,
        "criterion_value":
            MIN_BACKWARD_RETENTION_PERCENT,
    },
    {
        "column": (
            "absolute_forward_backward_hue_difference_deg"
        ),
        "ylabel": (
            r"$|\Delta h_{\mathrm{F-B}}|$ [$^\circ$]"
        ),
        "reference_value": None,
        "criterion_value": (
            MAX_FORWARD_BACKWARD_HUE_DIFFERENCE_DEG
        ),
    },
    {
        "column": "delta_C_backward",
        "ylabel": (
            r"$\Delta C_{ab,\mathrm{back}}^*$"
        ),
        "reference_value": 0.0,
        "criterion_value": None,
    },
]


# ------------------------------------------------------------
# Plot every wavelength and family
# ------------------------------------------------------------

number_of_scatter_points = 0

for row_index, wavelength_nm in enumerate(
    REFERENCE_WAVELENGTHS
):

    wavelength_mask = np.isclose(
        df_overview[
            "reference_design_wavelength_nm"
        ],
        float(wavelength_nm),
    )

    wavelength_data = (
        df_overview.loc[
            wavelength_mask
        ]
        .copy()
    )

    print(
        f"{wavelength_nm} nm:",
        len(wavelength_data),
        "cases",
    )

    for column_index, definition in enumerate(
        y_definitions
    ):

        ax = axes[
            row_index,
            column_index,
        ]

        # No change relative to the nonabsorbing reference.
        ax.axvline(
            0.0,
            color="black",
            linewidth=0.8,
            linestyle="--",
            zorder=0,
        )

        # Desired directionality improvement.
        ax.axvline(
            MIN_DIRECTIONAL_IMPROVEMENT_DB,
            color="0.55",
            linewidth=0.7,
            linestyle=":",
            zorder=0,
        )

        if (
            definition[
                "reference_value"
            ]
            is not None
        ):

            ax.axhline(
                definition[
                    "reference_value"
                ],
                color="0.55",
                linewidth=0.7,
                linestyle="--",
                zorder=0,
            )

        if (
            definition[
                "criterion_value"
            ]
            is not None
        ):

            ax.axhline(
                definition[
                    "criterion_value"
                ],
                color="0.65",
                linewidth=0.7,
                linestyle=":",
                zorder=0,
            )

        for case_family, marker in (
            CASE_FAMILY_MARKERS.items()
        ):

            family_mask = (
                wavelength_data[
                    "case_family_clean"
                ]
                == case_family
            )

            family_data = (
                wavelength_data.loc[
                    family_mask
                ]
                .copy()
            )

            if family_data.empty:
                continue

            x_values = family_data[
                "directionality_improvement_db"
            ]

            y_values = family_data[
                definition["column"]
            ]

            finite_mask = (
                np.isfinite(x_values)
                & np.isfinite(y_values)
            )

            family_data = family_data.loc[
                finite_mask
            ]

            x_values = x_values.loc[
                finite_mask
            ]

            y_values = y_values.loc[
                finite_mask
            ]

            if family_data.empty:
                continue

            number_of_scatter_points += len(
                family_data
            )

            if case_family == "reference":

                point_colours = [
                    "black"
                ] * len(family_data)

                point_sizes = np.full(
                    len(family_data),
                    55.0,
                )

                point_alpha = 1.0

            else:

                k_values = family_data[
                    "absorber_k"
                ]

                point_colours = [
                    (
                        k_cmap(
                            k_norm(
                                float(k_value)
                            )
                        )
                        if (
                            k_norm is not None
                            and np.isfinite(
                                k_value
                            )
                            and k_value > 0.0
                        )
                        else "0.65"
                    )
                    for k_value in k_values
                ]

                point_sizes = np.full(
                    len(family_data),
                    30.0,
                )

                point_alpha = 0.82

            ax.scatter(
                x_values,
                y_values,
                s=point_sizes,
                c=point_colours,
                marker=marker,
                edgecolors="black",
                linewidths=0.4,
                alpha=point_alpha,
                zorder=3,
            )

        if column_index == 0:

            ax.set_ylabel(
                (
                    f"{wavelength_nm} nm\n"
                    + definition[
                        "ylabel"
                    ]
                ),
                fontsize=8,
            )

        else:

            ax.set_ylabel(
                definition[
                    "ylabel"
                ],
                fontsize=8,
            )

        if row_index == (
            len(
                REFERENCE_WAVELENGTHS
            )
            - 1
        ):

            ax.set_xlabel(
                (
                    "Directionality improvement\n"
                    "relative to reference [dB]"
                ),
                fontsize=8,
            )

        ax.grid(
            True,
            color="0.90",
            linewidth=0.5,
            zorder=0,
        )


if number_of_scatter_points == 0:

    plt.close(fig)

    raise ValueError(
        "No scatter points were added. "
        "Inspect case_family_clean and numeric plotting columns."
    )

print(
    "Total scatter points added across all axes:",
    number_of_scatter_points,
)


# ------------------------------------------------------------
# Column titles
# ------------------------------------------------------------

axes[0, 0].set_title(
    "Useful backward\nscattering",
    fontsize=9,
)

axes[0, 1].set_title(
    "Forward/backward\nhue agreement",
    fontsize=9,
)

axes[0, 2].set_title(
    "Backward chroma\npreservation",
    fontsize=9,
)


# ------------------------------------------------------------
# Family legend
# ------------------------------------------------------------

family_handles = [
    Line2D(
        [0],
        [0],
        marker=marker,
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=0.5,
        markersize=5.5,
        label=CASE_FAMILY_LABELS[
            case_family
        ],
    )
    for case_family, marker
    in CASE_FAMILY_MARKERS.items()
]

family_legend = fig.legend(
    handles=family_handles,
    loc="outside upper center",
    ncol=3,
    frameon=True,
    fontsize=7,
)


# ------------------------------------------------------------
# Extinction-coefficient colour bar
# ------------------------------------------------------------

if k_norm is not None:

    scalar_mappable = (
        mpl.cm.ScalarMappable(
            norm=k_norm,
            cmap=k_cmap,
        )
    )

    scalar_mappable.set_array([])

    colourbar = fig.colorbar(
        scalar_mappable,
        ax=axes,
        location="right",
        shrink=0.82,
        pad=0.015,
    )

    colourbar.set_label(
        r"Extinction coefficient $k$",
        fontsize=8,
    )


# ------------------------------------------------------------
# Save and display
# ------------------------------------------------------------

if SAVE_DIRECTIONAL_OVERVIEW:

    FIGURE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    overview_png = (
        FIGURE_DIR
        / (
            DIRECTIONAL_OVERVIEW_BASENAME
            + ".png"
        )
    )

    overview_pdf = (
        FIGURE_DIR
        / (
            DIRECTIONAL_OVERVIEW_BASENAME
            + ".pdf"
        )
    )

    fig.canvas.draw()

    fig.savefig(
        overview_png,
        dpi=PNG_DPI,
        transparent=True,
        bbox_inches="tight",
    )

    fig.savefig(
        overview_pdf,
        transparent=True,
        bbox_inches="tight",
    )

    print("Saved:", overview_png)
    print("Saved:", overview_pdf)


plt.show()

Cell 1: Load full checkpoints and inspect their structure

In [ ]:
# ============================================================
# LOAD FULL FORWARD-SUPPRESSION CHECKPOINTS
#
# Each checkpoint is expected to contain:
#
# {
#     "task": task_dictionary,
#     "backward": backward_sweep_result,
#     "forward": forward_sweep_result,
#     "compact": compact_dataframe,
# }
# ============================================================

from pathlib import Path
import pickle
import re
import warnings

import numpy as np
import pandas as pd

from IPython.display import display


FULL_CHECKPOINT_DIRECTORY = (
    OUTPUT_DIR
    / "full_checkpoints"
)

full_checkpoint_paths = sorted(
    FULL_CHECKPOINT_DIRECTORY.glob(
        "task_*.pkl"
    )
)

if not full_checkpoint_paths:
    raise FileNotFoundError(
        "No full checkpoint files were found in:\n"
        f"{FULL_CHECKPOINT_DIRECTORY}"
    )

print(
    "Full checkpoints:",
    len(full_checkpoint_paths),
)


def load_full_checkpoint(path):
    """
    Load one complete forward-suppression checkpoint.
    """

    with path.open("rb") as handle:
        payload = pickle.load(handle)

    required_keys = {
        "task",
        "backward",
        "forward",
        "compact",
    }

    missing_keys = (
        required_keys
        - set(payload.keys())
    )

    if missing_keys:
        raise KeyError(
            f"{path.name} is missing keys: "
            f"{sorted(missing_keys)}"
        )

    return payload


example_checkpoint = load_full_checkpoint(
    full_checkpoint_paths[0]
)

print(
    "Top-level checkpoint keys:",
    list(example_checkpoint.keys()),
)

print(
    "\nExample task:"
)

display(
    pd.DataFrame(
        [example_checkpoint["task"]]
    ).T.rename(
        columns={0: "value"}
    )
)

Cell 2: Robust recursive array extraction helpers

In [ ]:
# ============================================================
# RECURSIVE ARRAY-EXTRACTION HELPERS
#
# These functions inspect dictionaries, objects, lists, and
# dataframes without modifying the stored checkpoint objects.
# ============================================================

def iter_nested_items(
    object_to_inspect,
    *,
    path="root",
    maximum_depth=8,
    current_depth=0,
    visited=None,
):
    """
    Yield nested leaf values together with their access paths.
    """

    if visited is None:
        visited = set()

    if current_depth > maximum_depth:
        return

    object_id = id(
        object_to_inspect
    )

    if object_id in visited:
        return

    visited.add(
        object_id
    )

    # --------------------------------------------------------
    # Dictionary
    # --------------------------------------------------------

    if isinstance(
        object_to_inspect,
        dict,
    ):

        for key, value in (
            object_to_inspect.items()
        ):

            nested_path = (
                f"{path}[{key!r}]"
            )

            yield from iter_nested_items(
                value,
                path=nested_path,
                maximum_depth=maximum_depth,
                current_depth=(
                    current_depth + 1
                ),
                visited=visited,
            )

        return

    # --------------------------------------------------------
    # Pandas dataframe
    # --------------------------------------------------------

    if isinstance(
        object_to_inspect,
        pd.DataFrame,
    ):

        for column in (
            object_to_inspect.columns
        ):

            series = (
                object_to_inspect[
                    column
                ]
            )

            yield (
                f"{path}.dataframe[{column!r}]",
                series,
            )

            for row_index, value in (
                series.items()
            ):

                if isinstance(
                    value,
                    np.ndarray,
                ):

                    yield (
                        (
                            f"{path}.dataframe"
                            f"[{column!r}]"
                            f"[{row_index!r}]"
                        ),
                        value,
                    )

        return

    # --------------------------------------------------------
    # Pandas series
    # --------------------------------------------------------

    if isinstance(
        object_to_inspect,
        pd.Series,
    ):

        yield (
            path,
            object_to_inspect,
        )

        for row_index, value in (
            object_to_inspect.items()
        ):

            if isinstance(
                value,
                np.ndarray,
            ):

                yield (
                    f"{path}[{row_index!r}]",
                    value,
                )

        return

    # --------------------------------------------------------
    # NumPy array
    # --------------------------------------------------------

    if isinstance(
        object_to_inspect,
        np.ndarray,
    ):

        yield (
            path,
            object_to_inspect,
        )

        return

    # --------------------------------------------------------
    # Lists and tuples
    # --------------------------------------------------------

    if isinstance(
        object_to_inspect,
        (list, tuple),
    ):

        for index, value in enumerate(
            object_to_inspect
        ):

            nested_path = (
                f"{path}[{index}]"
            )

            yield from iter_nested_items(
                value,
                path=nested_path,
                maximum_depth=maximum_depth,
                current_depth=(
                    current_depth + 1
                ),
                visited=visited,
            )

        return

    # --------------------------------------------------------
    # Normal Python object
    # --------------------------------------------------------

    if hasattr(
        object_to_inspect,
        "__dict__",
    ):

        for attribute, value in (
            vars(
                object_to_inspect
            ).items()
        ):

            if attribute.startswith(
                "__"
            ):
                continue

            nested_path = (
                f"{path}.{attribute}"
            )

            yield from iter_nested_items(
                value,
                path=nested_path,
                maximum_depth=maximum_depth,
                current_depth=(
                    current_depth + 1
                ),
                visited=visited,
            )


def collect_array_inventory(
    object_to_inspect,
    *,
    root_name="root",
):
    """
    Return a dataframe describing all nested NumPy arrays.
    """

    records = []

    for path, value in iter_nested_items(
        object_to_inspect,
        path=root_name,
    ):

        if not isinstance(
            value,
            np.ndarray,
        ):
            continue

        array = np.asarray(
            value
        )

        record = {
            "path": path,
            "shape": array.shape,
            "ndim": array.ndim,
            "size": array.size,
            "dtype": str(
                array.dtype
            ),
            "minimum": np.nan,
            "maximum": np.nan,
        }

        if (
            array.size > 0
            and np.issubdtype(
                array.dtype,
                np.number,
            )
        ):

            finite = array[
                np.isfinite(array)
            ]

            if finite.size > 0:

                record["minimum"] = float(
                    np.nanmin(finite)
                )

                record["maximum"] = float(
                    np.nanmax(finite)
                )

        records.append(
            record
        )

    return (
        pd.DataFrame(records)
        .sort_values(
            [
                "ndim",
                "size",
                "path",
            ]
        )
        .reset_index(drop=True)
    )


example_backward_inventory = (
    collect_array_inventory(
        example_checkpoint[
            "backward"
        ],
        root_name="backward",
    )
)

example_forward_inventory = (
    collect_array_inventory(
        example_checkpoint[
            "forward"
        ],
        root_name="forward",
    )
)

print(
    "Backward array inventory:"
)

display(
    example_backward_inventory
)

print(
    "Forward array inventory:"
)

display(
    example_forward_inventory
)

In [ ]:
# ============================================================
# REAL NUMERIC ARRAY VALIDATION
# ============================================================

def as_real_numeric_array(
    values,
    *,
    allow_negligible_imaginary=False,
    imaginary_tolerance=1e-12,
):
    """
    Convert an array to float only if the stored quantity is real.

    Complex electromagnetic-field amplitudes are rejected rather
    than silently converted by discarding their imaginary part.

    If allow_negligible_imaginary=True, a complex array is accepted
    only when its imaginary component is negligible relative to its
    real component.
    """

    array = np.asarray(values)

    if not np.issubdtype(
        array.dtype,
        np.number,
    ):
        return None

    if np.iscomplexobj(array):

        if not allow_negligible_imaginary:
            return None

        finite_real = np.real(array)[
            np.isfinite(np.real(array))
        ]

        finite_imaginary = np.imag(array)[
            np.isfinite(np.imag(array))
        ]

        if (
            finite_real.size == 0
            or finite_imaginary.size == 0
        ):
            return None

        real_scale = max(
            float(
                np.nanmax(
                    np.abs(finite_real)
                )
            ),
            1.0,
        )

        maximum_imaginary = float(
            np.nanmax(
                np.abs(finite_imaginary)
            )
        )

        if (
            maximum_imaginary
            > imaginary_tolerance
            * real_scale
        ):
            return None

        array = np.real(array)

    return np.asarray(
        array,
        dtype=float,
    )

Cell 3: Adaptive spectral and angular-array extractor

In [ ]:
# ============================================================
# ADAPTIVE EXTRACTION OF SPECTRAL AND ANGULAR ARRAYS
# ============================================================

WAVELENGTH_KEYWORDS = [
    "wavelength",
    "lambda",
]

COLLECTED_KEYWORDS = [
    "q_collected_geom",
    "collected_geom",
    "collected",
]

THETA_KEYWORDS = [
    "theta",
    "angle",
]

DIFFERENTIAL_EFFICIENCY_KEYWORDS = [
    "dq",
    "differential",
    "d_q",
    "q_sca_angle",
    "qsca_angle",
    "angular_efficiency",
]

DIFFERENTIAL_CROSS_SECTION_KEYWORDS = [
    "differential_cross",
    "dcsca",
    "d_csca",
    "cross_section_angle",
]


def normalized_path_text(
    path,
):
    return (
        str(path)
        .lower()
        .replace(" ", "_")
    )


def score_array_path(
    path,
    *,
    required_keywords,
    preferred_keywords=None,
    rejected_keywords=None,
):
    """
    Score an array path by keyword matching.
    """

    text = normalized_path_text(
        path
    )

    score = 0

    for keyword in required_keywords:

        if keyword.lower() in text:
            score += 10

    if preferred_keywords:

        for keyword in preferred_keywords:

            if keyword.lower() in text:
                score += 2

    if rejected_keywords:

        for keyword in rejected_keywords:

            if keyword.lower() in text:
                score -= 20

    return score


def nested_arrays(
    object_to_inspect,
    *,
    root_name,
):
    """
    Return all nested NumPy arrays as path-array pairs.
    """

    arrays = []

    for path, value in iter_nested_items(
        object_to_inspect,
        path=root_name,
    ):

        if isinstance(
            value,
            np.ndarray,
        ):

            arrays.append(
                (
                    path,
                    np.asarray(value),
                )
            )

    return arrays


def select_best_1d_array(
    object_to_inspect,
    *,
    root_name,
    required_keywords,
    expected_length=None,
    minimum=None,
    maximum=None,
):
    """
    Select the highest-scoring one-dimensional array.
    """

    candidates = []

    for path, array in nested_arrays(
        object_to_inspect,
        root_name=root_name,
    ):

        squeezed = np.squeeze(
            array
        )

        if squeezed.ndim != 1:
            continue

        if (
            expected_length is not None
            and len(squeezed)
            != expected_length
        ):
            continue

        if not np.issubdtype(
            squeezed.dtype,
            np.number,
        ):
            continue

        finite = squeezed[
            np.isfinite(squeezed)
        ]

        if finite.size == 0:
            continue

        if (
            minimum is not None
            and np.nanmin(finite)
            < minimum
        ):
            continue

        if (
            maximum is not None
            and np.nanmax(finite)
            > maximum
        ):
            continue

        score = score_array_path(
            path,
            required_keywords=(
                required_keywords
            ),
        )

        candidates.append(
            (
                score,
                path,
                squeezed.astype(float),
            )
        )

    if not candidates:
        return None, None

    candidates.sort(
        key=lambda item: (
            item[0],
            len(item[2]),
        ),
        reverse=True,
    )

    score, path, array = (
        candidates[0]
    )

    return array, path


def select_wavelength_array(
    sweep_result,
    *,
    root_name,
):
    """
    Select a real one-dimensional wavelength array and return nm.

    Complex arrays are rejected.
    """

    candidates = []

    for path, array in nested_arrays(
        sweep_result,
        root_name=root_name,
    ):

        real_array = as_real_numeric_array(
            array
        )

        if real_array is None:
            continue

        squeezed = np.squeeze(
            real_array
        )

        if squeezed.ndim != 1:
            continue

        if squeezed.size < 10:
            continue

        finite = squeezed[
            np.isfinite(squeezed)
        ]

        if finite.size == 0:
            continue

        minimum_value = float(
            np.nanmin(finite)
        )

        maximum_value = float(
            np.nanmax(finite)
        )

        score = score_array_path(
            path,
            required_keywords=(
                WAVELENGTH_KEYWORDS
            ),
        )

        # Wavelengths stored in metres.
        if (
            minimum_value > 0.0
            and maximum_value < 1e-3
        ):

            wavelength_nm = (
                squeezed * 1e9
            )

            score += 4

        # Wavelengths already stored in nanometres.
        elif (
            minimum_value >= 100.0
            and maximum_value <= 5000.0
        ):

            wavelength_nm = squeezed

            score += 4

        else:
            continue

        if (
            np.nanmin(wavelength_nm)
            <= 400.0
            and np.nanmax(wavelength_nm)
            >= 750.0
        ):
            score += 4

        candidates.append(
            (
                score,
                path,
                wavelength_nm,
            )
        )

    if not candidates:
        raise KeyError(
            f"No real wavelength array could be identified "
            f"in {root_name}."
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            len(item[2]),
        ),
        reverse=True,
    )

    score, path, wavelength_nm = (
        candidates[0]
    )

    return wavelength_nm, path


def select_collected_spectrum(
    sweep_result,
    wavelength_nm,
    *,
    root_name,
):
    """
    Select a real one-dimensional collected-efficiency spectrum.

    Complex field-amplitude arrays are rejected.
    """

    expected_length = len(
        wavelength_nm
    )

    candidates = []

    for path, array in nested_arrays(
        sweep_result,
        root_name=root_name,
    ):

        real_array = as_real_numeric_array(
            array
        )

        if real_array is None:
            continue

        squeezed = np.squeeze(
            real_array
        )

        if squeezed.ndim != 1:
            continue

        if len(squeezed) != expected_length:
            continue

        path_text = normalized_path_text(
            path
        )

        if (
            "wavelength" in path_text
            or "theta" in path_text
            or "angle" in path_text
        ):
            continue

        score = score_array_path(
            path,
            required_keywords=(
                COLLECTED_KEYWORDS
            ),
            preferred_keywords=[
                "q_collected_geom",
                "collected_geom",
                "efficiency",
                "spectrum",
                "integrated",
            ],
            rejected_keywords=[
                "wavelength",
                "theta",
                "angle",
                "field",
                "coefficient",
            ],
        )

        # A collected scattering efficiency should be nonnegative
        # apart from insignificant numerical noise.
        finite = squeezed[
            np.isfinite(squeezed)
        ]

        if finite.size == 0:
            continue

        negative_tolerance = (
            1e-10
            * max(
                1.0,
                float(
                    np.nanmax(
                        np.abs(finite)
                    )
                ),
            )
        )

        if np.nanmin(finite) < -negative_tolerance:
            continue

        candidates.append(
            (
                score,
                path,
                squeezed,
            )
        )

    if not candidates:
        raise KeyError(
            "No real collected spectrum with length "
            f"{expected_length} was identified in {root_name}."
        )

    candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    score, path, spectrum = (
        candidates[0]
    )

    if score <= 0:
        warnings.warn(
            "The collected-spectrum path was selected primarily "
            "from its array dimensions. Verify this path:\n"
            f"{path}"
        )

    # Remove only insignificant negative numerical noise.
    spectrum = np.where(
        (
            spectrum < 0.0
        )
        & np.isclose(
            spectrum,
            0.0,
            atol=1e-10,
        ),
        0.0,
        spectrum,
    )

    return spectrum, path


def extract_directional_spectrum(
    checkpoint_payload,
    direction,
):
    """
    Extract wavelength in nm and the collected spectrum.
    """

    sweep_result = (
        checkpoint_payload[
            direction
        ]
    )

    wavelength_nm, wavelength_path = (
        select_wavelength_array(
            sweep_result,
            root_name=direction,
        )
    )

    spectrum, spectrum_path = (
        select_collected_spectrum(
            sweep_result,
            wavelength_nm,
            root_name=direction,
        )
    )

    return {
        "wavelength_nm": wavelength_nm,
        "spectrum": spectrum,
        "wavelength_path": wavelength_path,
        "spectrum_path": spectrum_path,
    }


example_backward_spectrum = (
    extract_directional_spectrum(
        example_checkpoint,
        "backward",
    )
)

example_forward_spectrum = (
    extract_directional_spectrum(
        example_checkpoint,
        "forward",
    )
)

print(
    "Selected backward wavelength path:"
)

print(
    example_backward_spectrum[
        "wavelength_path"
    ]
)

print(
    "Selected backward spectrum path:"
)

print(
    example_backward_spectrum[
        "spectrum_path"
    ]
)

print(
    "\nSelected forward wavelength path:"
)

print(
    example_forward_spectrum[
        "wavelength_path"
    ]
)

print(
    "Selected forward spectrum path:"
)

print(
    example_forward_spectrum[
        "spectrum_path"
    ]
)

Cell 4: Build a checkpoint index

In [ ]:
# ============================================================
# INDEX ALL FULL CHECKPOINTS
# ============================================================

checkpoint_index_records = []

for checkpoint_path in full_checkpoint_paths:

    payload = load_full_checkpoint(
        checkpoint_path
    )

    task = payload["task"]

    checkpoint_index_records.append(
        {
            "checkpoint_path": (
                checkpoint_path
            ),
            "task_index": task.get(
                "task_index"
            ),
            "task_hash": task.get(
                "task_hash"
            ),
            "reference_design_wavelength_nm": (
                task.get(
                    "reference_design_wavelength_nm"
                )
            ),
            "case_family": task.get(
                "case_family"
            ),
            "scenario": task.get(
                "scenario"
            ),
            "absorber_location": task.get(
                "absorber_location"
            ),
            "absorber_k": task.get(
                "absorber_k"
            ),
            "core_thickness_factor": (
                task.get(
                    "core_thickness_factor"
                )
            ),
            "n_shells": task.get(
                "n_shells"
            ),
            "outer_shell_thickness_nm": (
                task.get(
                    "outer_shell_thickness_nm"
                )
            ),
            "predicted_diameter_nm": (
                task.get(
                    "predicted_diameter_nm"
                )
            ),
        }
    )


df_checkpoint_index = (
    pd.DataFrame(
        checkpoint_index_records
    )
)

numeric_index_columns = [
    "task_index",
    "reference_design_wavelength_nm",
    "absorber_k",
    "core_thickness_factor",
    "n_shells",
    "outer_shell_thickness_nm",
    "predicted_diameter_nm",
]

for column in numeric_index_columns:

    if column in df_checkpoint_index:

        df_checkpoint_index[
            column
        ] = pd.to_numeric(
            df_checkpoint_index[
                column
            ],
            errors="coerce",
        )


print(
    "Indexed checkpoints:",
    len(df_checkpoint_index),
)

display(
    df_checkpoint_index.head()
)

display(
    df_checkpoint_index.groupby(
        [
            "reference_design_wavelength_nm",
            "case_family",
            "scenario",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "number_of_cases"
    )
    .reset_index()
)

Cell 5: Backward and forward spectra for every absorber configuration

This produces one figure per design wavelength. Each column is one absorber configuration:

low-index domains;
all layers;
outer absorbing shell;
core with constant shell number;
core with approximately constant diameter.

The upper row is backward collection and the lower row is forward collection.

For configurations involving an additional geometrical parameter, line style identifies the geometry:

outer shell: coating thickness;
core: core-thickness factor.

Line colour identifies kkk.

In [ ]:
# ============================================================
# BACKWARD AND FORWARD SPECTRA FOR EVERY ABSORBER CONFIGURATION
#
# Automatic y limits:
#   The upper limit is selected from the 99th percentile of all
#   finite plotted points for the corresponding direction and
#   reference wavelength.
#
# Manual limits:
#   Limits can be specified globally or per reference wavelength.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

from matplotlib.lines import Line2D


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SPECTRAL_REFERENCE_WAVELENGTHS = [
    380,
    580,
    760,
]

SPECTRAL_K_VALUES = [
    0.01,
    0.1,
    0.5,
    1.0,
]

SPECTRAL_FIGURE_WIDTH_CM = 18.0
SPECTRAL_FIGURE_HEIGHT_CM = 8.6

SPECTRAL_LINE_WIDTH = 0.85
SPECTRAL_REFERENCE_LINE_WIDTH = 1.4
SPECTRAL_ALPHA = 0.78

SAVE_SPECTRAL_FIGURES = True


# ------------------------------------------------------------
# Y-axis settings
# ------------------------------------------------------------

# Set to 0.99 so that 99% of all finite spectral points lie below
# the automatically selected upper limit.
SPECTRAL_Y_PERCENTILE = 0.99

# Additional headroom above the chosen percentile.
SPECTRAL_Y_HEADROOM_FACTOR = 1.05

# If True, automatic lower limits are fixed at zero.
SPECTRAL_Y_START_AT_ZERO = True


# Global manual overrides.
#
# Use None for automatic selection.
#
# Examples:
# MANUAL_BACKWARD_YLIM = (0.0, 1.5)
# MANUAL_FORWARD_YLIM = (0.0, 8.0)
MANUAL_BACKWARD_YLIM = None
MANUAL_FORWARD_YLIM = None


# Optional wavelength-specific limits.
#
# These override the global limits above.
#
# Examples:
# MANUAL_BACKWARD_YLIMS_BY_WAVELENGTH = {
#     380: (0.0, 1.5),
#     580: (0.0, 1.2),
#     760: (0.0, 1.8),
# }
#
# Use None or omit a wavelength for automatic selection.
MANUAL_BACKWARD_YLIMS_BY_WAVELENGTH = {
    380: None,
    580: None,
    760: None,
}

MANUAL_FORWARD_YLIMS_BY_WAVELENGTH = {
    380: None,
    580: None,
    760: None,
}


spectral_column_definitions = [
    {
        "title": "Low-index domains",
        "case_family": "low_index",
        "scenario": "fixed_reference_geometry",
        "geometry_column": None,
    },
    {
        "title": "All layers",
        "case_family": "all_layers",
        "scenario": "fixed_reference_geometry",
        "geometry_column": None,
    },
    {
        "title": "Additional outer shell",
        "case_family": "outer_shell",
        "scenario": "extra_outer_shell",
        "geometry_column": (
            "outer_shell_thickness_nm"
        ),
    },
    {
        "title": (
            "Core absorber\n"
            "constant shell number"
        ),
        "case_family": "core",
        "scenario": (
            "constant_shell_number"
        ),
        "geometry_column": (
            "core_thickness_factor"
        ),
    },
    {
        "title": (
            "Core absorber "
            "approximately\nconstant diameter"
        ),
        "case_family": "core",
        "scenario": "constant_diameter",
        "geometry_column": (
            "core_thickness_factor"
        ),
    },
]


K_COLOURS = {
    0.01: "#440154",
    0.1: "#31688E",
    0.5: "#35B779",
    1.0: "#FDE725",
}

GEOMETRY_LINESTYLES = [
    "-",
    "--",
    ":",
    "-.",
    (0, (5, 1)),
    (0, (3, 1, 1, 1)),
    (0, (1, 1)),
    (0, (7, 2)),
]


# ============================================================
# HELPERS
# ============================================================

def closest_dictionary_key(
    value,
    dictionary,
):
    """
    Return the numerical dictionary key nearest to value.
    """

    keys = np.asarray(
        list(dictionary.keys()),
        dtype=float,
    )

    return float(
        keys[
            np.argmin(
                np.abs(
                    keys - float(value)
                )
            )
        ]
    )


def select_reference_checkpoint(
    checkpoint_index,
    wavelength_nm,
):
    """
    Select the unique nonabsorbing reference checkpoint.
    """

    mask = (
        np.isclose(
            pd.to_numeric(
                checkpoint_index[
                    "reference_design_wavelength_nm"
                ],
                errors="coerce",
            ),
            float(wavelength_nm),
        )
        & checkpoint_index[
            "case_family"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("reference")
    )

    selected = (
        checkpoint_index.loc[
            mask
        ]
    )

    if len(selected) != 1:
        raise ValueError(
            f"Expected one {wavelength_nm} nm reference "
            f"checkpoint, found {len(selected)}."
        )

    return selected.iloc[0]


def calculate_percentile_ylim(
    spectra,
    *,
    percentile=0.99,
    headroom_factor=1.05,
    start_at_zero=True,
):
    """
    Calculate y limits containing the requested fraction of all
    finite spectral points.

    The percentile is calculated over all wavelengths and all curves
    plotted for one direction and reference wavelength.
    """

    finite_arrays = []

    for spectrum in spectra:

        values = as_real_numeric_array(
            spectrum
        )

        if values is None:
            continue

        finite = values[
            np.isfinite(values)
        ]

        if finite.size > 0:
            finite_arrays.append(
                finite
            )

    if not finite_arrays:
        return (0.0, 1.0)

    all_values = np.concatenate(
        finite_arrays
    )

    if start_at_zero:

        lower_limit = 0.0

        positive_values = all_values[
            all_values >= 0.0
        ]

        if positive_values.size == 0:
            return (0.0, 1.0)

        upper_limit = float(
            np.nanquantile(
                positive_values,
                percentile,
            )
        )

        upper_limit *= (
            headroom_factor
        )

    else:

        tail_fraction = (
            1.0 - percentile
        ) / 2.0

        lower_limit = float(
            np.nanquantile(
                all_values,
                tail_fraction,
            )
        )

        upper_limit = float(
            np.nanquantile(
                all_values,
                1.0 - tail_fraction,
            )
        )

        interval = (
            upper_limit
            - lower_limit
        )

        lower_limit -= (
            0.025 * interval
        )

        upper_limit += (
            0.025 * interval
        )

    if (
        not np.isfinite(
            upper_limit
        )
        or upper_limit <= lower_limit
    ):

        upper_limit = (
            lower_limit + 1.0
        )

    return (
        lower_limit,
        upper_limit,
    )


def resolve_spectral_ylim(
    *,
    wavelength_nm,
    direction,
    spectra,
):
    """
    Select wavelength-specific manual limits, global manual limits,
    or automatic percentile limits, in that order.
    """

    if direction == "backward":

        wavelength_specific = (
            MANUAL_BACKWARD_YLIMS_BY_WAVELENGTH.get(
                wavelength_nm
            )
        )

        global_manual = (
            MANUAL_BACKWARD_YLIM
        )

    elif direction == "forward":

        wavelength_specific = (
            MANUAL_FORWARD_YLIMS_BY_WAVELENGTH.get(
                wavelength_nm
            )
        )

        global_manual = (
            MANUAL_FORWARD_YLIM
        )

    else:

        raise ValueError(
            f"Unknown direction: {direction}"
        )

    if wavelength_specific is not None:

        selected_limits = tuple(
            map(
                float,
                wavelength_specific,
            )
        )

        source = (
            "wavelength-specific manual"
        )

    elif global_manual is not None:

        selected_limits = tuple(
            map(
                float,
                global_manual,
            )
        )

        source = "global manual"

    else:

        selected_limits = (
            calculate_percentile_ylim(
                spectra,
                percentile=(
                    SPECTRAL_Y_PERCENTILE
                ),
                headroom_factor=(
                    SPECTRAL_Y_HEADROOM_FACTOR
                ),
                start_at_zero=(
                    SPECTRAL_Y_START_AT_ZERO
                ),
            )
        )

        source = (
            f"automatic "
            f"{100 * SPECTRAL_Y_PERCENTILE:.1f}%"
        )

    if (
        len(selected_limits) != 2
        or not np.all(
            np.isfinite(
                selected_limits
            )
        )
        or selected_limits[1]
        <= selected_limits[0]
    ):

        raise ValueError(
            f"Invalid {direction} y limits for "
            f"{wavelength_nm} nm: {selected_limits}"
        )

    return (
        selected_limits,
        source,
    )


# ============================================================
# LOAD AND CACHE ALL SPECTRA NEEDED BY THE FIGURES
# ============================================================

spectral_cache = {}


for wavelength_nm in (
    SPECTRAL_REFERENCE_WAVELENGTHS
):

    reference_index_row = (
        select_reference_checkpoint(
            df_checkpoint_index,
            wavelength_nm,
        )
    )

    reference_payload = (
        load_full_checkpoint(
            Path(
                reference_index_row[
                    "checkpoint_path"
                ]
            )
        )
    )

    reference_spectra = {
        direction: (
            extract_directional_spectrum(
                reference_payload,
                direction,
            )
        )
        for direction in (
            "backward",
            "forward",
        )
    }

    wavelength_cache = {
        "reference": (
            reference_spectra
        ),
        "columns": [],
    }

    for definition in (
        spectral_column_definitions
    ):

        selection_mask = (
            np.isclose(
                pd.to_numeric(
                    df_checkpoint_index[
                        "reference_design_wavelength_nm"
                    ],
                    errors="coerce",
                ),
                float(wavelength_nm),
            )
            & df_checkpoint_index[
                "case_family"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq(
                definition[
                    "case_family"
                ]
            )
            & df_checkpoint_index[
                "scenario"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq(
                definition[
                    "scenario"
                ]
            )
        )

        selected_cases = (
            df_checkpoint_index.loc[
                selection_mask
            ]
            .copy()
        )

        geometry_column = (
            definition[
                "geometry_column"
            ]
        )

        if geometry_column is None:

            selected_cases[
                "_geometry_value"
            ] = 0.0

        else:

            selected_cases[
                "_geometry_value"
            ] = pd.to_numeric(
                selected_cases[
                    geometry_column
                ],
                errors="coerce",
            )

        geometry_values = sorted(
            selected_cases[
                "_geometry_value"
            ]
            .dropna()
            .unique()
            .tolist()
        )

        geometry_linestyle_map = {
            geometry_value: (
                GEOMETRY_LINESTYLES[
                    index
                    % len(
                        GEOMETRY_LINESTYLES
                    )
                ]
            )
            for index, geometry_value
            in enumerate(
                geometry_values
            )
        }

        column_cases = []

        for _, case_row in (
            selected_cases.sort_values(
                [
                    "_geometry_value",
                    "absorber_k",
                ]
            ).iterrows()
        ):

            payload = load_full_checkpoint(
                Path(
                    case_row[
                        "checkpoint_path"
                    ]
                )
            )

            extracted_by_direction = {
                direction: (
                    extract_directional_spectrum(
                        payload,
                        direction,
                    )
                )
                for direction in (
                    "backward",
                    "forward",
                )
            }

            column_cases.append(
                {
                    "absorber_k": float(
                        case_row[
                            "absorber_k"
                        ]
                    ),
                    "geometry_value": float(
                        case_row[
                            "_geometry_value"
                        ]
                    ),
                    "spectra": (
                        extracted_by_direction
                    ),
                }
            )

        wavelength_cache[
            "columns"
        ].append(
            {
                "definition": definition,
                "cases": column_cases,
                "geometry_linestyle_map": (
                    geometry_linestyle_map
                ),
            }
        )

    spectral_cache[
        wavelength_nm
    ] = wavelength_cache


# ============================================================
# CREATE FIGURES
# ============================================================

for wavelength_nm in (
    SPECTRAL_REFERENCE_WAVELENGTHS
):

    wavelength_cache = (
        spectral_cache[
            wavelength_nm
        ]
    )

    # --------------------------------------------------------
    # Collect all plotted values for robust y limits
    # --------------------------------------------------------

    all_spectra_by_direction = {
        "backward": [
            wavelength_cache[
                "reference"
            ][
                "backward"
            ][
                "spectrum"
            ]
        ],
        "forward": [
            wavelength_cache[
                "reference"
            ][
                "forward"
            ][
                "spectrum"
            ]
        ],
    }

    for column_cache in (
        wavelength_cache[
            "columns"
        ]
    ):

        for case in (
            column_cache[
                "cases"
            ]
        ):

            for direction in (
                "backward",
                "forward",
            ):

                all_spectra_by_direction[
                    direction
                ].append(
                    case[
                        "spectra"
                    ][
                        direction
                    ][
                        "spectrum"
                    ]
                )

    backward_ylim, backward_source = (
        resolve_spectral_ylim(
            wavelength_nm=wavelength_nm,
            direction="backward",
            spectra=(
                all_spectra_by_direction[
                    "backward"
                ]
            ),
        )
    )

    forward_ylim, forward_source = (
        resolve_spectral_ylim(
            wavelength_nm=wavelength_nm,
            direction="forward",
            spectra=(
                all_spectra_by_direction[
                    "forward"
                ]
            ),
        )
    )

    print(
        f"{wavelength_nm} nm backward ylim: "
        f"{backward_ylim} ({backward_source})"
    )

    print(
        f"{wavelength_nm} nm forward ylim: "
        f"{forward_ylim} ({forward_source})"
    )

    cm_to_inches = 1.0 / 2.54

    fig, axes = plt.subplots(
        2,
        len(
            spectral_column_definitions
        ),
        figsize=(
            SPECTRAL_FIGURE_WIDTH_CM
            * cm_to_inches,
            SPECTRAL_FIGURE_HEIGHT_CM
            * cm_to_inches,
        ),
        sharex=True,
        sharey="row",
        constrained_layout=True,
    )

    for column_index, column_cache in enumerate(
        wavelength_cache[
            "columns"
        ]
    ):

        definition = (
            column_cache[
                "definition"
            ]
        )

        geometry_linestyle_map = (
            column_cache[
                "geometry_linestyle_map"
            ]
        )

        for direction_index, direction in enumerate(
            (
                "backward",
                "forward",
            )
        ):

            ax = axes[
                direction_index,
                column_index,
            ]

            reference = (
                wavelength_cache[
                    "reference"
                ][
                    direction
                ]
            )

            ax.plot(
                reference[
                    "wavelength_nm"
                ],
                reference[
                    "spectrum"
                ],
                color="black",
                linewidth=(
                    SPECTRAL_REFERENCE_LINE_WIDTH
                ),
                label="Nonabsorbing reference",
                zorder=5,
            )

            for case in (
                column_cache[
                    "cases"
                ]
            ):

                k_value = float(
                    case[
                        "absorber_k"
                    ]
                )

                nearest_k = (
                    closest_dictionary_key(
                        k_value,
                        K_COLOURS,
                    )
                )

                geometry_value = float(
                    case[
                        "geometry_value"
                    ]
                )

                extracted = (
                    case[
                        "spectra"
                    ][
                        direction
                    ]
                )

                ax.plot(
                    extracted[
                        "wavelength_nm"
                    ],
                    extracted[
                        "spectrum"
                    ],
                    color=K_COLOURS[
                        nearest_k
                    ],
                    linestyle=(
                        geometry_linestyle_map[
                            geometry_value
                        ]
                    ),
                    linewidth=(
                        SPECTRAL_LINE_WIDTH
                    ),
                    alpha=SPECTRAL_ALPHA,
                    zorder=2,
                )

            if direction_index == 0:

                ax.set_title(
                    definition["title"],
                    fontsize=8,
                )

                ax.set_ylim(
                    *backward_ylim
                )

            else:

                ax.set_ylim(
                    *forward_ylim
                )

            if column_index == 0:

                ax.set_ylabel(
                    (
                        r"$Q_{\mathrm{back}}$"
                        if direction == "backward"
                        else r"$Q_{\mathrm{forward}}$"
                    ),
                    fontsize=8,
                )

            if direction_index == 1:

                ax.set_xlabel(
                    "Wavelength [nm]",
                    fontsize=8,
                )

            ax.set_xlim(
                380.0,
                780.0,
            )

            ax.grid(False)

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    k_handles = [
        Line2D(
            [0],
            [0],
            color=K_COLOURS[
                k_value
            ],
            linewidth=1.4,
            label=rf"$k={k_value:g}$",
        )
        for k_value in SPECTRAL_K_VALUES
    ]

    reference_handle = Line2D(
        [0],
        [0],
        color="black",
        linewidth=(
            SPECTRAL_REFERENCE_LINE_WIDTH
        ),
        label="Nonabsorbing reference",
    )

    fig.legend(
        handles=[
            reference_handle,
            *k_handles,
        ],
        loc="outside lower center",
        ncol=5,
        frameon=True,
        fontsize=7,
    )

    fig.suptitle(
        (
            "Reference design "
            rf"$\lambda_0={wavelength_nm}$ nm"
        ),
        fontsize=9,
    )

    if SAVE_SPECTRAL_FIGURES:

        png_path = (
            FIGURE_DIR
            / (
                "directional_spectra_"
                f"{wavelength_nm}nm.png"
            )
        )

        pdf_path = (
            FIGURE_DIR
            / (
                "directional_spectra_"
                f"{wavelength_nm}nm.pdf"
            )
        )

        fig.savefig(
            png_path,
            dpi=PNG_DPI,
            transparent=True,
            bbox_inches="tight",
        )

        fig.savefig(
            pdf_path,
            transparent=True,
            bbox_inches="tight",
        )

        print("Saved:", png_path)
        print("Saved:", pdf_path)

    plt.show()

Cell 6: Identify theta and differential-scattering arrays

In [ ]:
# ============================================================
# IDENTIFY ANGULAR AND DIFFERENTIAL-SCATTERING ARRAYS
# ============================================================

def select_theta_array(
    sweep_result,
    *,
    root_name,
):
    """
    Select the polar-angle array and return radians.
    """

    candidates = []

    for path, array in nested_arrays(
        sweep_result,
        root_name=root_name,
    ):

        real_array = as_real_numeric_array(
            array
        )

        if real_array is None:
            continue

        squeezed = np.squeeze(
            real_array
        )

        if squeezed.ndim != 1:
            continue

        if squeezed.size < 30:
            continue

        if not np.issubdtype(
            squeezed.dtype,
            np.number,
        ):
            continue

        finite = squeezed[
            np.isfinite(squeezed)
        ]

        if finite.size == 0:
            continue

        minimum_value = float(
            np.nanmin(finite)
        )

        maximum_value = float(
            np.nanmax(finite)
        )

        score = score_array_path(
            path,
            required_keywords=(
                THETA_KEYWORDS
            ),
        )

        if (
            minimum_value >= -1e-8
            and maximum_value
            <= np.pi + 1e-6
        ):

            theta_rad = (squeezed)

            score += 5

        elif (
            minimum_value >= -1e-8
            and maximum_value
            <= 180.0 + 1e-6
        ):

            theta_rad = np.deg2rad(squeezed)

            score += 4

        else:

            continue

        candidates.append(
            (
                score,
                path,
                theta_rad,
            )
        )

    if not candidates:

        raise KeyError(
            f"No polar-angle array was found in {root_name}."
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            len(item[2]),
        ),
        reverse=True,
    )

    score, path, theta_rad = candidates[0]

    return theta_rad, path


def select_differential_array(
    sweep_result,
    *,
    root_name,
    n_wavelengths,
    n_angles,
):
    """
    Select a wavelength-by-angle differential scattering array.

    Preference is given to efficiency-like arrays. If none is found,
    differential cross-section-like names are accepted.
    """

    candidates = []

    for path, array in nested_arrays(
        sweep_result,
        root_name=root_name,
    ):

        real_array = as_real_numeric_array(
            array
        )

        if real_array is None:
            continue

        squeezed = np.squeeze(
            real_array
        )

        if squeezed.ndim != 2:
            continue

        shape = squeezed.shape

        compatible = (
            shape
            in [
                (
                    n_wavelengths,
                    n_angles,
                ),
                (
                    n_angles,
                    n_wavelengths,
                ),
            ]
        )

        if not compatible:
            continue

        path_text = normalized_path_text(
            path
        )

        efficiency_score = (
            score_array_path(
                path,
                required_keywords=(
                    DIFFERENTIAL_EFFICIENCY_KEYWORDS
                ),
                preferred_keywords=[
                    "efficiency",
                    "q",
                    "sca",
                ],
            )
        )

        cross_section_score = (
            score_array_path(
                path,
                required_keywords=(
                    DIFFERENTIAL_CROSS_SECTION_KEYWORDS
                ),
                preferred_keywords=[
                    "cross",
                    "section",
                    "sca",
                ],
            )
        )

        score = max(
            efficiency_score,
            cross_section_score,
        )

        quantity_type = (
            "differential efficiency"
            if efficiency_score
            >= cross_section_score
            else
            "differential cross-section"
        )

        if shape == (
            n_angles,
            n_wavelengths,
        ):

            oriented = (
                squeezed.T
            )

        else:

            oriented = squeezed

        candidates.append(
            (
                score,
                path,
                oriented,
                quantity_type,
            )
        )

    if not candidates:

        raise KeyError(
            "No two-dimensional differential-scattering array "
            f"with shape ({n_wavelengths}, {n_angles}) or "
            f"({n_angles}, {n_wavelengths}) was found."
        )

    candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    score, path, array, quantity_type = (
        candidates[0]
    )

    return array, path, quantity_type


example_sweep_result = (
    example_checkpoint[
        "backward"
    ]
)

example_wavelength_nm, _ = (
    select_wavelength_array(
        example_sweep_result,
        root_name="example",
    )
)

example_theta_rad, example_theta_path = (
    select_theta_array(
        example_sweep_result,
        root_name="example",
    )
)

(
    example_differential,
    example_differential_path,
    example_differential_type,
) = select_differential_array(
    example_sweep_result,
    root_name="example",
    n_wavelengths=len(
        example_wavelength_nm
    ),
    n_angles=len(
        example_theta_rad
    ),
)


print(
    "Selected theta path:"
)

print(
    example_theta_path
)

print(
    "\nSelected differential-scattering path:"
)

print(
    example_differential_path
)

print(
    "\nQuantity type:"
)

print(
    example_differential_type
)

print(
    "\nDifferential array shape:"
)

print(
    example_differential.shape
)

Cell 7: Polar plots at the backward peak and two nearby wavelengths

The following settings select:

nonabsorbing reference;
weak absorption, k=0.01k=0.01k=0.01;
moderate absorption, k=0.1k=0.1k=0.1;
strong absorption, k=1.0k=1.0k=1.0.

The default absorber example is absorption in all layers. Change POLAR_CASE_FAMILY and POLAR_SCENARIO to inspect another configuration.

For outer-shell and core cases, specify POLAR_GEOMETRY_VALUE.

In [ ]:
# ============================================================
# POLAR DIFFERENTIAL-SCATTERING PLOTS
#
# Curves:
#   nonabsorbing reference
#   weak absorption
#   moderate absorption
#   strong absorption
#
# Wavelengths:
#   backward-collected peak wavelength
#   peak - wavelength offset
#   peak + wavelength offset
# ============================================================

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------

POLAR_REFERENCE_WAVELENGTH_NM = 580

POLAR_CASE_FAMILY = "all_layers"
POLAR_SCENARIO = "fixed_reference_geometry"

# Used only for core or outer-shell cases.
#
# Core:
#   geometry column = core_thickness_factor
#
# Outer shell:
#   geometry column = outer_shell_thickness_nm
POLAR_GEOMETRY_VALUE = None

POLAR_K_VALUES = [
    0.01,
    0.1,
    1.0,
]

POLAR_K_LABELS = {
    0.01: "Weak absorption",
    0.1: "Moderate absorption",
    1.0: "Strong absorption",
}

POLAR_WAVELENGTH_OFFSET_NM = 50.0

POLAR_NORMALIZE_EACH_PANEL = False
POLAR_LOGARITHMIC_RADIUS = True

POLAR_WIDTH_CM = 17.5
POLAR_HEIGHT_CM = 6.4

SAVE_POLAR_FIGURE = True


POLAR_COLOURS = {
    0.0: "black",
    0.01: "#377EB8",
    0.1: "#FF7F00",
    1.0: "#E41A1C",
}


def select_absorbing_checkpoint(
    checkpoint_index,
    *,
    wavelength_nm,
    case_family,
    scenario,
    k_value,
    geometry_value=None,
):
    """
    Select one absorber checkpoint.
    """

    mask = (
        np.isclose(
            checkpoint_index[
                "reference_design_wavelength_nm"
            ],
            float(wavelength_nm),
        )
        & checkpoint_index[
            "case_family"
        ]
        .astype(str)
        .str.lower()
        .eq(
            str(case_family).lower()
        )
        & checkpoint_index[
            "scenario"
        ]
        .astype(str)
        .str.lower()
        .eq(
            str(scenario).lower()
        )
        & np.isclose(
            checkpoint_index[
                "absorber_k"
            ],
            float(k_value),
        )
    )

    selected = (
        checkpoint_index.loc[
            mask
        ]
        .copy()
    )

    if geometry_value is not None:

        if case_family == "core":

            geometry_column = (
                "core_thickness_factor"
            )

        elif case_family == "outer_shell":

            geometry_column = (
                "outer_shell_thickness_nm"
            )

        else:

            geometry_column = None

        if geometry_column is not None:

            selected = selected.loc[
                np.isclose(
                    selected[
                        geometry_column
                    ],
                    float(
                        geometry_value
                    ),
                )
            ]

    if len(selected) != 1:

        display(
            selected
        )

        raise ValueError(
            "Expected exactly one selected absorber checkpoint, "
            f"found {len(selected)} for "
            f"{wavelength_nm} nm, family={case_family}, "
            f"scenario={scenario}, k={k_value}, "
            f"geometry={geometry_value}."
        )

    return selected.iloc[0]


# ------------------------------------------------------------
# Load the nonabsorbing reference
# ------------------------------------------------------------

reference_row = select_reference_checkpoint(
    df_checkpoint_index,
    POLAR_REFERENCE_WAVELENGTH_NM,
)

reference_payload = load_full_checkpoint(
    Path(
        reference_row[
            "checkpoint_path"
        ]
    )
)

# Backward and forward payloads should contain the same full
# angular solution. Use the backward result consistently.
reference_result = (
    reference_payload[
        "backward"
    ]
)

reference_wavelength_nm, _ = (
    select_wavelength_array(
        reference_result,
        root_name="reference",
    )
)

reference_theta_rad, _ = (
    select_theta_array(
        reference_result,
        root_name="reference",
    )
)

(
    reference_differential,
    differential_path,
    differential_quantity_type,
) = select_differential_array(
    reference_result,
    root_name="reference",
    n_wavelengths=len(
        reference_wavelength_nm
    ),
    n_angles=len(
        reference_theta_rad
    ),
)


# ------------------------------------------------------------
# Identify the backward-collected peak wavelength of the
# nonabsorbing reference
# ------------------------------------------------------------

reference_collected = (
    extract_directional_spectrum(
        reference_payload,
        "backward",
    )
)

reference_peak_index = int(
    np.nanargmax(
        reference_collected[
            "spectrum"
        ]
    )
)

reference_peak_wavelength_nm = float(
    reference_collected[
        "wavelength_nm"
    ][
        reference_peak_index
    ]
)


target_wavelengths_nm = np.array(
    [
        reference_peak_wavelength_nm
        - POLAR_WAVELENGTH_OFFSET_NM,

        reference_peak_wavelength_nm,

        reference_peak_wavelength_nm
        + POLAR_WAVELENGTH_OFFSET_NM,
    ],
    dtype=float,
)

target_wavelengths_nm = np.clip(
    target_wavelengths_nm,
    np.nanmin(
        reference_wavelength_nm
    ),
    np.nanmax(
        reference_wavelength_nm
    ),
)

print(
    "Reference backward peak:",
    f"{reference_peak_wavelength_nm:.1f} nm",
)

print(
    "Polar-plot wavelengths:",
    target_wavelengths_nm,
)

print(
    "Differential quantity:",
    differential_quantity_type,
)

print(
    "Differential path:",
    differential_path,
)


# ------------------------------------------------------------
# Load reference and absorption cases
# ------------------------------------------------------------

polar_cases = [
    {
        "label": "Nonabsorbing reference",
        "k": 0.0,
        "payload": reference_payload,
    }
]

for k_value in POLAR_K_VALUES:

    selected_row = (
        select_absorbing_checkpoint(
            df_checkpoint_index,
            wavelength_nm=(
                POLAR_REFERENCE_WAVELENGTH_NM
            ),
            case_family=(
                POLAR_CASE_FAMILY
            ),
            scenario=(
                POLAR_SCENARIO
            ),
            k_value=k_value,
            geometry_value=(
                POLAR_GEOMETRY_VALUE
            ),
        )
    )

    payload = load_full_checkpoint(
        Path(
            selected_row[
                "checkpoint_path"
            ]
        )
    )

    polar_cases.append(
        {
            "label": (
                POLAR_K_LABELS[
                    k_value
                ]
                + rf", $k={k_value:g}$"
            ),
            "k": k_value,
            "payload": payload,
        }
    )


# ------------------------------------------------------------
# Extract differential arrays
# ------------------------------------------------------------

polar_case_arrays = []

for case in polar_cases:

    result = (
        case["payload"][
            "backward"
        ]
    )

    wavelength_nm, _ = (
        select_wavelength_array(
            result,
            root_name=(
                case["label"]
            ),
        )
    )

    theta_rad, _ = (
        select_theta_array(
            result,
            root_name=(
                case["label"]
            ),
        )
    )

    differential, _, _ = (
        select_differential_array(
            result,
            root_name=(
                case["label"]
            ),
            n_wavelengths=len(
                wavelength_nm
            ),
            n_angles=len(
                theta_rad
            ),
        )
    )

    polar_case_arrays.append(
        {
            **case,
            "wavelength_nm": (
                wavelength_nm
            ),
            "theta_rad": theta_rad,
            "differential": differential,
        }
    )


# ------------------------------------------------------------
# Create polar plots
# ------------------------------------------------------------

cm_to_inches = 1.0 / 2.54

fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        POLAR_WIDTH_CM
        * cm_to_inches,
        POLAR_HEIGHT_CM
        * cm_to_inches,
    ),
    subplot_kw={
        "projection": "polar",
    },
    constrained_layout=True,
)

axes = np.atleast_1d(
    axes
)


for panel_index, target_wavelength_nm in enumerate(
    target_wavelengths_nm
):

    ax = axes[
        panel_index
    ]

    panel_curves = []

    for case in polar_case_arrays:

        wavelength_index = int(
            np.nanargmin(
                np.abs(
                    case[
                        "wavelength_nm"
                    ]
                    - target_wavelength_nm
                )
            )
        )

        actual_wavelength_nm = float(
            case[
                "wavelength_nm"
            ][
                wavelength_index
            ]
        )

        radial_values = (
            case[
                "differential"
            ][
                wavelength_index,
                :
            ]
        )

        radial_values = np.asarray(
            radial_values,
            dtype=float,
        )

        radial_values[
            radial_values < 0.0
        ] = np.nan

        panel_curves.append(
            {
                "case": case,
                "actual_wavelength_nm": (
                    actual_wavelength_nm
                ),
                "radial_values": (
                    radial_values
                ),
            }
        )

    if POLAR_NORMALIZE_EACH_PANEL:

        panel_maximum = np.nanmax(
            [
                np.nanmax(
                    curve[
                        "radial_values"
                    ]
                )
                for curve in panel_curves
            ]
        )

    else:

        panel_maximum = 1.0

    for curve in panel_curves:

        radial_values = (
            curve[
                "radial_values"
            ]
            / panel_maximum
        )

        k_value = float(
            curve["case"]["k"]
        )

        theta = curve["case"]["theta_rad"]

        theta_full = np.concatenate([
            theta,
            2.0 * np.pi - theta[::-1],
        ])

        radial_full = np.concatenate([
            radial_values,
            radial_values[::-1],
        ])

        ax.plot(
            theta_full,
            radial_full,
            color=POLAR_COLOURS[k_value],
            linewidth=1.2,
            label=curve["case"]["label"],
        )

    # Incident propagation direction at theta = 0.
    ax.set_theta_zero_location(
        "E"
    )

    ax.set_theta_direction(
        1
    )

    ax.set_thetagrids(
        [
            0,
            45,
            90,
            135,
            180,
            225,
            270,
            315,
        ],
        labels=[
            r"$0^\circ$",
            r"$45^\circ$",
            r"$90^\circ$",
            r"$135^\circ$",
            r"$180^\circ$",
            r"$225^\circ$",
            r"$270^\circ$",
            r"$315^\circ$",
        ],
    )

    if POLAR_LOGARITHMIC_RADIUS:

        ax.set_rscale(
            "log"
        )

    ax.set_title(
        (
            f"{target_wavelength_nm:.0f} nm"
        ),
        fontsize=9,
    )

    ax.grid(
        True,
        color="0.80",
        linewidth=0.55,
    )


# ------------------------------------------------------------
# Legend and figure description
# ------------------------------------------------------------

figure_title = (
    f"$\\lambda_0="
    f"{POLAR_REFERENCE_WAVELENGTH_NM}$ nm, "
    f"{POLAR_CASE_FAMILY.replace('_', ' ')}, "
    f"{POLAR_SCENARIO.replace('_', ' ')}"
)

if POLAR_GEOMETRY_VALUE is not None:

    figure_title += (
        f", geometry value "
        f"{POLAR_GEOMETRY_VALUE:g}"
    )

fig.suptitle(
    figure_title,
    fontsize=9,
)

fig.legend(
    handles=axes[0].get_legend_handles_labels()[0],
    labels=axes[0].get_legend_handles_labels()[1],
    loc="outside lower center",
    ncol=2,
    frameon=True,
    fontsize=7,
)


if SAVE_POLAR_FIGURE:

    geometry_suffix = (
        ""
        if POLAR_GEOMETRY_VALUE
        is None
        else (
            "_geometry_"
            f"{POLAR_GEOMETRY_VALUE:g}"
        )
    )

    polar_basename = (
        "polar_differential_scattering_"
        f"{POLAR_REFERENCE_WAVELENGTH_NM}nm_"
        f"{POLAR_CASE_FAMILY}_"
        f"{POLAR_SCENARIO}"
        f"{geometry_suffix}"
    )

    polar_png = (
        FIGURE_DIR
        / f"{polar_basename}.png"
    )

    polar_pdf = (
        FIGURE_DIR
        / f"{polar_basename}.pdf"
    )

    fig.savefig(
        polar_png,
        dpi=PNG_DPI,
        transparent=True,
        bbox_inches="tight",
    )

    fig.savefig(
        polar_pdf,
        transparent=True,
        bbox_inches="tight",
    )

    print(
        "Saved:",
        polar_png,
    )

    print(
        "Saved:",
        polar_pdf,
    )


plt.show()

In [ ]:
# ============================================================
# EXTRACT DIFFERENTIAL ARRAYS FOR PRESENTATION CASES
# ============================================================

polar_case_arrays = []

for case in presentation_polar_cases:

    result = case["payload"]["backward"]

    wavelength_nm, wavelength_path = (
        select_wavelength_array(
            result,
            root_name=case["key"],
        )
    )

    theta_rad, theta_path = (
        select_theta_array(
            result,
            root_name=case["key"],
        )
    )

    (
        differential,
        differential_path,
        differential_quantity_type,
    ) = select_differential_array(
        result,
        root_name=case["key"],
        n_wavelengths=len(
            wavelength_nm
        ),
        n_angles=len(
            theta_rad
        ),
    )

    polar_case_arrays.append(
        {
            **case,
            "wavelength_nm": wavelength_nm,
            "theta_rad": theta_rad,
            "differential": differential,
            "wavelength_path": wavelength_path,
            "theta_path": theta_path,
            "differential_path": differential_path,
            "differential_quantity_type": (
                differential_quantity_type
            ),
        }
    )

print(
    "Prepared presentation cases:",
    [
        case["label"]
        for case in polar_case_arrays
    ],
)

In [ ]:
# ============================================================
# ALL-WAVELENGTH POLAR AND ANGULAR DIFFERENTIAL-SCATTERING PLOTS
#
# This cell uses the arrays already stored in:
#     polar_case_arrays
#
# It creates:
#   1. Polar differential scattering for all wavelengths
#   2. Polar differential scattering at selected wavelengths
#   3. Differential scattering versus theta for all wavelengths
#
# Convention:
#   theta = 0 degrees   -> forward scattering
#   theta = 180 degrees -> backward scattering
# ============================================================

from importlib import reload
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import bragg_onion._notebook_tools.scattering_plots

reload(
    bragg_onion._notebook_tools.scattering_plots
)

from bragg_onion._notebook_tools.scattering_plots import (
    plot_polar_differential_scattering,
    plot_differential_vs_theta,
)


# ============================================================
# USER SETTINGS
# ============================================================

# Select the case by its position in polar_case_arrays.
#
# With your current construction:
#   0 -> nonabsorbing reference
#   1 -> weak absorption, k = 0.01
#   2 -> moderate absorption, k = 0.1
#   3 -> strong absorption, k = 1.0
SELECTED_POLAR_CASE_INDEX = 0

# Alternatively, select by exact or partial label.
# Set to None to use SELECTED_POLAR_CASE_INDEX.
#
# Examples:
# SELECTED_POLAR_CASE_LABEL = "Nonabsorbing reference"
# Prepared presentation cases:
# ['Nonabsorbing reference',
# 'Core absorber, approximately constant diameter\n$k=1$, $\\gamma_\\mathrm{c}=50$',
# 'Core absorber, constant shell number\n$k=1$, $\\gamma_\\mathrm{c}=50$']
SELECTED_POLAR_CASE_LABEL = "Nonabsorbing reference"


# Wavelengths for the selected-wavelength polar diagram.
#
# If None, use:
#   backward reference peak - 150 nm
#   backward reference peak
#   backward reference peak + 150 nm
MANUAL_SELECTED_WAVELENGTHS_NM = None

WAVELENGTH_OFFSET_NM = 150.0


# Plot controls
PLOT_ALL_WAVELENGTH_POLAR = True
PLOT_SELECTED_WAVELENGTH_POLAR = True
PLOT_DIFFERENTIAL_VS_THETA = True

# "db" reproduces the appearance of the attached example.
Y_MODE = "db"

SHOW_SELECTED_WAVELENGTH_LEGEND = True

SAVE_SCATTERING_PLOTS = True

SCATTERING_PLOT_OUTPUT_DIRECTORY = (
    FIGURE_DIR
    / "angular_scattering"
)

SCATTERING_PLOT_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

SCATTERING_PLOT_DPI = 600


# ============================================================
# VALIDATE AVAILABLE CASES
# ============================================================

if (
    "polar_case_arrays" not in globals()
    or len(polar_case_arrays) == 0
):
    raise NameError(
        "polar_case_arrays is unavailable or empty. "
        "Run the preceding checkpoint-selection and differential-array "
        "extraction cells first."
    )


print("Available polar cases:")

for case_index, case in enumerate(
    polar_case_arrays
):
    print(
        f"  {case_index}: "
        f"{case.get('label', '<unlabelled case>')}"
    )


# ============================================================
# SELECT THE REQUESTED CASE
# ============================================================

if SELECTED_POLAR_CASE_LABEL is not None:

    matching_cases = [
        case
        for case in polar_case_arrays
        if (
            SELECTED_POLAR_CASE_LABEL.lower()
            in str(
                case.get(
                    "label",
                    "",
                )
            ).lower()
        )
    ]

    if len(matching_cases) != 1:

        print(
            "\nMatching labels:",
            [
                case.get(
                    "label",
                    "",
                )
                for case in matching_cases
            ],
        )

        raise ValueError(
            "The label selection must match exactly one case. "
            f"Found {len(matching_cases)} matches for "
            f"{SELECTED_POLAR_CASE_LABEL!r}."
        )

    selected_polar_case = (
        matching_cases[0]
    )

else:

    if not (
        0
        <= SELECTED_POLAR_CASE_INDEX
        < len(polar_case_arrays)
    ):
        raise IndexError(
            "SELECTED_POLAR_CASE_INDEX is outside the available "
            f"range 0 to {len(polar_case_arrays) - 1}."
        )

    selected_polar_case = (
        polar_case_arrays[
            SELECTED_POLAR_CASE_INDEX
        ]
    )


selected_case_label = str(
    selected_polar_case.get(
        "label",
        (
            "case_"
            f"{SELECTED_POLAR_CASE_INDEX}"
        ),
    )
)

print(
    "\nSelected case:",
    selected_case_label,
)


# ============================================================
# PREPARE theta, wavelength, AND DIFFERENTIAL ARRAYS
# ============================================================

theta_rad = as_real_numeric_array(
    selected_polar_case[
        "theta_rad"
    ]
)

wavelengths_nm = (
    as_real_numeric_array(
        selected_polar_case[
            "wavelength_nm"
        ]
    )
)

differential_scattering = (
    as_real_numeric_array(
        selected_polar_case[
            "differential"
        ]
    )
)


if theta_rad is None:
    raise TypeError(
        "The selected theta array is complex or nonnumeric."
    )

if wavelengths_nm is None:
    raise TypeError(
        "The selected wavelength array is complex or nonnumeric."
    )

if differential_scattering is None:
    raise TypeError(
        "The selected differential-scattering array is complex "
        "or nonnumeric."
    )


theta_rad = np.squeeze(
    theta_rad
)

wavelengths_nm = np.squeeze(
    wavelengths_nm
)

differential_scattering = np.squeeze(
    differential_scattering
)


if theta_rad.ndim != 1:
    raise ValueError(
        "theta_rad must be one-dimensional, but its shape is "
        f"{theta_rad.shape}."
    )

if wavelengths_nm.ndim != 1:
    raise ValueError(
        "wavelengths_nm must be one-dimensional, but its shape is "
        f"{wavelengths_nm.shape}."
    )

if differential_scattering.ndim != 2:
    raise ValueError(
        "The differential-scattering array must be two-dimensional, "
        f"but its shape is {differential_scattering.shape}."
    )


# Orient the differential array as:
#     wavelength index x theta index
expected_shape = (
    len(wavelengths_nm),
    len(theta_rad),
)

transposed_shape = (
    len(theta_rad),
    len(wavelengths_nm),
)

if differential_scattering.shape == expected_shape:

    dcs_geom = (
        differential_scattering.copy()
    )

elif differential_scattering.shape == transposed_shape:

    dcs_geom = (
        differential_scattering.T.copy()
    )

else:

    raise ValueError(
        "Differential-scattering array dimensions do not match "
        "the wavelength and theta arrays.\n"
        f"Differential shape: {differential_scattering.shape}\n"
        f"Expected: {expected_shape} or {transposed_shape}"
    )


theta_deg = np.rad2deg(
    theta_rad
)


# Make sure theta increases from 0 to 180 degrees.
theta_order = np.argsort(
    theta_deg
)

theta_deg = theta_deg[
    theta_order
]

dcs_geom = dcs_geom[
    :,
    theta_order,
]


# Make sure wavelength increases.
wavelength_order = np.argsort(
    wavelengths_nm
)

wavelengths_nm = wavelengths_nm[
    wavelength_order
]

dcs_geom = dcs_geom[
    wavelength_order,
    :,
]


# Remove nonphysical negative values.
#
# Small negative values may result from numerical noise. They are
# replaced by NaN rather than converted to positive values.
negative_mask = (
    dcs_geom < 0.0
)

number_of_negative_values = int(
    np.count_nonzero(
        negative_mask
    )
)

if number_of_negative_values > 0:

    print(
        "Negative differential-scattering values replaced by NaN:",
        number_of_negative_values,
    )

    dcs_geom[
        negative_mask
    ] = np.nan


if not np.isfinite(
    dcs_geom
).any():
    raise ValueError(
        "The differential-scattering array contains no finite values."
    )


print("\nPrepared angular-scattering data")
print("--------------------------------")
print("Theta range       :", theta_deg[0], "to", theta_deg[-1], "degrees")
print("Wavelength range  :", wavelengths_nm[0], "to", wavelengths_nm[-1], "nm")
print("Number of angles  :", len(theta_deg))
print("Number wavelengths:", len(wavelengths_nm))
print("DCS shape         :", dcs_geom.shape)
print("Finite fraction   :", np.isfinite(dcs_geom).mean())


# ============================================================
# SELECT THREE REPRESENTATIVE WAVELENGTHS
# ============================================================

if (
    "reference_peak_wavelength_nm"
    in globals()
    and np.isfinite(
        reference_peak_wavelength_nm
    )
):

    backward_peak_nm = float(
        reference_peak_wavelength_nm
    )

elif (
    "reference_backward_peak_nm"
    in globals()
    and np.isfinite(
        reference_backward_peak_nm
    )
):

    backward_peak_nm = float(
        reference_backward_peak_nm
    )

else:

    # Recalculate the peak from the reference backward spectrum.
    reference_row = select_reference_checkpoint(
        df_checkpoint_index,
        POLAR_REFERENCE_WAVELENGTH_NM,
    )

    reference_payload_local = (
        load_full_checkpoint(
            Path(
                reference_row[
                    "checkpoint_path"
                ]
            )
        )
    )

    reference_backward_spectrum = (
        extract_directional_spectrum(
            reference_payload_local,
            "backward",
        )
    )

    peak_index = int(
        np.nanargmax(
            reference_backward_spectrum[
                "spectrum"
            ]
        )
    )

    backward_peak_nm = float(
        reference_backward_spectrum[
            "wavelength_nm"
        ][
            peak_index
        ]
    )


if MANUAL_SELECTED_WAVELENGTHS_NM is None:

    requested_selected_wavelengths_nm = np.array(
        [
            backward_peak_nm
            - WAVELENGTH_OFFSET_NM,

            backward_peak_nm,

            backward_peak_nm
            + WAVELENGTH_OFFSET_NM,
        ],
        dtype=float,
    )

else:

    requested_selected_wavelengths_nm = np.asarray(
        MANUAL_SELECTED_WAVELENGTHS_NM,
        dtype=float,
    )


# Select the nearest actually calculated wavelength.
selected_wavelengths_nm = np.array(
    [
        wavelengths_nm[
            np.argmin(
                np.abs(
                    wavelengths_nm
                    - requested_wavelength_nm
                )
            )
        ]
        for requested_wavelength_nm in (
            requested_selected_wavelengths_nm
        )
    ],
    dtype=float,
)

selected_wavelengths_nm = np.unique(
    selected_wavelengths_nm
)


print(
    "\nReference backward peak:",
    f"{backward_peak_nm:.1f} nm",
)

print(
    "Requested selected wavelengths:",
    requested_selected_wavelengths_nm,
)

print(
    "Actual selected wavelengths:",
    selected_wavelengths_nm,
)


# ============================================================
# SAFE OUTPUT NAME
# ============================================================

safe_case_name = "".join(
    character
    if character.isalnum()
    else "_"
    for character in selected_case_label
)

while "__" in safe_case_name:
    safe_case_name = safe_case_name.replace(
        "__",
        "_",
    )

safe_case_name = safe_case_name.strip(
    "_"
).lower()


# ============================================================
# 1. POLAR PLOT WITH ALL WAVELENGTHS
# ============================================================

if PLOT_ALL_WAVELENGTH_POLAR:

    plt.close("all")

    polar_all_result = (
        plot_polar_differential_scattering(
            theta_deg,
            wavelengths_nm,
            dcs_geom,
            plot_all=True,
            y_mode=Y_MODE,
        )
    )

    current_figure = plt.gcf()

    current_figure.suptitle(
        (
            selected_case_label
            + "\nall calculated wavelengths"
        ),
        fontsize=10,
    )

    if SAVE_SCATTERING_PLOTS:

        polar_all_png = (
            SCATTERING_PLOT_OUTPUT_DIRECTORY
            / (
                "polar_all_wavelengths_"
                + safe_case_name
                + ".png"
            )
        )

        polar_all_pdf = (
            SCATTERING_PLOT_OUTPUT_DIRECTORY
            / (
                "polar_all_wavelengths_"
                + safe_case_name
                + ".pdf"
            )
        )

        current_figure.savefig(
            polar_all_png,
            dpi=SCATTERING_PLOT_DPI,
            transparent=False,
            bbox_inches="tight",
        )

        current_figure.savefig(
            polar_all_pdf,
            transparent=False,
            bbox_inches="tight",
        )

        print(
            "\nSaved:",
            polar_all_png,
        )

        print(
            "Saved:",
            polar_all_pdf,
        )

    plt.show()


# ============================================================
# 2. POLAR PLOT AT THE THREE SELECTED WAVELENGTHS
# ============================================================

if PLOT_SELECTED_WAVELENGTH_POLAR:

    plt.close("all")

    polar_selected_result = (
        plot_polar_differential_scattering(
            theta_deg,
            wavelengths_nm,
            dcs_geom,
            plot_all=False,
            selected_wavelengths_nm=(
                selected_wavelengths_nm.tolist()
            ),
            show_legend=(
                SHOW_SELECTED_WAVELENGTH_LEGEND
            ),
            y_mode=Y_MODE,
        )
    )

    current_figure = plt.gcf()

    current_figure.suptitle(
        (
            selected_case_label
            + "\n"
            + (
                "reference backward peak "
                r"and $\pm 150$ nm"
            )
        ),
        fontsize=10,
    )

    if SAVE_SCATTERING_PLOTS:

        polar_selected_png = (
            SCATTERING_PLOT_OUTPUT_DIRECTORY
            / (
                "polar_selected_wavelengths_"
                + safe_case_name
                + ".png"
            )
        )

        polar_selected_pdf = (
            SCATTERING_PLOT_OUTPUT_DIRECTORY
            / (
                "polar_selected_wavelengths_"
                + safe_case_name
                + ".pdf"
            )
        )

        current_figure.savefig(
            polar_selected_png,
            dpi=SCATTERING_PLOT_DPI,
            transparent=False,
            bbox_inches="tight",
        )

        current_figure.savefig(
            polar_selected_pdf,
            transparent=False,
            bbox_inches="tight",
        )

        print(
            "\nSaved:",
            polar_selected_png,
        )

        print(
            "Saved:",
            polar_selected_pdf,
        )

    plt.show()


# ============================================================
# 3. DIFFERENTIAL SCATTERING VERSUS THETA
# ============================================================

if PLOT_DIFFERENTIAL_VS_THETA:

    plt.close("all")

    angular_result = (
        plot_differential_vs_theta(
            theta_deg,
            wavelengths_nm,
            dcs_geom,
            plot_all=True,
            y_mode=Y_MODE,
        )
    )

    current_figure = plt.gcf()

    current_figure.suptitle(
        (
            selected_case_label
            + "\ndifferential scattering versus angle"
        ),
        fontsize=10,
    )

    if SAVE_SCATTERING_PLOTS:

        angular_png = (
            SCATTERING_PLOT_OUTPUT_DIRECTORY
            / (
                "differential_vs_theta_"
                + safe_case_name
                + ".png"
            )
        )

        angular_pdf = (
            SCATTERING_PLOT_OUTPUT_DIRECTORY
            / (
                "differential_vs_theta_"
                + safe_case_name
                + ".pdf"
            )
        )

        current_figure.savefig(
            angular_png,
            dpi=SCATTERING_PLOT_DPI,
            transparent=False,
            bbox_inches="tight",
        )

        current_figure.savefig(
            angular_pdf,
            transparent=False,
            bbox_inches="tight",
        )

        print(
            "\nSaved:",
            angular_png,
        )

        print(
            "Saved:",
            angular_pdf,
        )

    plt.show()

For presentation selective comparison

In [ ]:
# ============================================================
# SELECT PRESENTATION CASES
#
# Cases:
#   1. Nonabsorbing 580 nm reference
#   2. Core absorber, k = 1, approximately constant diameter
#   3. Core absorber, k = 1, constant shell number
#
# By default, the largest core-thickness factor present in both
# core scenarios is selected.
#
# Manual selection can be enabled below.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# USER SETTINGS
# ============================================================

PRESENTATION_REFERENCE_WAVELENGTH_NM = 580.0
PRESENTATION_ABSORBER_K = 1.0

# If None, select the largest core-thickness factor available in
# both scenarios.
MANUAL_COMMON_CORE_THICKNESS_FACTOR = None

# Alternatively, allow different core factors in the two scenarios.
# Set USE_SEPARATE_MANUAL_CORE_FACTORS = True to enable.
USE_SEPARATE_MANUAL_CORE_FACTORS = False

MANUAL_CORE_FACTOR_CONSTANT_DIAMETER = 20.0
MANUAL_CORE_FACTOR_CONSTANT_SHELL_NUMBER = 20.0


# ============================================================
# NORMALIZE CHECKPOINT METADATA
# ============================================================

df_checkpoint_selection = (
    df_checkpoint_index.copy()
)

for column in [
    "reference_design_wavelength_nm",
    "absorber_k",
    "core_thickness_factor",
    "n_shells",
    "predicted_diameter_nm",
]:

    if column in df_checkpoint_selection.columns:

        df_checkpoint_selection[
            column
        ] = pd.to_numeric(
            df_checkpoint_selection[column],
            errors="coerce",
        )


for column in [
    "case_family",
    "scenario",
]:

    if column in df_checkpoint_selection.columns:

        df_checkpoint_selection[
            f"{column}_clean"
        ] = (
            df_checkpoint_selection[column]
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(
                " ",
                "_",
                regex=False,
            )
            .str.replace(
                "-",
                "_",
                regex=False,
            )
        )


# ============================================================
# SELECT NONABSORBING REFERENCE
# ============================================================

reference_mask_presentation = (
    np.isclose(
        df_checkpoint_selection[
            "reference_design_wavelength_nm"
        ],
        PRESENTATION_REFERENCE_WAVELENGTH_NM,
    )
    & df_checkpoint_selection[
        "case_family_clean"
    ].eq("reference")
)

reference_candidates = (
    df_checkpoint_selection.loc[
        reference_mask_presentation
    ]
)

if len(reference_candidates) != 1:

    display(reference_candidates)

    raise ValueError(
        "Expected exactly one nonabsorbing reference checkpoint "
        f"for {PRESENTATION_REFERENCE_WAVELENGTH_NM:.0f} nm, "
        f"but found {len(reference_candidates)}."
    )

presentation_reference_row = (
    reference_candidates.iloc[0]
)


# ============================================================
# FIND AVAILABLE CORE FACTORS
# ============================================================

core_base_mask = (
    np.isclose(
        df_checkpoint_selection[
            "reference_design_wavelength_nm"
        ],
        PRESENTATION_REFERENCE_WAVELENGTH_NM,
    )
    & df_checkpoint_selection[
        "case_family_clean"
    ].eq("core")
    & np.isclose(
        df_checkpoint_selection[
            "absorber_k"
        ],
        PRESENTATION_ABSORBER_K,
    )
)


constant_diameter_candidates = (
    df_checkpoint_selection.loc[
        core_base_mask
        & df_checkpoint_selection[
            "scenario_clean"
        ].eq("constant_diameter")
    ]
    .copy()
)

constant_shell_candidates = (
    df_checkpoint_selection.loc[
        core_base_mask
        & df_checkpoint_selection[
            "scenario_clean"
        ].eq("constant_shell_number")
    ]
    .copy()
)


if constant_diameter_candidates.empty:

    raise ValueError(
        "No approximately constant-diameter core cases were found "
        f"for k={PRESENTATION_ABSORBER_K:g}."
    )

if constant_shell_candidates.empty:

    raise ValueError(
        "No constant-shell-number core cases were found "
        f"for k={PRESENTATION_ABSORBER_K:g}."
    )


available_constant_diameter_factors = np.sort(
    constant_diameter_candidates[
        "core_thickness_factor"
    ]
    .dropna()
    .unique()
)

available_constant_shell_factors = np.sort(
    constant_shell_candidates[
        "core_thickness_factor"
    ]
    .dropna()
    .unique()
)

common_core_factors = np.intersect1d(
    available_constant_diameter_factors,
    available_constant_shell_factors,
)


print(
    "Constant-diameter core factors:",
    available_constant_diameter_factors,
)

print(
    "Constant-shell-number core factors:",
    available_constant_shell_factors,
)

print(
    "Common core factors:",
    common_core_factors,
)


# ============================================================
# CHOOSE CORE FACTORS
# ============================================================

if USE_SEPARATE_MANUAL_CORE_FACTORS:

    selected_constant_diameter_factor = float(
        MANUAL_CORE_FACTOR_CONSTANT_DIAMETER
    )

    selected_constant_shell_factor = float(
        MANUAL_CORE_FACTOR_CONSTANT_SHELL_NUMBER
    )

else:

    if common_core_factors.size == 0:

        raise ValueError(
            "The two scenarios have no common core-thickness factor. "
            "Enable USE_SEPARATE_MANUAL_CORE_FACTORS."
        )

    if MANUAL_COMMON_CORE_THICKNESS_FACTOR is None:

        selected_common_factor = float(
            np.nanmax(common_core_factors)
        )

    else:

        selected_common_factor = float(
            MANUAL_COMMON_CORE_THICKNESS_FACTOR
        )

        if not np.any(
            np.isclose(
                common_core_factors,
                selected_common_factor,
            )
        ):

            raise ValueError(
                "The requested common core factor is unavailable.\n"
                f"Requested: {selected_common_factor:g}\n"
                f"Available: {common_core_factors}"
            )

    selected_constant_diameter_factor = (
        selected_common_factor
    )

    selected_constant_shell_factor = (
        selected_common_factor
    )


# ============================================================
# SELECT UNIQUE CORE CHECKPOINTS
# ============================================================

def select_unique_core_checkpoint(
    candidates,
    *,
    core_factor,
    scenario_label,
):
    """
    Select exactly one core checkpoint at a requested factor.
    """

    selected = candidates.loc[
        np.isclose(
            candidates[
                "core_thickness_factor"
            ],
            float(core_factor),
        )
    ]

    if len(selected) != 1:

        display(selected)

        raise ValueError(
            f"Expected one {scenario_label} core checkpoint at "
            f"factor {core_factor:g}, found {len(selected)}."
        )

    return selected.iloc[0]


presentation_constant_diameter_row = (
    select_unique_core_checkpoint(
        constant_diameter_candidates,
        core_factor=(
            selected_constant_diameter_factor
        ),
        scenario_label="constant-diameter",
    )
)

presentation_constant_shell_row = (
    select_unique_core_checkpoint(
        constant_shell_candidates,
        core_factor=(
            selected_constant_shell_factor
        ),
        scenario_label="constant-shell-number",
    )
)


# ============================================================
# LOAD FULL CHECKPOINTS
# ============================================================

presentation_cases = [
    {
        "key": "reference",
        "label": "Nonabsorbing reference",
        "short_label": "Reference",
        "colour": "black",
        "linestyle": "-",
        "linewidth": 1.8,
        "row": presentation_reference_row,
    },
    {
        "key": "constant_diameter",
        "label": (
            "Core absorber, approximately constant diameter\n"
            rf"$k={PRESENTATION_ABSORBER_K:g}$, "
            rf"$\gamma_\mathrm{{c}}="
            f"{selected_constant_diameter_factor:g}$"
        ),
        "short_label": (
            "Core absorber\n"
            "constant diameter"
        ),
        "colour": "#0072B2",
        "linestyle": "--",
        "linewidth": 1.5,
        "row": presentation_constant_diameter_row,
    },
    {
        "key": "constant_shell_number",
        "label": (
            "Core absorber, constant shell number\n"
            rf"$k={PRESENTATION_ABSORBER_K:g}$, "
            rf"$\gamma_\mathrm{{c}}="
            f"{selected_constant_shell_factor:g}$"
        ),
        "short_label": (
            "Core absorber\n"
            "constant shell number"
        ),
        "colour": "#D55E00",
        "linestyle": "-.",
        "linewidth": 1.5,
        "row": presentation_constant_shell_row,
    },
]


for case in presentation_cases:

    case["payload"] = load_full_checkpoint(
        Path(
            case["row"][
                "checkpoint_path"
            ]
        )
    )

    case["spectra"] = {
        direction: extract_directional_spectrum(
            case["payload"],
            direction,
        )
        for direction in [
            "backward",
            "forward",
        ]
    }


# ============================================================
# CASE SUMMARY
# ============================================================

presentation_case_summary = []

for case in presentation_cases:

    row = case["row"]

    presentation_case_summary.append(
        {
            "case": case["key"],
            "label": case["label"],
            "core_thickness_factor": (
                pd.to_numeric(
                    pd.Series(
                        [
                            row.get(
                                "core_thickness_factor",
                                np.nan,
                            )
                        ]
                    ),
                    errors="coerce",
                ).iloc[0]
            ),
            "n_shells": (
                pd.to_numeric(
                    pd.Series(
                        [
                            row.get(
                                "n_shells",
                                np.nan,
                            )
                        ]
                    ),
                    errors="coerce",
                ).iloc[0]
            ),
            "predicted_diameter_um": (
                pd.to_numeric(
                    pd.Series(
                        [
                            row.get(
                                "predicted_diameter_nm",
                                np.nan,
                            )
                        ]
                    ),
                    errors="coerce",
                ).iloc[0]
                / 1000.0
            ),
            "absorber_k": (
                pd.to_numeric(
                    pd.Series(
                        [
                            row.get(
                                "absorber_k",
                                0.0,
                            )
                        ]
                    ),
                    errors="coerce",
                ).iloc[0]
            ),
            "checkpoint": Path(
                row["checkpoint_path"]
            ).name,
        }
    )


df_presentation_cases = pd.DataFrame(
    presentation_case_summary
)

display(
    df_presentation_cases.style.format(
        {
            "core_thickness_factor": "{:.1f}",
            "n_shells": "{:.0f}",
            "predicted_diameter_um": "{:.3f}",
            "absorber_k": "{:.3g}",
        }
    )
)

Cell 2: Backward and forward spectra for the three selected cases

In [ ]:
# ============================================================
# PRESENTATION FIGURE:
# BACKWARD AND FORWARD COLLECTED SPECTRA
#
# Upper panel:
#   backward-collected efficiency
#
# Lower panel:
#   forward-collected efficiency
#
# Cases:
#   nonabsorbing reference
#   core absorber at approximately constant diameter
#   core absorber at constant shell number
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# SETTINGS
# ============================================================

PRESENTATION_SPECTRA_WIDTH_CM = 13.5
PRESENTATION_SPECTRA_HEIGHT_CM = 10.0

PRESENTATION_XLIM_NM = (
    380.0,
    780.0,
)

# Use None for percentile-based selection.
PRESENTATION_BACKWARD_YLIM = None
PRESENTATION_FORWARD_YLIM = None

PRESENTATION_Y_PERCENTILE = 0.995
PRESENTATION_Y_HEADROOM = 1.08

SAVE_PRESENTATION_SPECTRA = True

PRESENTATION_SPECTRA_BASENAME = (
    "presentation_580nm_core_absorber_spectra"
)


# ============================================================
# DETERMINE Y LIMITS
# ============================================================

presentation_spectra_by_direction = {
    direction: [
        case["spectra"][direction]["spectrum"]
        for case in presentation_cases
    ]
    for direction in [
        "backward",
        "forward",
    ]
}


if PRESENTATION_BACKWARD_YLIM is None:

    backward_ylim = calculate_percentile_ylim(
        presentation_spectra_by_direction["backward"],
        percentile=PRESENTATION_Y_PERCENTILE,
        headroom_factor=PRESENTATION_Y_HEADROOM,
        start_at_zero=True,
    )

else:

    backward_ylim = tuple(
        map(
            float,
            PRESENTATION_BACKWARD_YLIM,
        )
    )


if PRESENTATION_FORWARD_YLIM is None:

    forward_ylim = calculate_percentile_ylim(
        presentation_spectra_by_direction["forward"],
        percentile=PRESENTATION_Y_PERCENTILE,
        headroom_factor=PRESENTATION_Y_HEADROOM,
        start_at_zero=True,
    )

else:

    forward_ylim = tuple(
        map(
            float,
            PRESENTATION_FORWARD_YLIM,
        )
    )


print(
    "Backward y limits:",
    backward_ylim,
)

print(
    "Forward y limits:",
    forward_ylim,
)


# ============================================================
# CREATE FIGURE
# ============================================================

cm_to_inches = 1.0 / 2.54

fig, axes = plt.subplots(
    2,
    1,
    figsize=(
        PRESENTATION_SPECTRA_WIDTH_CM * cm_to_inches,
        PRESENTATION_SPECTRA_HEIGHT_CM * cm_to_inches,
    ),
    sharex=True,
    constrained_layout=True,
)


for axis_index, direction in enumerate(
    [
        "backward",
        "forward",
    ]
):

    ax = axes[axis_index]

    for case in presentation_cases:

        extracted = case["spectra"][direction]

        ax.plot(
            extracted["wavelength_nm"],
            extracted["spectrum"],
            color=case["colour"],
            linestyle=case["linestyle"],
            linewidth=case["linewidth"],
            label=case["label"],
            zorder=(
                5
                if case["key"] == "reference"
                else 3
            ),
        )

    if direction == "backward":

        ax.set_ylabel(
            r"$Q_{\mathrm{back}}$"
        )

        ax.set_ylim(
            *backward_ylim
        )

        ax.set_title(
            "Backward-collected scattering"
        )

    else:

        ax.set_ylabel(
            r"$Q_{\mathrm{forward}}$"
        )

        ax.set_ylim(
            *forward_ylim
        )

        ax.set_title(
            "Forward-collected scattering"
        )

    ax.set_xlim(
        *PRESENTATION_XLIM_NM
    )

    ax.grid(
        False
    )


axes[-1].set_xlabel(
    "Wavelength [nm]"
)


# ============================================================
# LEGEND
# ============================================================

presentation_handles = [
    Line2D(
        [0],
        [0],
        color=case["colour"],
        linestyle=case["linestyle"],
        linewidth=case["linewidth"],
        label=case["label"],
    )
    for case in presentation_cases
]

fig.legend(
    handles=presentation_handles,
    loc="outside lower center",
    ncol=1,
    frameon=True,
    fontsize=8,
)


fig.suptitle(
    (
        r"$\lambda_0=580$ nm: "
        "effect of a strongly absorbing core"
    ),
    fontsize=10,
)


# ============================================================
# SAVE
# ============================================================

if SAVE_PRESENTATION_SPECTRA:

    FIGURE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    spectra_png = (
        FIGURE_DIR
        / (
            PRESENTATION_SPECTRA_BASENAME
            + ".png"
        )
    )

    spectra_pdf = (
        FIGURE_DIR
        / (
            PRESENTATION_SPECTRA_BASENAME
            + ".pdf"
        )
    )

    fig.savefig(
        spectra_png,
        dpi=PNG_DPI,
        transparent=True,
        bbox_inches="tight",
    )

    fig.savefig(
        spectra_pdf,
        transparent=True,
        bbox_inches="tight",
    )

    print("Saved:", spectra_png)
    print("Saved:", spectra_pdf)


plt.show()

Cell 3: Colour matrix for backward and forward spectra

This cell uses the already-calculated hex_backward and hex_forward values from df_paired. The colour patches therefore represent the CIELAB-to-sRGB conversion of the actual unnormalized spectra.

In [ ]:
# # ---------------------------------------------------------------------------
# # FIGURE SETUP
# # ---------------------------------------------------------------------------
# COLOUR_FIGURE_WIDTH_CM = 17.5
# COLOUR_FIGURE_HEIGHT_CM = 6.4

# from matplotlib.patches import Circle
# with mpl.rc_context({
#     "figure.constrained_layout.use": False,
#     "figure.autolayout": False,
# }):

#     fig = plt.figure(
#         figsize=(
#             COLOUR_FIGURE_WIDTH_CM / 2.54,
#             COLOUR_FIGURE_HEIGHT_CM / 2.54,
#         )
#     )

#     gs = fig.add_gridspec(
#         1,
#         2,
#         width_ratios=[0.95, 1.35],
#         left=0.055,
#         right=0.985,
#         bottom=0.14,
#         top=0.86,
#         wspace=0.28,
#     )

#     ax_colour = fig.add_subplot(gs[0, 0])
#     ax_ab = fig.add_subplot(gs[0, 1])

#     # =======================================================================
#     # LEFT PANEL — DIRECTIONAL COLOURS
#     # =======================================================================

#     n_cases = len(case_detail_df)

#     ax_colour.set_xlim(0.0, 3.8)
#     ax_colour.set_ylim(-0.2, n_cases + 0.8)
#     ax_colour.axis("off")

#     ax_colour.set_title(
#         "Calculated directional colours",
#         fontsize=9,
#         fontweight="semibold",
#         pad=12,
#     )

#     ax_colour.text(
#         2.00,
#         n_cases + 0.22,
#         "Backward",
#         ha="center",
#         va="bottom",
#         fontsize=8,
#     )

#     ax_colour.text(
#         3.15,
#         n_cases + 0.22,
#         "Forward",
#         ha="center",
#         va="bottom",
#         fontsize=8,
#     )

#     x_backward = 2.00
#     x_forward = 3.15

#     patch_width = 0.74
#     patch_height = 0.58

#     for i, record in case_detail_df.iterrows():

#         y = n_cases - 1 - i + 0.35

#         # ---------------------------------------------------------------
#         # sample number
#         # ---------------------------------------------------------------

#         ax_colour.text(
#             0.15,
#             y,
#             f"{i+1}",
#             fontsize=9,
#             fontweight="bold",
#             va="center",
#         )

#         # ax_colour.text(
#         #     0.45,
#         #     y,
#         #     record["case"],
#         #     fontsize=8,
#         #     va="center",
#         # )

#         ax_colour.set_xlim(0.0, 4.8)

#         x_backward = 3.0
#         x_forward  = 4.1

#         case_label = (
#             record["case"]
#             .replace("constant diameter", "\nconstant diameter")
#             .replace("constant shell number", "\nconstant shell\nnumber")
#         )

#         ax_colour.text(
#             0.45,
#             y,
#             case_label,
#             fontsize=8,
#             va="center",
#         )
#         # ---------------------------------------------------------------
#         # guide line
#         # ---------------------------------------------------------------

#         ax_colour.plot(
#             [
#                 x_backward + patch_width / 2,
#                 x_forward - patch_width / 2,
#             ],
#             [y, y],
#             color="0.85",
#             linewidth=0.8,
#             zorder=0,
#         )

#         # ---------------------------------------------------------------
#         # backward swatch
#         # ---------------------------------------------------------------

#         ax_colour.add_patch(
#             Rectangle(
#                 (
#                     x_backward - patch_width / 2,
#                     y - patch_height / 2,
#                 ),
#                 patch_width,
#                 patch_height,
#                 facecolor=record["hex_backward"],
#                 edgecolor="black",
#                 linewidth=0.8,
#                 zorder=2,
#             )
#         )

#         # ---------------------------------------------------------------
#         # forward swatch
#         # ---------------------------------------------------------------

#         ax_colour.add_patch(
#             Rectangle(
#                 (
#                     x_forward - patch_width / 2,
#                     y - patch_height / 2,
#                 ),
#                 patch_width,
#                 patch_height,
#                 facecolor=record["hex_forward"],
#                 edgecolor="black",
#                 linewidth=0.8,
#                 zorder=2,
#             )
#         )

#     # =======================================================================
#     # RIGHT PANEL — CIELAB a*-b* PLANE
#     # =======================================================================

#     a_backward = case_detail_df["a_backward"].to_numpy(dtype=float)
#     a_forward = case_detail_df["a_forward"].to_numpy(dtype=float)

#     b_backward = case_detail_df["b_backward"].to_numpy(dtype=float)
#     b_forward = case_detail_df["b_forward"].to_numpy(dtype=float)

#     all_a = np.concatenate([a_backward, a_forward])
#     all_b = np.concatenate([b_backward, b_forward])

#     finite = np.isfinite(all_a) & np.isfinite(all_b)

#     if not np.any(finite):
#         raise ValueError(
#             "No finite CIELAB a*-b* coordinates were available."
#         )

#     # -----------------------------------------------------------------------
#     # LIMITS
#     # -----------------------------------------------------------------------

#     if MANUAL_A_LIMITS is not None:
#         a_limits = tuple(MANUAL_A_LIMITS)
#     else:
#         a_limits = None

#     if MANUAL_B_LIMITS is not None:
#         b_limits = tuple(MANUAL_B_LIMITS)
#     else:
#         b_limits = None

#     if a_limits is None or b_limits is None:

#         minimum = min(
#             np.nanmin(all_a[finite]),
#             np.nanmin(all_b[finite]),
#         )

#         maximum = max(
#             np.nanmax(all_a[finite]),
#             np.nanmax(all_b[finite]),
#         )

#         span = maximum - minimum

#         if span <= 0:
#             span = 1.0

#         centre = 0.5 * (minimum + maximum)

#         half_range = 0.5 * span * (1.0 + AB_LIMIT_HEADROOM)

#         automatic_limits = (
#             centre - half_range,
#             centre + half_range,
#         )

#         if a_limits is None:
#             a_limits = automatic_limits

#         if b_limits is None:
#             b_limits = automatic_limits

#     ax_ab.set_xlim(a_limits)
#     ax_ab.set_ylim(b_limits)

#     ax_ab.set_aspect("equal")

#     # -----------------------------------------------------------------------
#     # GRID + AXES
#     # -----------------------------------------------------------------------

#     ax_ab.grid(
#         True,
#         linestyle="--",
#         linewidth=0.5,
#         alpha=0.35,
#         zorder=0,
#     )

#     ax_ab.axhline(
#         0,
#         color="0.45",
#         linewidth=0.9,
#         zorder=1,
#     )

#     ax_ab.axvline(
#         0,
#         color="0.45",
#         linewidth=0.9,
#         zorder=1,
#     )

#     # -----------------------------------------------------------------------
#     # CHROMA CIRCLES
#     # -----------------------------------------------------------------------

#     max_radius = np.ceil(
#         np.max(np.sqrt(all_a[finite] ** 2 + all_b[finite] ** 2))
#     )

#     for r in np.arange(10, max_radius + 10, 10):

#         ax_ab.add_patch(
#             Circle(
#                 (0, 0),
#                 r,
#                 fill=False,
#                 linestyle=":",
#                 linewidth=0.5,
#                 color="0.75",
#                 zorder=0,
#             )
#         )

#     ax_ab.set_xlabel(r"$a^*$")
#     ax_ab.set_ylabel(r"$b^*$")

#     ax_ab.set_title(
#         r"CIELAB $a^*$-$b^*$ plane",
#         fontsize=9,
#         fontweight="semibold",
#         pad=8,
#     )

#     # -----------------------------------------------------------------------
#     # TRAJECTORIES
#     # -----------------------------------------------------------------------

#     trajectory_colours = [
#         "#1f77b4",
#         "#d62728",
#         "#2ca02c",
#         "#9467bd",
#         "#ff7f0e",
#         "#8c564b",
#     ]

#     for i, record in case_detail_df.iterrows():

#         a_b = record["a_backward"]
#         b_b = record["b_backward"]

#         a_f = record["a_forward"]
#         b_f = record["b_forward"]

#         if not all(np.isfinite([a_b, b_b, a_f, b_f])):
#             continue

#         traj_color = trajectory_colours[
#             i % len(trajectory_colours)
#         ]

#         # ---------------------------------------------------------------
#         # coloured directional arrow
#         # ---------------------------------------------------------------

#         ax_ab.annotate(
#             "",
#             xy=(a_f, b_f),
#             xytext=(a_b, b_b),
#             arrowprops=dict(
#                 arrowstyle="-|>",
#                 color=traj_color,
#                 linewidth=1.6,
#                 alpha=0.85,
#                 shrinkA=6,
#                 shrinkB=6,
#                 mutation_scale=12,
#             ),
#             zorder=2,
#         )

#         # ---------------------------------------------------------------
#         # backward point
#         # ---------------------------------------------------------------

#         ax_ab.scatter(
#             a_b,
#             b_b,
#             s=AB_MARKER_SIZE * 1.25,
#             marker=BACKWARD_MARKER,
#             facecolor=record["hex_backward"],
#             edgecolor="black",
#             linewidth=AB_MARKER_EDGE_WIDTH,
#             zorder=4,
#         )

#         # ---------------------------------------------------------------
#         # forward point
#         # ---------------------------------------------------------------

#         ax_ab.scatter(
#             a_f,
#             b_f,
#             s=AB_MARKER_SIZE * 1.25,
#             marker=FORWARD_MARKER,
#             facecolor=record["hex_forward"],
#             edgecolor="black",
#             linewidth=AB_MARKER_EDGE_WIDTH,
#             zorder=4,
#         )

#         # ---------------------------------------------------------------
#         # number inside marker
#         # ---------------------------------------------------------------

#         ax_ab.text(
#             a_b,
#             b_b,
#             f"{i+1}",
#             color="white",
#             fontsize=6,
#             fontweight="bold",
#             ha="center",
#             va="center",
#             zorder=5,
#         )

#     # =======================================================================
#     # LEGEND
#     # =======================================================================

#     legend_handles = [

#         Line2D(
#             [0],
#             [0],
#             marker=BACKWARD_MARKER,
#             linestyle="None",
#             markerfacecolor="white",
#             markeredgecolor="black",
#             markersize=6.5,
#             label="Backward",
#         ),

#         Line2D(
#             [0],
#             [0],
#             marker=FORWARD_MARKER,
#             linestyle="None",
#             markerfacecolor="white",
#             markeredgecolor="black",
#             markersize=6.5,
#             label="Forward",
#         ),

#         Line2D(
#             [0],
#             [0],
#             color="black",
#             linewidth=1.4,
#             label=r"Backward $\rightarrow$ Forward",
#         ),
#     ]

#     ax_ab.legend(
#         handles=legend_handles,
#         loc="upper left",
#         frameon=False,
#         fontsize=7.5,
#         handletextpad=0.5,
#     )

#     # =======================================================================
#     # FIGURE TITLE
#     # =======================================================================

#     fig.suptitle(
#         r"$\lambda_0 = 580\,\mathrm{nm}$",
#         fontsize=10,
#         y=0.96,
#     )

#     # =======================================================================
#     # SAVE
#     # =======================================================================

#     if SAVE_COLOUR_FIGURE:

#         output_png = FIGURE_DIR / (
#             f"{COLOUR_FIGURE_BASENAME}.png"
#         )

#         output_pdf = FIGURE_DIR / (
#             f"{COLOUR_FIGURE_BASENAME}.pdf"
#         )

#         fig.savefig(
#             output_png,
#             dpi=PNG_DPI,
#             bbox_inches="tight",
#         )

#         fig.savefig(
#             output_pdf,
#             bbox_inches="tight",
#         )

#         print("\nSaved:")
#         print(f"  {output_png}")
#         print(f"  {output_pdf}")

#     plt.show()

Cell 4: Select cases for the polar plots

By default, the polar plots use the same three cases. You can instead select only two cases or manually define other core factors.

In [ ]:
# ============================================================
# SELECT CASES FOR POLAR DIFFERENTIAL-SCATTERING PLOTS
#
# Defaults:
#   reference
#   core absorber, approximately constant diameter
#   core absorber, constant shell number
#
# Set POLAR_SELECTED_CASE_KEYS to any subset of:
#   "reference"
#   "constant_diameter"
#   "constant_shell_number"
# ============================================================

POLAR_SELECTED_CASE_KEYS = [
    "reference",
    "constant_diameter",
    "constant_shell_number",
]

# Example with only two cases:
#
# POLAR_SELECTED_CASE_KEYS = [
#     "reference",
#     "constant_diameter",
# ]


presentation_polar_cases = [
    case
    for case in presentation_cases
    if case["key"] in POLAR_SELECTED_CASE_KEYS
]


if len(presentation_polar_cases) not in [
    2,
    3,
]:

    raise ValueError(
        "Select exactly two or three polar-plot cases."
    )


print(
    "Polar cases:"
)

for case in presentation_polar_cases:

    print(
        "  ",
        case["label"],
    )

Cell 5: Three polar plots at the reference peak and ±150 nm

The peak wavelength is determined from the backward-collected spectrum of the nonabsorbing reference.

In [ ]:
# ============================================================
# POLAR DIFFERENTIAL-SCATTERING COMPARISON
#
# Panels:
#   reference backward peak - 150 nm
#   reference backward peak
#   reference backward peak + 150 nm
#
# Curves:
#   two or three manually selected presentation cases
#
# Convention:
#   0 degrees   = exact forward scattering
#   180 degrees = exact backward scattering
# ============================================================

import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

POLAR_WAVELENGTH_OFFSET_NM = 150.0

# "absolute":
#     use the calculated differential quantity directly
#
# "panel":
#     normalize all curves in each wavelength panel by the largest
#     value within that panel
#
# "curve":
#     normalize each individual curve to its own maximum
POLAR_NORMALIZATION_MODE = "absolute"

POLAR_LOGARITHMIC_RADIUS = True

POLAR_MINIMUM_POSITIVE_FRACTION = 1e-8

POLAR_WIDTH_CM = 15.92
POLAR_HEIGHT_CM = 7

POLAR_LINE_WIDTH = 0.7

SAVE_PRESENTATION_POLAR = True

PRESENTATION_POLAR_BASENAME = (
    "presentation_580nm_core_absorber_polar"
)


# ============================================================
# EXTRACT REFERENCE PEAK
# ============================================================

reference_case = next(
    case
    for case in presentation_cases
    if case["key"] == "reference"
)

reference_backward = (
    reference_case["spectra"]["backward"]
)

reference_peak_index = int(
    np.nanargmax(
        reference_backward["spectrum"]
    )
)

reference_peak_wavelength_nm = float(
    reference_backward["wavelength_nm"][
        reference_peak_index
    ]
)


polar_target_wavelengths_nm = np.array(
    [
        reference_peak_wavelength_nm
        - POLAR_WAVELENGTH_OFFSET_NM,

        reference_peak_wavelength_nm,

        reference_peak_wavelength_nm
        + POLAR_WAVELENGTH_OFFSET_NM,
    ],
    dtype=float,
)


reference_wavelength_minimum = float(
    np.nanmin(
        reference_backward["wavelength_nm"]
    )
)

reference_wavelength_maximum = float(
    np.nanmax(
        reference_backward["wavelength_nm"]
    )
)

polar_target_wavelengths_nm = np.clip(
    polar_target_wavelengths_nm,
    reference_wavelength_minimum,
    reference_wavelength_maximum,
)


print(
    "Reference backward peak wavelength:",
    f"{reference_peak_wavelength_nm:.1f} nm",
)

print(
    "Polar target wavelengths:",
    polar_target_wavelengths_nm,
)


# ============================================================
# EXTRACT ANGULAR ARRAYS
# ============================================================

polar_array_cases = []

selected_quantity_types = set()
selected_differential_paths = set()
selected_theta_paths = set()


for case in presentation_polar_cases:

    # The full angular scattering solution should be available
    # in either directional SweepResult. Use the backward object
    # consistently.
    sweep_result = (
        case["payload"]["backward"]
    )

    wavelength_nm, wavelength_path = (
        select_wavelength_array(
            sweep_result,
            root_name=case["key"],
        )
    )

    theta_rad, theta_path = (
        select_theta_array(
            sweep_result,
            root_name=case["key"],
        )
    )

    (
        differential_array,
        differential_path,
        differential_quantity_type,
    ) = select_differential_array(
        sweep_result,
        root_name=case["key"],
        n_wavelengths=len(wavelength_nm),
        n_angles=len(theta_rad),
    )

    selected_quantity_types.add(
        differential_quantity_type
    )

    selected_differential_paths.add(
        differential_path
    )

    selected_theta_paths.add(
        theta_path
    )

    polar_array_cases.append(
        {
            **case,
            "wavelength_nm": wavelength_nm,
            "wavelength_path": wavelength_path,
            "theta_rad": theta_rad,
            "theta_path": theta_path,
            "differential": differential_array,
            "differential_path": differential_path,
            "differential_quantity_type": (
                differential_quantity_type
            ),
        }
    )


print(
    "\nSelected differential quantity types:",
    selected_quantity_types,
)

print(
    "\nSelected differential paths:"
)

for path in sorted(
    selected_differential_paths
):

    print(
        "  ",
        path,
    )


# ============================================================
# PREPARE CURVES
# ============================================================

polar_panel_curves = []


for target_wavelength_nm in (
    polar_target_wavelengths_nm
):

    panel_curves = []

    for case in polar_array_cases:

        wavelength_index = int(
            np.nanargmin(
                np.abs(
                    case["wavelength_nm"]
                    - target_wavelength_nm
                )
            )
        )

        actual_wavelength_nm = float(
            case["wavelength_nm"][
                wavelength_index
            ]
        )

        radial_values = as_real_numeric_array(
            case["differential"][
                wavelength_index,
                :
            ]
        )

        if radial_values is None:

            raise TypeError(
                "The selected differential-scattering array "
                "is complex or nonnumeric."
            )

        radial_values = np.asarray(
            radial_values,
            dtype=float,
        )

        # Remove negative numerical artefacts.
        radial_values[
            radial_values < 0.0
        ] = np.nan

        panel_curves.append(
            {
                "case": case,
                "actual_wavelength_nm": (
                    actual_wavelength_nm
                ),
                "radial_values": radial_values,
            }
        )

    polar_panel_curves.append(
        {
            "target_wavelength_nm": (
                target_wavelength_nm
            ),
            "curves": panel_curves,
        }
    )


# ============================================================
# NORMALIZATION
# ============================================================

if POLAR_NORMALIZATION_MODE not in [
    "absolute",
    "panel",
    "curve",
]:

    raise ValueError(
        "POLAR_NORMALIZATION_MODE must be "
        "'absolute', 'panel', or 'curve'."
    )


for panel in polar_panel_curves:

    if POLAR_NORMALIZATION_MODE == "panel":

        panel_maximum = float(
            np.nanmax(
                [
                    np.nanmax(
                        curve["radial_values"]
                    )
                    for curve in panel["curves"]
                ]
            )
        )

    else:

        panel_maximum = 1.0

    for curve in panel["curves"]:

        if POLAR_NORMALIZATION_MODE == "curve":

            denominator = float(
                np.nanmax(
                    curve["radial_values"]
                )
            )

        elif POLAR_NORMALIZATION_MODE == "panel":

            denominator = panel_maximum

        else:

            denominator = 1.0

        if (
            not np.isfinite(denominator)
            or denominator <= 0.0
        ):

            denominator = 1.0

        curve["radial_values_plot"] = (
            curve["radial_values"]
            / denominator
        )


# ============================================================
# CREATE POLAR FIGURE
# ============================================================

cm_to_inches = 1.0 / 2.54

fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        POLAR_WIDTH_CM * cm_to_inches,
        POLAR_HEIGHT_CM * cm_to_inches,
    ),
    subplot_kw={
        "projection": "polar",
    },
    constrained_layout=True,
)

axes = np.atleast_1d(axes)


for panel_index, panel in enumerate(
    polar_panel_curves
):

    ax = axes[panel_index]

    all_positive_values = []

    for curve in panel["curves"]:

        radial_values = curve[
            "radial_values_plot"
        ].copy()

        positive = radial_values[
            np.isfinite(radial_values)
            & (radial_values > 0.0)
        ]

        if positive.size > 0:

            all_positive_values.append(
                positive
            )

        theta = curve["case"]["theta_rad"]

        # Mirror the axisymmetric theta = 0...pi result into the
        # lower half-plane for presentation.
        theta_full = np.concatenate(
            [
                theta,
                2.0 * np.pi
                - theta[-2:0:-1],
            ]
        )

        radial_full = np.concatenate(
            [
                radial_values,
                radial_values[-2:0:-1],
            ]
        )

        ax.plot(
            theta_full,
            radial_full,
            color=curve["case"]["colour"],
            linestyle=curve["case"]["linestyle"],
            linewidth=POLAR_LINE_WIDTH,
            label=curve["case"]["label"],
            alpha=0.7,
        )

    ax.set_theta_zero_location("E")

    ax.set_theta_direction(1)

    ax.set_thetagrids(
        [
            0,
            45,
            90,
            135,
            180,
            225,
            270,
            315,
        ],
        labels=[
            r"$0^\circ$",
            r"$45^\circ$",
            r"$90^\circ$",
            r"$135^\circ$",
            r"$180^\circ$",
            r"$225^\circ$",
            r"$270^\circ$",
            r"$315^\circ$",
        ],
    )

    if (
        POLAR_LOGARITHMIC_RADIUS
        and all_positive_values
    ):

        combined_positive = np.concatenate(
            all_positive_values
        )

        maximum_positive = float(
            np.nanmax(
                combined_positive
            )
        )

        minimum_positive = max(
            float(
                np.nanmin(
                    combined_positive
                )
            ),
            maximum_positive
            * POLAR_MINIMUM_POSITIVE_FRACTION,
        )

        ax.set_rscale("log")

        ax.set_ylim(
            minimum_positive,
            maximum_positive * 1.08,
        )

    ax.set_title(
        f"{panel['target_wavelength_nm']:.0f} nm",
        fontsize=9,
    )

    ax.grid(
        True,
        color="0.80",
        linewidth=0.55,
    )


# ============================================================
# LEGEND AND TITLE
# ============================================================

polar_handles = [
    Line2D(
        [0],
        [0],
        color=case["colour"],
        linestyle=case["linestyle"],
        linewidth=POLAR_LINE_WIDTH,
        label=case["label"],
    )
    for case in presentation_polar_cases
]


# fig.legend(
#     handles=polar_handles,
#     loc="outside lower center",
#     ncol=1,
#     frameon=True,
#     fontsize=7,
# )


normalization_description = {
    "absolute": "absolute differential scattering",
    "panel": "panel-normalized differential scattering",
    "curve": "individually normalized angular patterns",
}[POLAR_NORMALIZATION_MODE]


fig.suptitle(
    (
        r"$\lambda_0=580$ nm: "
        + normalization_description
    ),
    fontsize=10,
)


# ============================================================
# SAVE
# ============================================================

if SAVE_PRESENTATION_POLAR:

    FIGURE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    polar_png = (
        FIGURE_DIR
        / (
            PRESENTATION_POLAR_BASENAME
            + ".png"
        )
    )

    polar_pdf = (
        FIGURE_DIR
        / (
            PRESENTATION_POLAR_BASENAME
            + ".pdf"
        )
    )

    fig.savefig(
        polar_png,
        dpi=PNG_DPI,
        transparent=True,
        bbox_inches="tight",
    )

    fig.savefig(
        polar_pdf,
        transparent=True,
        bbox_inches="tight",
    )

    print("Saved:", polar_png)
    print("Saved:", polar_pdf)


plt.show()

Near field analysis

In [ ]:
# ============================================================
# RELOAD THE UPDATED FIELD MODULE
# ============================================================

from importlib import reload

import bragg_onion.fields

reload(
    bragg_onion.fields
)

from bragg_onion.fields import (
    compute_field_map,
)

print(
    "Reloaded:",
    bragg_onion.fields.__file__,
)

In [ ]:
# ============================================================
# CLEAN NEAR-FIELD ANALYSIS SETUP
#
# Cases:
#   1. Reference
#   2. Core absorber, approximately constant sphere diameter
#   3. Core absorber, constant shell number
#
# Absorbing cases:
#   reference design wavelength = 580 nm
#   core-thickness factor = 50
#   core extinction coefficient k = 1
#
# Wavelengths:
#   reference backward peak - 150 nm
#   reference backward peak
#   reference backward peak + 150 nm
# ============================================================

from pathlib import Path
from datetime import timedelta
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.patches import Circle
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import display

from bragg_onion.materials import ConstantDispersion

from bragg_onion.geometry import (
    ExtinctionModifier,
)

from bragg_onion._notebook_tools.field_analysis import (
    rebuild_stack_from_row,
)

from bragg_onion.fields import (
    compute_field_map,
)


# ============================================================
# MAIN USER SETTINGS
# ============================================================

CLEAN_REFERENCE_DESIGN_WAVELENGTH_NM = 580.0

CLEAN_CORE_THICKNESS_FACTOR = 50.0
CLEAN_CORE_EXTINCTION_K = 1.0

CLEAN_WAVELENGTH_OFFSET_NM = 150.0

# Set to a list such as [452, 602, 752] to override the
# automatically determined wavelengths.
CLEAN_MANUAL_WAVELENGTHS_NM = None


# ------------------------------------------------------------
# Calculation grid
# ------------------------------------------------------------

# The same maps are used for both electric-field and Poynting plots.
#
# The Poynting figure uses the complete map.
# The electric-field figure is visually cropped around the sphere.
CLEAN_MAP_EXTENT_FACTOR = 2.0

# Begin at 151. Increase to 201 only after the complete workflow
# has been validated.
CLEAN_MAP_NPTS = 151

CLEAN_N_MEDIUM = 1.0
CLEAN_INCIDENT_E0 = 1.0 + 0.0j

CLEAN_PLANES_TO_COMPUTE = [
    "xy",
    "xz",
]


# ------------------------------------------------------------
# Visual plotting extents
# ------------------------------------------------------------

# Electric-field plot limits are cropped to this multiple of each
# particle's outer radius.
CLEAN_ELECTRIC_DISPLAY_EXTENT_FACTOR = 1.10

# Poynting plots use the complete calculated domain by default.
#
# Use None for the complete calculated map or set a factor such
# as 1.8.
CLEAN_POYNTING_DISPLAY_EXTENT_FACTOR = None


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

CLEAN_OUTPUT_DIRECTORY = (
    FIGURE_DIR
    / "publication_near_field"
)

CLEAN_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

CLEAN_PNG_DPI = 500

CLEAN_CASE_KEYS = [
    "reference",
    "constant_diameter",
    "constant_shell_number",
]

CLEAN_CASE_LABELS = {
    "reference": "Reference",
    "constant_diameter": (
        "Core absorber\nconstant diameter"
    ),
    "constant_shell_number": (
        "Core absorber\nconstant shell number"
    ),
}


# ============================================================
# NORMALIZE CHECKPOINT INDEX
# ============================================================

df_clean_selection = (
    df_checkpoint_index.copy()
)

for column in [
    "reference_design_wavelength_nm",
    "absorber_k",
    "core_thickness_factor",
    "n_shells",
    "predicted_diameter_nm",
]:

    if column in df_clean_selection.columns:

        df_clean_selection[column] = (
            pd.to_numeric(
                df_clean_selection[column],
                errors="coerce",
            )
        )


def normalize_text_series(
    series,
):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(
            " ",
            "_",
            regex=False,
        )
        .str.replace(
            "-",
            "_",
            regex=False,
        )
    )


df_clean_selection[
    "_case_family_clean"
] = normalize_text_series(
    df_clean_selection[
        "case_family"
    ]
)

df_clean_selection[
    "_scenario_clean"
] = normalize_text_series(
    df_clean_selection[
        "scenario"
    ]
)


# ============================================================
# SELECT EXACT CASES
# ============================================================

def select_clean_case(
    *,
    case_family,
    scenario,
    absorber_k=None,
    core_thickness_factor=None,
):
    mask = (
        np.isclose(
            df_clean_selection[
                "reference_design_wavelength_nm"
            ],
            CLEAN_REFERENCE_DESIGN_WAVELENGTH_NM,
        )
        & df_clean_selection[
            "_case_family_clean"
        ].eq(case_family)
        & df_clean_selection[
            "_scenario_clean"
        ].eq(scenario)
    )

    if absorber_k is not None:

        mask &= np.isclose(
            df_clean_selection[
                "absorber_k"
            ],
            float(absorber_k),
        )

    if core_thickness_factor is not None:

        mask &= np.isclose(
            df_clean_selection[
                "core_thickness_factor"
            ],
            float(
                core_thickness_factor
            ),
        )

    selected = (
        df_clean_selection.loc[
            mask
        ]
        .copy()
    )

    if len(selected) != 1:

        display(selected)

        raise ValueError(
            "Expected exactly one selected checkpoint, "
            f"found {len(selected)} for "
            f"family={case_family!r}, "
            f"scenario={scenario!r}, "
            f"k={absorber_k}, "
            f"core factor={core_thickness_factor}."
        )

    return selected.iloc[0]


clean_checkpoint_rows = {
    "reference": select_clean_case(
        case_family="reference",
        scenario="reference",
    ),
    "constant_diameter": select_clean_case(
        case_family="core",
        scenario="constant_diameter",
        absorber_k=CLEAN_CORE_EXTINCTION_K,
        core_thickness_factor=(
            CLEAN_CORE_THICKNESS_FACTOR
        ),
    ),
    "constant_shell_number": select_clean_case(
        case_family="core",
        scenario="constant_shell_number",
        absorber_k=CLEAN_CORE_EXTINCTION_K,
        core_thickness_factor=(
            CLEAN_CORE_THICKNESS_FACTOR
        ),
    ),
}


# ============================================================
# MATCH CHECKPOINTS TO COMPLETE RESULT ROWS
# ============================================================

def match_clean_checkpoint_to_paired_row(
    checkpoint_row,
):
    if (
        "task_hash" in df_paired.columns
        and pd.notna(
            checkpoint_row.get(
                "task_hash",
                np.nan,
            )
        )
    ):

        selected = df_paired.loc[
            df_paired[
                "task_hash"
            ].astype(str)
            == str(
                checkpoint_row[
                    "task_hash"
                ]
            )
        ].copy()

    elif (
        "task_index" in df_paired.columns
        and pd.notna(
            checkpoint_row.get(
                "task_index",
                np.nan,
            )
        )
    ):

        selected = df_paired.loc[
            np.isclose(
                pd.to_numeric(
                    df_paired[
                        "task_index"
                    ],
                    errors="coerce",
                ),
                float(
                    checkpoint_row[
                        "task_index"
                    ]
                ),
            )
        ].copy()

    else:

        raise KeyError(
            "Neither task_hash nor task_index can be used "
            "to match the selected checkpoint."
        )

    if len(selected) != 1:

        display(selected)

        raise ValueError(
            "Expected exactly one paired row, "
            f"found {len(selected)}."
        )

    return selected.iloc[0]


def construct_clean_reconstruction_row(
    checkpoint_row,
):
    payload = load_full_checkpoint(
        Path(
            checkpoint_row[
                "checkpoint_path"
            ]
        )
    )

    paired_row = (
        match_clean_checkpoint_to_paired_row(
            checkpoint_row
        )
    )

    combined = {}

    combined.update(
        dict(
            payload.get(
                "task",
                {},
            )
        )
    )

    combined.update(
        checkpoint_row.to_dict()
    )

    combined.update(
        paired_row.to_dict()
    )

    row = pd.Series(
        combined,
        dtype=object,
    )

    # Peak/design wavelength
    if (
        "peak_wavelength_m" not in row.index
        or not np.isfinite(
            pd.to_numeric(
                pd.Series(
                    [
                        row.get(
                            "peak_wavelength_m",
                            np.nan,
                        )
                    ]
                ),
                errors="coerce",
            ).iloc[0]
        )
    ):

        row[
            "peak_wavelength_m"
        ] = (
            float(
                row[
                    "reference_design_wavelength_nm"
                ]
            )
            * 1e-9
        )

    # Required layer-count alias
    if (
        "n_layers" not in row.index
        or not np.isfinite(
            pd.to_numeric(
                pd.Series(
                    [
                        row.get(
                            "n_layers",
                            np.nan,
                        )
                    ]
                ),
                errors="coerce",
            ).iloc[0]
        )
    ):

        row[
            "n_layers"
        ] = int(
            row[
                "n_shells"
            ]
        )

    # Outermost layer
    valid_outer_layer = None

    for column in [
        "outer_layer",
        "actual_outer_layer",
        "reference_outer_layer",
    ]:

        candidate = str(
            row.get(
                column,
                "",
            )
        ).strip().upper()

        if candidate in {
            "A",
            "B",
        }:

            valid_outer_layer = candidate
            break

    if valid_outer_layer is None:

        valid_outer_layer = "A"

        warnings.warn(
            "No explicit outer-layer label was found. "
            "Using outer_layer='A'."
        )

    row[
        "outer_layer"
    ] = valid_outer_layer

    return row


clean_reconstruction_rows = {
    case_key: (
        construct_clean_reconstruction_row(
            checkpoint_row
        )
    )
    for case_key, checkpoint_row
    in clean_checkpoint_rows.items()
}


# ============================================================
# DETERMINE THE THREE WAVELENGTHS
# ============================================================

reference_payload = load_full_checkpoint(
    Path(
        clean_checkpoint_rows[
            "reference"
        ][
            "checkpoint_path"
        ]
    )
)

reference_backward_spectrum = (
    extract_directional_spectrum(
        reference_payload,
        "backward",
    )
)

reference_peak_index = int(
    np.nanargmax(
        reference_backward_spectrum[
            "spectrum"
        ]
    )
)

clean_reference_peak_nm = float(
    reference_backward_spectrum[
        "wavelength_nm"
    ][
        reference_peak_index
    ]
)


if CLEAN_MANUAL_WAVELENGTHS_NM is None:

    clean_wavelengths_nm = np.array(
        [
            clean_reference_peak_nm
            - CLEAN_WAVELENGTH_OFFSET_NM,

            clean_reference_peak_nm,

            clean_reference_peak_nm
            + CLEAN_WAVELENGTH_OFFSET_NM,
        ],
        dtype=float,
    )

else:

    clean_wavelengths_nm = np.asarray(
        CLEAN_MANUAL_WAVELENGTHS_NM,
        dtype=float,
    )


available_spectral_minimum_nm = float(
    np.nanmin(
        reference_backward_spectrum[
            "wavelength_nm"
        ]
    )
)

available_spectral_maximum_nm = float(
    np.nanmax(
        reference_backward_spectrum[
            "wavelength_nm"
        ]
    )
)

clean_wavelengths_nm = np.clip(
    clean_wavelengths_nm,
    available_spectral_minimum_nm,
    available_spectral_maximum_nm,
)

clean_wavelengths_nm = np.unique(
    clean_wavelengths_nm
)


# ============================================================
# REBUILD THE THREE STACKS
# ============================================================

clean_field_stacks = {}


for case_key, row in (
    clean_reconstruction_rows.items()
):

    material_A = (
        ConstantDispersion.from_nk(
            name="A",
            n=float(
                row[
                    "n_A"
                ]
            ),
            k=0.0,
            source=(
                "Lossless A for clean near-field analysis"
            ),
        )
    )

    material_B = (
        ConstantDispersion.from_nk(
            name="B",
            n=float(
                row[
                    "n_B"
                ]
            ),
            k=0.0,
            source=(
                "Lossless B for clean near-field analysis"
            ),
        )
    )

    if case_key == "reference":

        extinction_modifiers = None

    else:

        extinction_modifiers = (
            ExtinctionModifier(
                target="core",
                mode="substitute",
                profile=float(
                    CLEAN_CORE_EXTINCTION_K
                ),
                name=(
                    "core_only_"
                    f"k_{CLEAN_CORE_EXTINCTION_K:g}"
                ),
            ),
        )

    geometry, stack = (
        rebuild_stack_from_row(
            row,
            material_a=material_A,
            material_b=material_B,
            fixed_geometry_kwargs={},
            extinction_modifiers=(
                extinction_modifiers
            ),
            extra_outer_shell=None,
        )
    )

    clean_field_stacks[
        case_key
    ] = {
        "geometry": geometry,
        "stack": stack,
        "row": row,
    }


# ============================================================
# SUMMARY
# ============================================================

summary_records = []

for case_key in CLEAN_CASE_KEYS:

    row = clean_reconstruction_rows[
        case_key
    ]

    geometry = clean_field_stacks[
        case_key
    ][
        "geometry"
    ]

    summary_records.append(
        {
            "case": CLEAN_CASE_LABELS[
                case_key
            ].replace(
                "\n",
                " ",
            ),
            "n_layers": int(
                row[
                    "n_layers"
                ]
            ),
            "core_factor": float(
                row[
                    "core_thickness_factor"
                ]
            ),
            "absorber_k": (
                0.0
                if case_key == "reference"
                else CLEAN_CORE_EXTINCTION_K
            ),
            "outer_radius_um": (
                float(
                    geometry.outer_radius_m
                )
                * 1e6
            ),
            "diameter_um": (
                2.0
                * float(
                    geometry.outer_radius_m
                )
                * 1e6
            ),
        }
    )


print(
    "Reference backward peak:",
    f"{clean_reference_peak_nm:.1f} nm",
)

print(
    "Selected wavelengths:",
    clean_wavelengths_nm,
)

display(
    pd.DataFrame(
        summary_records
    ).style.format(
        {
            "n_layers": "{:.0f}",
            "core_factor": "{:.1f}",
            "absorber_k": "{:.3g}",
            "outer_radius_um": "{:.3f}",
            "diameter_um": "{:.3f}",
        }
    )
)

print(
    "Output directory:",
    CLEAN_OUTPUT_DIRECTORY,
)

Cell 2: Calculate or reuse the required field maps

This computes 18 maps:

3 cases×3 wavelengths×2 planes=18 maps.3\ \text{cases} \times 3\ \text{wavelengths} \times 2\ \text{planes} = 18\ \text{maps}.3 cases×3 wavelengths×2 planes=18 maps.

The xz maps are used for both electric-field and Poynting figures.

In [ ]:
# ============================================================
# COMPUTE CLEAN FIELD MAPS
#
# The same extended map is used for:
#   - cropped electric-field figures;
#   - full-domain Poynting-flow figures.
# ============================================================

CLEAN_RECOMPUTE_MAPS = False

clean_field_maps = (
    {}
    if CLEAN_RECOMPUTE_MAPS
    else globals().get(
        "clean_field_maps",
        {},
    )
)


requested_clean_keys = [
    (
        case_key,
        float(
            wavelength_nm
        ),
        plane,
    )
    for case_key in CLEAN_CASE_KEYS
    for wavelength_nm in clean_wavelengths_nm
    for plane in CLEAN_PLANES_TO_COMPUTE
]


missing_clean_keys = [
    key
    for key in requested_clean_keys
    if (
        CLEAN_RECOMPUTE_MAPS
        or key not in clean_field_maps
    )
]


print(
    "Requested maps:",
    len(
        requested_clean_keys
    ),
)

print(
    "Already available:",
    len(
        requested_clean_keys
    )
    - len(
        missing_clean_keys
    ),
)

print(
    "Remaining:",
    len(
        missing_clean_keys
    ),
)


completed_times = []


for map_index, (
    case_key,
    wavelength_nm,
    plane,
) in enumerate(
    missing_clean_keys,
    start=1,
):

    map_key = (
        case_key,
        wavelength_nm,
        plane,
    )

    print(
        "\n"
        f"[{map_index}/{len(missing_clean_keys)}] "
        f"Computing {map_key}",
        flush=True,
    )

    map_start = time.perf_counter()

    clean_field_maps[
        map_key
    ] = compute_field_map(
        stack=clean_field_stacks[
            case_key
        ][
            "stack"
        ],
        wavelength_m=(
            wavelength_nm
            * 1e-9
        ),
        n_medium=CLEAN_N_MEDIUM,
        plane=plane,
        npts=CLEAN_MAP_NPTS,
        extent_outer_radius_factor=(
            CLEAN_MAP_EXTENT_FACTOR
        ),
        incident_e0=(
            CLEAN_INCIDENT_E0
        ),
    )

    elapsed = (
        time.perf_counter()
        - map_start
    )

    completed_times.append(
        elapsed
    )

    mean_time = float(
        np.mean(
            completed_times
        )
    )

    remaining_count = (
        len(
            missing_clean_keys
        )
        - map_index
    )

    print(
        "Completed in:",
        timedelta(
            seconds=int(
                elapsed
            )
        ),
        flush=True,
    )

    print(
        "Estimated remaining:",
        timedelta(
            seconds=int(
                mean_time
                * remaining_count
            )
        ),
        flush=True,
    )


print(
    "\nStored clean maps:",
    len(
        clean_field_maps
    ),
)

Cell 3: Publication-ready total electric-field figures

This cell creates one 2×32\times32×3 figure for every selected wavelength:

top row: xy;
bottom row: xz;
three columns: the three physical cases.

Each panel receives an individual colorbar with exactly the same height as its image axis. The color limits are determined independently per panel using the requested 0th and 99th percentiles.

In [ ]:
# ============================================================
# SHARED NEAR-FIELD PLOTTING UTILITIES
#
# Features:
#   - stable logarithmic conversion without overflow;
#   - common color limits across panels;
#   - common colorbar without overlap;
#   - support for total, delta, and scattered-like Poynting flow;
#   - incident-edge and incident-facing hemisphere seeds.
# ============================================================

import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.patches import Circle
from matplotlib.ticker import MaxNLocator


# ============================================================
# STABLE LOGARITHMIC TRANSFORMS
# ============================================================

def stable_relative_db(
    values,
    reference_value,
    *,
    floor_fraction=1e-8,
    saturate_above_reference=True,
):
    """
    Calculate 10 log10(values / reference_value) without explicitly
    dividing very large values by a small reference.

    This avoids:
        RuntimeWarning: overflow encountered in divide
    """

    values = np.asarray(
        values,
        dtype=float,
    )

    reference_value = float(
        reference_value
    )

    if (
        not np.isfinite(reference_value)
        or reference_value <= 0.0
    ):
        raise ValueError(
            "reference_value must be finite and positive."
        )

    floor_fraction = float(
        floor_fraction
    )

    if not (
        0.0
        < floor_fraction
        < 1.0
    ):
        raise ValueError(
            "floor_fraction must lie between 0 and 1."
        )

    valid = (
        np.isfinite(values)
        & (values > 0.0)
    )

    result = np.full_like(
        values,
        np.nan,
        dtype=float,
    )

    log_reference = np.log10(
        reference_value
    )

    log_floor = (
        log_reference
        + np.log10(
            floor_fraction
        )
    )

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore",
    ):

        log_values = np.log10(
            values[
                valid
            ]
        )

        log_values = np.maximum(
            log_values,
            log_floor,
        )

        result[
            valid
        ] = (
            10.0
            * (
                log_values
                - log_reference
            )
        )

    if saturate_above_reference:

        result = np.minimum(
            result,
            0.0,
        )

    return result


def stable_field_db(
    magnitude,
    *,
    floor=1e-30,
):
    """
    Calculate 20 log10(|E|) safely.
    """

    magnitude = np.asarray(
        magnitude,
        dtype=float,
    )

    result = np.full_like(
        magnitude,
        np.nan,
        dtype=float,
    )

    valid = (
        np.isfinite(magnitude)
        & (magnitude >= 0.0)
    )

    result[
        valid
    ] = (
        20.0
        * np.log10(
            np.maximum(
                magnitude[
                    valid
                ],
                float(floor),
            )
        )
    )

    return result


# ============================================================
# COMMON COLORBAR
# ============================================================

def add_shared_colorbar(
    fig,
    axes,
    mappable,
    *,
    label,
    tick_count=6,
    pad=0.025,
    fraction=0.035,
    shrink=0.96,
):
    """
    Add one colorbar shared by all supplied axes.

    The colorbar spans the combined plotting area and is kept
    separate from neighboring panels.
    """

    axes_array = np.asarray(
        axes,
        dtype=object,
    )

    colorbar = fig.colorbar(
        mappable,
        ax=axes_array.ravel().tolist(),
        location="right",
        fraction=fraction,
        pad=pad,
        shrink=shrink,
        aspect=35,
    )

    colorbar.set_label(
        label,
        fontsize=8,
        labelpad=5,
    )

    colorbar.ax.tick_params(
        labelsize=7,
        length=2.5,
        width=0.6,
        pad=2,
    )

    colorbar.locator = MaxNLocator(
        nbins=tick_count
    )

    colorbar.update_ticks()

    return colorbar


# ============================================================
# BOUNDARIES
# ============================================================

def add_shell_and_sphere_boundaries(
    ax,
    field_map,
    *,
    show_shells=True,
    shell_color="white",
    shell_linewidth=0.16,
    shell_alpha=0.20,
    sphere_color="white",
    sphere_linewidth=0.95,
    sphere_alpha=0.95,
):
    """
    Draw shell interfaces and emphasize the outer sphere boundary.
    """

    if show_shells:

        for radius_m in (
            field_map.radii_m[:-1]
        ):

            ax.add_patch(
                Circle(
                    (0.0, 0.0),
                    radius=(
                        float(radius_m)
                        * 1e9
                    ),
                    fill=False,
                    edgecolor=shell_color,
                    linewidth=shell_linewidth,
                    alpha=shell_alpha,
                    zorder=6,
                )
            )

    ax.add_patch(
        Circle(
            (0.0, 0.0),
            radius=(
                field_map.outer_radius_m
                * 1e9
            ),
            fill=False,
            edgecolor=sphere_color,
            linewidth=sphere_linewidth,
            alpha=sphere_alpha,
            zorder=7,
        )
    )


# ============================================================
# POYNTING QUANTITY SELECTION
# ============================================================

def get_poynting_magnitude(
    field_map,
    *,
    flow_kind,
):
    """
    Return the selected Poynting magnitude.
    """

    if flow_kind == "total":

        return np.asarray(
            field_map.s_total_mag,
            dtype=float,
        )

    if flow_kind == "delta":

        return np.asarray(
            field_map.s_delta_mag,
            dtype=float,
        )

    if flow_kind == "scattered":

        return np.asarray(
            field_map.s_sca_like_mag,
            dtype=float,
        )

    raise ValueError(
        "flow_kind must be 'total', 'delta', or 'scattered'."
    )


def get_poynting_components(
    field_map,
    *,
    flow_kind,
):
    """
    Return the selected in-plane Poynting components.
    """

    if flow_kind == "total":

        U = np.asarray(
            field_map.plane_u_total,
            dtype=float,
        )

        V = np.asarray(
            field_map.plane_v_total,
            dtype=float,
        )

    elif flow_kind == "delta":

        U = np.asarray(
            field_map.plane_u_delta,
            dtype=float,
        )

        V = np.asarray(
            field_map.plane_v_delta,
            dtype=float,
        )

    elif flow_kind == "scattered":

        U = np.asarray(
            field_map.plane_u_sca_like,
            dtype=float,
        )

        V = np.asarray(
            field_map.plane_v_sca_like,
            dtype=float,
        )

    else:

        raise ValueError(
            "flow_kind must be 'total', 'delta', or 'scattered'."
        )

    return (
        U.copy(),
        V.copy(),
    )


def prepare_poynting_vectors(
    field_map,
    *,
    flow_kind,
    minimum_speed_percentile=0.5,
    normalize_vectors=False,
    vector_dynamic_range_decades=3.0,
):
    """
    Prepare robust in-plane Poynting components for streamplot.
    """

    U, V = get_poynting_components(
        field_map,
        flow_kind=flow_kind,
    )

    speed = np.hypot(
        U,
        V,
    )

    valid = (
        np.isfinite(U)
        & np.isfinite(V)
        & np.isfinite(speed)
        & (speed > 0.0)
    )

    if not np.any(valid):

        raise ValueError(
            "No finite, nonzero in-plane Poynting vectors."
        )

    if minimum_speed_percentile > 0.0:

        threshold = float(
            np.nanpercentile(
                speed[
                    valid
                ],
                minimum_speed_percentile,
            )
        )

        valid &= (
            speed >= threshold
        )

    if not np.any(valid):

        raise ValueError(
            "All Poynting vectors were removed by filtering."
        )

    robust_scale = float(
        np.nanpercentile(
            speed[
                valid
            ],
            99.0,
        )
    )

    if (
        not np.isfinite(robust_scale)
        or robust_scale <= 0.0
    ):

        robust_scale = float(
            np.nanmax(
                speed[
                    valid
                ]
            )
        )

    U_relative = (
        U / robust_scale
    )

    V_relative = (
        V / robust_scale
    )

    relative_speed = np.hypot(
        U_relative,
        V_relative,
    )

    usable = (
        valid
        & np.isfinite(
            relative_speed
        )
        & (
            relative_speed > 0.0
        )
    )

    U_plot = np.zeros_like(
        U_relative
    )

    V_plot = np.zeros_like(
        V_relative
    )

    if normalize_vectors:

        U_plot[
            usable
        ] = (
            U_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
        )

        V_plot[
            usable
        ] = (
            V_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
        )

    else:

        minimum_relative_speed = (
            10.0
            ** (
                -float(
                    vector_dynamic_range_decades
                )
            )
        )

        clipped_speed = np.clip(
            relative_speed,
            minimum_relative_speed,
            1.0,
        )

        compressed_speed = (
            (
                np.log10(
                    clipped_speed
                )
                - np.log10(
                    minimum_relative_speed
                )
            )
            / (
                -np.log10(
                    minimum_relative_speed
                )
            )
        )

        compressed_speed = (
            0.05
            + 0.95
            * compressed_speed
        )

        U_plot[
            usable
        ] = (
            U_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
            * compressed_speed[
                usable
            ]
        )

        V_plot[
            usable
        ] = (
            V_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
            * compressed_speed[
                usable
            ]
        )

    return (
        np.ma.array(
            U_plot,
            mask=~usable,
        ),
        np.ma.array(
            V_plot,
            mask=~usable,
        ),
        usable,
    )


# ============================================================
# STREAMLINE SEEDS
# ============================================================

def make_incident_edge_seeds(
    field_map,
    *,
    n_seeds=32,
    margin_grid_cells=2.0,
    start_fraction=0.04,
    end_fraction=0.96,
):
    """
    Place seeds along the incident-facing lower z boundary.
    """

    if field_map.plane != "xz":

        raise ValueError(
            "Incident-edge seeds require an XZ map."
        )

    x_nm = (
        field_map.coord1_m
        * 1e9
    )

    z_nm = (
        field_map.coord2_m
        * 1e9
    )

    dz_nm = abs(
        float(
            z_nm[1]
            - z_nm[0]
        )
    )

    seed_x = np.linspace(
        x_nm[0]
        + start_fraction
        * (
            x_nm[-1]
            - x_nm[0]
        ),
        x_nm[0]
        + end_fraction
        * (
            x_nm[-1]
            - x_nm[0]
        ),
        n_seeds,
    )

    seed_z = np.full_like(
        seed_x,
        z_nm[0]
        + margin_grid_cells
        * dz_nm,
    )

    return np.column_stack(
        [
            seed_x,
            seed_z,
        ]
    )


def make_incident_hemisphere_seeds(
    field_map,
    *,
    n_seeds=20,
    radial_offset_grid_cells=1.5,
    angular_margin_deg=8.0,
):
    """
    Place seeds just outside the incident-facing hemisphere.

    Incident propagation is +z. Therefore the incident-facing
    hemisphere is the lower half of the circle, z < 0.
    """

    if field_map.plane != "xz":

        raise ValueError(
            "Hemisphere seeds require an XZ map."
        )

    dx_nm = abs(
        float(
            field_map.coord1_m[1]
            - field_map.coord1_m[0]
        )
        * 1e9
    )

    dz_nm = abs(
        float(
            field_map.coord2_m[1]
            - field_map.coord2_m[0]
        )
        * 1e9
    )

    grid_spacing_nm = max(
        dx_nm,
        dz_nm,
    )

    radius_nm = (
        field_map.outer_radius_m
        * 1e9
        + radial_offset_grid_cells
        * grid_spacing_nm
    )

    # Lower hemisphere: 180 to 360 degrees.
    angles_deg = np.linspace(
        180.0
        + angular_margin_deg,
        360.0
        - angular_margin_deg,
        n_seeds,
    )

    angles_rad = np.deg2rad(
        angles_deg
    )

    return np.column_stack(
        [
            radius_nm
            * np.cos(
                angles_rad
            ),
            radius_nm
            * np.sin(
                angles_rad
            ),
        ]
    )


def make_full_surface_seeds(
    field_map,
    *,
    n_seeds=48,
    radial_offset_grid_cells=1.5,
):
    """
    Place seeds around the complete outer sphere surface.
    """

    dx_nm = abs(
        float(
            field_map.coord1_m[1]
            - field_map.coord1_m[0]
        )
        * 1e9
    )

    dz_nm = abs(
        float(
            field_map.coord2_m[1]
            - field_map.coord2_m[0]
        )
        * 1e9
    )

    radius_nm = (
        field_map.outer_radius_m
        * 1e9
        + radial_offset_grid_cells
        * max(
            dx_nm,
            dz_nm,
        )
    )

    angles = np.linspace(
        0.0,
        2.0 * np.pi,
        n_seeds,
        endpoint=False,
    )

    return np.column_stack(
        [
            radius_nm
            * np.cos(
                angles
            ),
            radius_nm
            * np.sin(
                angles
            ),
        ]
    )


def filter_streamline_seeds(
    field_map,
    seeds_nm,
    valid_vector_mask,
):
    """
    Retain seeds inside the domain and on valid vector cells.
    """

    seeds_nm = np.asarray(
        seeds_nm,
        dtype=float,
    )

    x_nm = (
        field_map.coord1_m
        * 1e9
    )

    z_nm = (
        field_map.coord2_m
        * 1e9
    )

    accepted = []

    for seed_x, seed_z in seeds_nm:

        if not (
            x_nm[1]
            < seed_x
            < x_nm[-2]
            and z_nm[1]
            < seed_z
            < z_nm[-2]
        ):

            continue

        ix = int(
            np.argmin(
                np.abs(
                    x_nm
                    - seed_x
                )
            )
        )

        iz = int(
            np.argmin(
                np.abs(
                    z_nm
                    - seed_z
                )
            )
        )

        if valid_vector_mask[
            iz,
            ix
        ]:

            accepted.append(
                (
                    seed_x,
                    seed_z,
                )
            )

    return np.asarray(
        accepted,
        dtype=float,
    )

In [ ]:
# ============================================================
# TOTAL ELECTRIC FIELD WITH SHARED COLOR LIMITS
#
# One figure per wavelength.
#
# Rows:
#   XY plane
#   XZ plane
#
# Columns:
#   Reference
#   Core absorber, constant diameter
#   Core absorber, constant shell number
#
# One common colorbar and common vmin/vmax across all six panels.
# ============================================================

ELECTRIC_FIELD_CLIP_LOW = 0.0
ELECTRIC_FIELD_CLIP_HIGH = 99.0

ELECTRIC_FIELD_FLOOR = 1e-30
ELECTRIC_FIELD_CMAP = "turbo"

ELECTRIC_FIELD_PLANES = [
    "xy",
    "xz",
]

ELECTRIC_FIELD_DISPLAY_EXTENT_FACTOR = 1.10

ELECTRIC_FIELD_FIGURE_WIDTH_CM = 18.5
ELECTRIC_FIELD_FIGURE_HEIGHT_CM = 11.2

ELECTRIC_FIELD_SHOW_SHELLS = True

SAVE_SHARED_ELECTRIC_FIELD_FIGURES = True


# ------------------------------------------------------------
# Optional color-limit mode
# ------------------------------------------------------------

# "per_wavelength":
#     one common scale for the six panels in each figure.
#
# "global":
#     one common scale for all wavelengths, all cases, all planes.
ELECTRIC_FIELD_LIMIT_MODE = "global"


def collect_electric_field_db(
    wavelength_nm,
):
    """
    Collect E-field dB arrays for all six panels at one wavelength.
    """

    arrays = {}

    for plane in ELECTRIC_FIELD_PLANES:

        for case_key in CLEAN_CASE_KEYS:

            field_map = clean_field_maps[
                (
                    case_key,
                    float(
                        wavelength_nm
                    ),
                    plane,
                )
            ]

            arrays[
                (
                    case_key,
                    plane,
                )
            ] = stable_field_db(
                field_map.e_total_mag,
                floor=ELECTRIC_FIELD_FLOOR,
            )

    return arrays


# ------------------------------------------------------------
# Calculate optional global limits
# ------------------------------------------------------------

if ELECTRIC_FIELD_LIMIT_MODE == "global":

    all_global_values = []

    for wavelength_nm in clean_wavelengths_nm:

        arrays = collect_electric_field_db(
            wavelength_nm
        )

        for array in arrays.values():

            finite = array[
                np.isfinite(array)
            ]

            if finite.size > 0:

                all_global_values.append(
                    finite
                )

    all_global_values = np.concatenate(
        all_global_values
    )

    global_electric_vmin = float(
        np.nanpercentile(
            all_global_values,
            ELECTRIC_FIELD_CLIP_LOW,
        )
    )

    global_electric_vmax = float(
        np.nanpercentile(
            all_global_values,
            ELECTRIC_FIELD_CLIP_HIGH,
        )
    )

    print(
        "Global electric-field limits:",
        global_electric_vmin,
        global_electric_vmax,
    )


cm_to_inches = 1.0 / 2.54


with plt.rc_context(
    {
        "figure.dpi": 150,
        "font.size": 7.5,
        "axes.titlesize": 8.5,
        "axes.labelsize": 7.5,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
):

    for wavelength_nm in clean_wavelengths_nm:

        electric_arrays = (
            collect_electric_field_db(
                wavelength_nm
            )
        )

        if (
            ELECTRIC_FIELD_LIMIT_MODE
            == "per_wavelength"
        ):

            pooled_values = np.concatenate(
                [
                    array[
                        np.isfinite(array)
                    ]
                    for array in (
                        electric_arrays.values()
                    )
                    if np.isfinite(array).any()
                ]
            )

            electric_vmin = float(
                np.nanpercentile(
                    pooled_values,
                    ELECTRIC_FIELD_CLIP_LOW,
                )
            )

            electric_vmax = float(
                np.nanpercentile(
                    pooled_values,
                    ELECTRIC_FIELD_CLIP_HIGH,
                )
            )

        elif (
            ELECTRIC_FIELD_LIMIT_MODE
            == "global"
        ):

            electric_vmin = (
                global_electric_vmin
            )

            electric_vmax = (
                global_electric_vmax
            )

        else:

            raise ValueError(
                "ELECTRIC_FIELD_LIMIT_MODE must be "
                "'per_wavelength' or 'global'."
            )

        print(
            f"{wavelength_nm:.0f} nm common E limits: "
            f"{electric_vmin:.2f} to {electric_vmax:.2f} dB"
        )

        fig, axes = plt.subplots(
            nrows=2,
            ncols=3,
            figsize=(
                ELECTRIC_FIELD_FIGURE_WIDTH_CM
                * cm_to_inches,
                ELECTRIC_FIELD_FIGURE_HEIGHT_CM
                * cm_to_inches,
            ),
            squeeze=False,
            constrained_layout=True,
        )

        fig.set_constrained_layout_pads(
            w_pad=0.025,
            h_pad=0.025,
            wspace=0.025,
            hspace=0.025,
        )

        shared_image = None

        for row_index, plane in enumerate(
            ELECTRIC_FIELD_PLANES
        ):

            for column_index, case_key in enumerate(
                CLEAN_CASE_KEYS
            ):

                ax = axes[
                    row_index,
                    column_index,
                ]

                field_map = clean_field_maps[
                    (
                        case_key,
                        float(
                            wavelength_nm
                        ),
                        plane,
                    )
                ]

                electric_db = (
                    electric_arrays[
                        (
                            case_key,
                            plane,
                        )
                    ]
                )

                extent_nm = [
                    float(
                        field_map.coord1_m[0]
                        * 1e9
                    ),
                    float(
                        field_map.coord1_m[-1]
                        * 1e9
                    ),
                    float(
                        field_map.coord2_m[0]
                        * 1e9
                    ),
                    float(
                        field_map.coord2_m[-1]
                        * 1e9
                    ),
                ]

                shared_image = ax.imshow(
                    electric_db,
                    origin="lower",
                    extent=extent_nm,
                    cmap=ELECTRIC_FIELD_CMAP,
                    vmin=electric_vmin,
                    vmax=electric_vmax,
                    interpolation="nearest",
                    aspect="equal",
                    rasterized=True,
                    zorder=0,
                )

                add_shell_and_sphere_boundaries(
                    ax,
                    field_map,
                    show_shells=(
                        ELECTRIC_FIELD_SHOW_SHELLS
                    ),
                )

                display_extent_nm = (
                    ELECTRIC_FIELD_DISPLAY_EXTENT_FACTOR
                    * field_map.outer_radius_m
                    * 1e9
                )

                ax.set_xlim(
                    -display_extent_nm,
                    display_extent_nm,
                )

                ax.set_ylim(
                    -display_extent_nm,
                    display_extent_nm,
                )

                ax.set_aspect(
                    "equal",
                    adjustable="box",
                )

                ax.xaxis.set_major_locator(
                    MaxNLocator(
                        nbins=3
                    )
                )

                ax.yaxis.set_major_locator(
                    MaxNLocator(
                        nbins=3
                    )
                )

                ax.minorticks_off()

                if row_index == 0:

                    ax.set_title(
                        CLEAN_CASE_LABELS[
                            case_key
                        ],
                        pad=4,
                        linespacing=0.9,
                    )

                    ax.tick_params(
                        axis="x",
                        labelbottom=False,
                    )

                else:

                    ax.set_xlabel(
                        "x [nm]"
                    )

                if column_index == 0:

                    second_axis_label = (
                        "y [nm]"
                        if plane == "xy"
                        else "z [nm]"
                    )

                    ax.set_ylabel(
                        (
                            plane.upper()
                            + " plane\n"
                            + second_axis_label
                        )
                    )

                else:

                    ax.tick_params(
                        axis="y",
                        labelleft=False,
                    )

        add_shared_colorbar(
            fig,
            axes,
            shared_image,
            label=r"$20\log_{10}|E|$ [dB]",
            tick_count=7,
            pad=0.025,
            fraction=0.030,
            shrink=0.98,
        )

        fig.suptitle(
            (
                rf"$\lambda="
                f"{wavelength_nm:.0f}"
                r"\,\mathrm{nm}$, "
                r"total electric-field magnitude"
            ),
            fontsize=10,
        )

        if SAVE_SHARED_ELECTRIC_FIELD_FIGURES:

            basename = (
                "electric_field_total_shared_scale_"
                f"{wavelength_nm:.0f}nm"
            )

            png_path = (
                CLEAN_OUTPUT_DIRECTORY
                / f"{basename}.png"
            )

            pdf_path = (
                CLEAN_OUTPUT_DIRECTORY
                / f"{basename}.pdf"
            )

            fig.savefig(
                png_path,
                dpi=CLEAN_PNG_DPI,
                facecolor="white",
                transparent=False,
                bbox_inches="tight",
                pad_inches=0.04,
            )

            fig.savefig(
                pdf_path,
                facecolor="white",
                transparent=False,
                bbox_inches="tight",
                pad_inches=0.04,
            )

            print("Saved:", png_path)
            print("Saved:", pdf_path)

        plt.show()

        plt.close(fig)

Cell 4: Robust Poynting helpers

This cell is independent of your experimental streamline functions. It incorporates the requested plotting settings but avoids overflow by using np.hypot() and rescaling the vector field before passing it to Matplotlib.

The background reproduces the requested 75th-percentile normalization. Values above the selected reference are saturated at 0 dB0\ \mathrm{dB}0 dB.

In [ ]:
# ============================================================
# CLEAN POYNTING-FLOW HELPERS
#
# Total flow:
#   seeds placed on the incident-facing lower z boundary.
#
# Delta flow:
#   seeds placed around the outer sphere surface.
# ============================================================

def prepare_clean_poynting_background(
    magnitude,
    *,
    reference_percentile=75.0,
    floor_fraction=1e-8,
):
    """
    Convert Poynting magnitude to relative dB.

    The chosen reference percentile is mapped to 0 dB.
    Larger values are saturated at 0 dB.
    """

    magnitude = np.asarray(
        magnitude,
        dtype=float,
    )

    valid = (
        np.isfinite(magnitude)
        & (magnitude > 0.0)
    )

    if not np.any(valid):

        raise ValueError(
            "No finite positive Poynting magnitudes are available."
        )

    reference_value = float(
        np.nanpercentile(
            magnitude[
                valid
            ],
            reference_percentile,
        )
    )

    if (
        not np.isfinite(
            reference_value
        )
        or reference_value <= 0.0
    ):

        raise ValueError(
            "The Poynting background reference is invalid."
        )

    floor_value = max(
        reference_value
        * floor_fraction,
        np.finfo(float).tiny,
    )

    background_db = np.full_like(
        magnitude,
        np.nan,
        dtype=float,
    )

    background_db[
        valid
    ] = (
        10.0
        * np.log10(
            np.maximum(
                magnitude[
                    valid
                ],
                floor_value,
            )
            / reference_value
        )
    )

    background_db = np.minimum(
        background_db,
        0.0,
    )

    return (
        background_db,
        -80.0,
        0.0,
        reference_value,
    )


def prepare_clean_poynting_vectors(
    field_map,
    *,
    flow_kind,
    minimum_speed_percentile=0.5,
    normalize_vectors=False,
    vector_dynamic_range_decades=3.0,
):
    """
    Select and robustly scale in-plane Poynting vectors.
    """

    if flow_kind == "total":

        U = np.asarray(
            field_map.plane_u_total,
            dtype=float,
        ).copy()

        V = np.asarray(
            field_map.plane_v_total,
            dtype=float,
        ).copy()

    elif flow_kind == "delta":

        U = np.asarray(
            field_map.plane_u_delta,
            dtype=float,
        ).copy()

        V = np.asarray(
            field_map.plane_v_delta,
            dtype=float,
        ).copy()

    else:

        raise ValueError(
            "flow_kind must be 'total' or 'delta'."
        )

    speed = np.hypot(
        U,
        V,
    )

    valid = (
        np.isfinite(U)
        & np.isfinite(V)
        & np.isfinite(speed)
        & (speed > 0.0)
    )

    if not np.any(valid):

        raise ValueError(
            "No finite nonzero in-plane Poynting vectors."
        )

    if minimum_speed_percentile > 0.0:

        threshold = float(
            np.nanpercentile(
                speed[
                    valid
                ],
                minimum_speed_percentile,
            )
        )

        valid &= (
            speed >= threshold
        )

    robust_scale = float(
        np.nanpercentile(
            speed[
                valid
            ],
            99.0,
        )
    )

    if (
        not np.isfinite(
            robust_scale
        )
        or robust_scale <= 0.0
    ):

        robust_scale = float(
            np.nanmax(
                speed[
                    valid
                ]
            )
        )

    U_relative = (
        U / robust_scale
    )

    V_relative = (
        V / robust_scale
    )

    relative_speed = np.hypot(
        U_relative,
        V_relative,
    )

    usable = (
        valid
        & np.isfinite(
            relative_speed
        )
        & (
            relative_speed > 0.0
        )
    )

    U_plot = np.zeros_like(
        U_relative
    )

    V_plot = np.zeros_like(
        V_relative
    )

    if normalize_vectors:

        U_plot[
            usable
        ] = (
            U_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
        )

        V_plot[
            usable
        ] = (
            V_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
        )

    else:

        minimum_relative_speed = (
            10.0
            ** (
                -float(
                    vector_dynamic_range_decades
                )
            )
        )

        clipped_speed = np.clip(
            relative_speed,
            minimum_relative_speed,
            1.0,
        )

        compressed_speed = (
            (
                np.log10(
                    clipped_speed
                )
                - np.log10(
                    minimum_relative_speed
                )
            )
            / (
                -np.log10(
                    minimum_relative_speed
                )
            )
        )

        compressed_speed = (
            0.05
            + 0.95
            * compressed_speed
        )

        U_plot[
            usable
        ] = (
            U_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
            * compressed_speed[
                usable
            ]
        )

        V_plot[
            usable
        ] = (
            V_relative[
                usable
            ]
            / relative_speed[
                usable
            ]
            * compressed_speed[
                usable
            ]
        )

    vector_mask = ~usable

    return (
        np.ma.array(
            U_plot,
            mask=vector_mask,
        ),
        np.ma.array(
            V_plot,
            mask=vector_mask,
        ),
        usable,
    )


def make_total_incident_seeds(
    field_map,
    *,
    n_seeds=50,
    margin_grid_cells=2.0,
    start_fraction=0.03,
    end_fraction=0.97,
):
    """
    Place seeds along the incident-facing lower z boundary.
    """

    if field_map.plane != "xz":

        raise ValueError(
            "Total-flow incident seeds are intended for the XZ plane."
        )

    x_min_nm = float(
        field_map.coord1_m[0]
        * 1e9
    )

    x_max_nm = float(
        field_map.coord1_m[-1]
        * 1e9
    )

    z_min_nm = float(
        field_map.coord2_m[0]
        * 1e9
    )

    dz_nm = abs(
        float(
            field_map.coord2_m[1]
            - field_map.coord2_m[0]
        )
        * 1e9
    )

    x_values = np.linspace(
        x_min_nm
        + start_fraction
        * (
            x_max_nm
            - x_min_nm
        ),
        x_min_nm
        + end_fraction
        * (
            x_max_nm
            - x_min_nm
        ),
        n_seeds,
    )

    z_value = (
        z_min_nm
        + margin_grid_cells
        * dz_nm
    )

    return np.column_stack(
        [
            x_values,
            np.full_like(
                x_values,
                z_value,
            ),
        ]
    )


def make_delta_surface_seeds(
    field_map,
    *,
    n_seeds=64,
    radial_offset_grid_cells=1.5,
):
    """
    Place delta-flow seeds just outside the outer sphere surface.
    """

    grid_spacing_nm = max(
        abs(
            float(
                field_map.coord1_m[1]
                - field_map.coord1_m[0]
            )
            * 1e9
        ),
        abs(
            float(
                field_map.coord2_m[1]
                - field_map.coord2_m[0]
            )
            * 1e9
        ),
    )

    seed_radius_nm = (
        field_map.outer_radius_m
        * 1e9
        + radial_offset_grid_cells
        * grid_spacing_nm
    )

    angles = np.linspace(
        0.0,
        2.0 * np.pi,
        n_seeds,
        endpoint=False,
    )

    return np.column_stack(
        [
            seed_radius_nm
            * np.cos(
                angles
            ),
            seed_radius_nm
            * np.sin(
                angles
            ),
        ]
    )


def filter_clean_streamline_seeds(
    field_map,
    seeds_nm,
    valid_vector_mask,
):
    """
    Remove seeds outside the box or on invalid vector cells.
    """

    seeds_nm = np.asarray(
        seeds_nm,
        dtype=float,
    )

    x_nm = (
        field_map.coord1_m
        * 1e9
    )

    z_nm = (
        field_map.coord2_m
        * 1e9
    )

    accepted = []

    for seed_x, seed_z in seeds_nm:

        if not (
            x_nm[1]
            < seed_x
            < x_nm[-2]
            and z_nm[1]
            < seed_z
            < z_nm[-2]
        ):

            continue

        x_index = int(
            np.argmin(
                np.abs(
                    x_nm
                    - seed_x
                )
            )
        )

        z_index = int(
            np.argmin(
                np.abs(
                    z_nm
                    - seed_z
                )
            )
        )

        if valid_vector_mask[
            z_index,
            x_index
        ]:

            accepted.append(
                (
                    seed_x,
                    seed_z,
                )
            )

    if not accepted:

        raise ValueError(
            "No valid streamline seeds remain after filtering."
        )

    return np.asarray(
        accepted,
        dtype=float,
    )


def add_clean_poynting_boundaries(
    ax,
    field_map,
):
    """
    Draw shell interfaces and emphasize the sphere boundary.
    """

    for radius_m in (
        field_map.radii_m[:-1]
    ):

        ax.add_patch(
            Circle(
                (
                    0.0,
                    0.0,
                ),
                radius=(
                    float(
                        radius_m
                    )
                    * 1e9
                ),
                fill=False,
                edgecolor="white",
                linewidth=0.16,
                alpha=0.18,
                zorder=6,
            )
        )

    ax.add_patch(
        Circle(
            (
                0.0,
                0.0,
            ),
            radius=(
                field_map.outer_radius_m
                * 1e9
            ),
            fill=False,
            edgecolor="white",
            linewidth=0.9,
            alpha=0.95,
            zorder=7,
        )
    )

Cell 5: Publication-ready total Poynting-flow figures

This produces one three-panel figure per wavelength.

It uses exactly the main settings you requested:

flow_kind="total"
background_kind="total"
plasma colormap
75th-percentile reference
cyan streamlines
bottom-edge seeds
forward integration
normalize_vectors=False
three decades compressed dynamic range

The internal robust scaling prevents multiplication overflow in streamplot.

In [ ]:
# ============================================================
# TOTAL POYNTING FLOW WITH SHARED COLORBAR
#
# XZ plane only.
#
# Seeds:
#   1. lower incident boundary;
#   2. incident-facing sphere hemisphere.
#
# One common S75 reference and one shared colorbar per wavelength.
# ============================================================

TOTAL_POYNTING_CMAP = "plasma"

TOTAL_POYNTING_FLOOR_FRACTION = 1e-8
TOTAL_POYNTING_REFERENCE_PERCENTILE = 75.0

TOTAL_POYNTING_STREAMLINE_COLOR = "#00E5FF"
TOTAL_POYNTING_STREAMLINE_WIDTH = 0.85

# Reduced from 3.0.
TOTAL_POYNTING_STREAMLINE_DENSITY = 1.0

# Reduced from 2.0.
TOTAL_POYNTING_ARROW_SIZE = 0.9

TOTAL_POYNTING_EDGE_SEED_COUNT = 28
TOTAL_POYNTING_HEMISPHERE_SEED_COUNT = 18

TOTAL_POYNTING_USE_HEMISPHERE_SEEDS = False

TOTAL_POYNTING_MINIMUM_SPEED_PERCENTILE = 0.5
TOTAL_POYNTING_NORMALIZE_VECTORS = False
TOTAL_POYNTING_DYNAMIC_RANGE_DECADES = 3.0

TOTAL_POYNTING_MINLENGTH = 0.025
TOTAL_POYNTING_MAXLENGTH = 10.0
TOTAL_POYNTING_BROKEN_STREAMLINES = True

TOTAL_POYNTING_DISPLAY_EXTENT_FACTOR = None

TOTAL_POYNTING_FIGURE_WIDTH_CM = 18.0
TOTAL_POYNTING_FIGURE_HEIGHT_CM = 6.3

SAVE_SHARED_TOTAL_POYNTING = True


cm_to_inches = 1.0 / 2.54


with plt.rc_context(
    {
        "figure.dpi": 150,
        "font.size": 7.5,
        "axes.titlesize": 8.5,
        "axes.labelsize": 7.5,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
):

    for wavelength_nm in clean_wavelengths_nm:

        field_maps_at_wavelength = {
            case_key: clean_field_maps[
                (
                    case_key,
                    float(
                        wavelength_nm
                    ),
                    "xz",
                )
            ]
            for case_key in CLEAN_CASE_KEYS
        }

        # ----------------------------------------------------
        # Common reference across all three cases
        # ----------------------------------------------------

        pooled_poynting_values = []

        for field_map in (
            field_maps_at_wavelength.values()
        ):

            magnitude = get_poynting_magnitude(
                field_map,
                flow_kind="total",
            )

            valid = (
                np.isfinite(magnitude)
                & (magnitude > 0.0)
            )

            pooled_poynting_values.append(
                magnitude[
                    valid
                ]
            )

        pooled_poynting_values = np.concatenate(
            pooled_poynting_values
        )

        common_reference_value = float(
            np.nanpercentile(
                pooled_poynting_values,
                TOTAL_POYNTING_REFERENCE_PERCENTILE,
            )
        )

        print(
            f"{wavelength_nm:.0f} nm common total S75:",
            f"{common_reference_value:.5g}",
        )

        fig, axes = plt.subplots(
            nrows=1,
            ncols=3,
            figsize=(
                TOTAL_POYNTING_FIGURE_WIDTH_CM
                * cm_to_inches,
                TOTAL_POYNTING_FIGURE_HEIGHT_CM
                * cm_to_inches,
            ),
            squeeze=False,
            constrained_layout=True,
        )

        fig.set_constrained_layout_pads(
            w_pad=0.03,
            h_pad=0.025,
            wspace=0.04,
            hspace=0.02,
        )

        shared_image = None

        for column_index, case_key in enumerate(
            CLEAN_CASE_KEYS
        ):

            ax = axes[
                0,
                column_index,
            ]

            field_map = (
                field_maps_at_wavelength[
                    case_key
                ]
            )

            magnitude = get_poynting_magnitude(
                field_map,
                flow_kind="total",
            )

            background_db = stable_relative_db(
                magnitude,
                common_reference_value,
                floor_fraction=(
                    TOTAL_POYNTING_FLOOR_FRACTION
                ),
                saturate_above_reference=True,
            )

            extent_nm = [
                float(
                    field_map.coord1_m[0]
                    * 1e9
                ),
                float(
                    field_map.coord1_m[-1]
                    * 1e9
                ),
                float(
                    field_map.coord2_m[0]
                    * 1e9
                ),
                float(
                    field_map.coord2_m[-1]
                    * 1e9
                ),
            ]

            shared_image = ax.imshow(
                background_db,
                origin="lower",
                extent=extent_nm,
                cmap=TOTAL_POYNTING_CMAP,
                vmin=-80.0,
                vmax=0.0,
                interpolation="nearest",
                aspect="equal",
                rasterized=True,
                zorder=0,
            )

            (
                U_plot,
                V_plot,
                valid_vectors,
            ) = prepare_poynting_vectors(
                field_map,
                flow_kind="total",
                minimum_speed_percentile=(
                    TOTAL_POYNTING_MINIMUM_SPEED_PERCENTILE
                ),
                normalize_vectors=(
                    TOTAL_POYNTING_NORMALIZE_VECTORS
                ),
                vector_dynamic_range_decades=(
                    TOTAL_POYNTING_DYNAMIC_RANGE_DECADES
                ),
            )

            edge_seeds = (
                make_incident_edge_seeds(
                    field_map,
                    n_seeds=(
                        TOTAL_POYNTING_EDGE_SEED_COUNT
                    ),
                    margin_grid_cells=2.0,
                    start_fraction=0.04,
                    end_fraction=0.96,
                )
            )

            seed_groups = [
                edge_seeds
            ]

            if (
                TOTAL_POYNTING_USE_HEMISPHERE_SEEDS
            ):

                hemisphere_seeds = (
                    make_incident_hemisphere_seeds(
                        field_map,
                        n_seeds=(
                            TOTAL_POYNTING_HEMISPHERE_SEED_COUNT
                        ),
                        radial_offset_grid_cells=1.5,
                        angular_margin_deg=8.0,
                    )
                )

                seed_groups.append(
                    hemisphere_seeds
                )

            combined_seeds = np.vstack(
                seed_groups
            )

            filtered_seeds = (
                filter_streamline_seeds(
                    field_map,
                    combined_seeds,
                    valid_vectors,
                )
            )

            if filtered_seeds.size == 0:

                warnings.warn(
                    f"No valid streamline seeds for {case_key}."
                )

            else:

                stream_arguments = {
                    "x": (
                        field_map.coord1_m
                        * 1e9
                    ),
                    "y": (
                        field_map.coord2_m
                        * 1e9
                    ),
                    "u": U_plot,
                    "v": V_plot,
                    "start_points": (
                        filtered_seeds
                    ),
                    "color": (
                        TOTAL_POYNTING_STREAMLINE_COLOR
                    ),
                    "linewidth": (
                        TOTAL_POYNTING_STREAMLINE_WIDTH
                    ),
                    "density": (
                        TOTAL_POYNTING_STREAMLINE_DENSITY
                    ),
                    "arrowsize": (
                        TOTAL_POYNTING_ARROW_SIZE
                    ),
                    "arrowstyle": "->",
                    "integration_direction": "forward",
                    "minlength": (
                        TOTAL_POYNTING_MINLENGTH
                    ),
                    "maxlength": (
                        TOTAL_POYNTING_MAXLENGTH
                    ),
                    "broken_streamlines": (
                        TOTAL_POYNTING_BROKEN_STREAMLINES
                    ),
                    "zorder": 5,
                }

                try:

                    ax.streamplot(
                        **stream_arguments
                    )

                except IndexError:

                    warnings.warn(
                        "A zero-length streamline was found. "
                        "Retrying without hemisphere seeds."
                    )

                    filtered_edge_seeds = (
                        filter_streamline_seeds(
                            field_map,
                            edge_seeds,
                            valid_vectors,
                        )
                    )

                    stream_arguments[
                        "start_points"
                    ] = filtered_edge_seeds

                    stream_arguments[
                        "broken_streamlines"
                    ] = True

                    ax.streamplot(
                        **stream_arguments
                    )

            add_shell_and_sphere_boundaries(
                ax,
                field_map,
                show_shells=True,
                shell_linewidth=0.16,
                shell_alpha=0.18,
                sphere_linewidth=0.95,
            )

            if (
                TOTAL_POYNTING_DISPLAY_EXTENT_FACTOR
                is not None
            ):

                display_extent_nm = (
                    TOTAL_POYNTING_DISPLAY_EXTENT_FACTOR
                    * field_map.outer_radius_m
                    * 1e9
                )

                ax.set_xlim(
                    -display_extent_nm,
                    display_extent_nm,
                )

                ax.set_ylim(
                    -display_extent_nm,
                    display_extent_nm,
                )

            ax.set_aspect(
                "equal",
                adjustable="box",
            )

            ax.set_title(
                CLEAN_CASE_LABELS[
                    case_key
                ],
                pad=4,
                linespacing=0.9,
            )

            ax.set_xlabel(
                "x [nm]"
            )

            if column_index == 0:

                ax.set_ylabel(
                    "z [nm]"
                )

            else:

                ax.tick_params(
                    axis="y",
                    labelleft=False,
                )

            ax.xaxis.set_major_locator(
                MaxNLocator(
                    nbins=3
                )
            )

            ax.yaxis.set_major_locator(
                MaxNLocator(
                    nbins=3
                )
            )

        add_shared_colorbar(
            fig,
            axes,
            shared_image,
            label=(
                r"$10\log_{10}"
                r"(|\mathbf{S}|/S_{75,\mathrm{common}})$ [dB]"
            ),
            tick_count=5,
            pad=0.025,
            fraction=0.028,
            shrink=0.98,
        )

        fig.suptitle(
            (
                rf"$\lambda="
                f"{wavelength_nm:.0f}"
                r"\,\mathrm{nm}$, "
                r"total Poynting flow, XZ plane"
            ),
            fontsize=10,
        )

        if SAVE_SHARED_TOTAL_POYNTING:

            basename = (
                "poynting_total_shared_scale_"
                f"{wavelength_nm:.0f}nm_"
                "xz"
            )

            png_path = (
                CLEAN_OUTPUT_DIRECTORY
                / f"{basename}.png"
            )

            pdf_path = (
                CLEAN_OUTPUT_DIRECTORY
                / f"{basename}.pdf"
            )

            fig.savefig(
                png_path,
                dpi=CLEAN_PNG_DPI,
                facecolor="white",
                transparent=False,
                bbox_inches="tight",
                pad_inches=0.04,
            )

            fig.savefig(
                pdf_path,
                facecolor="white",
                transparent=False,
                bbox_inches="tight",
                pad_inches=0.04,
            )

            print("Saved:", png_path)
            print("Saved:", pdf_path)

        plt.show()

        plt.close(fig)

Cell 6: Experimental delta Poynting-flow figures

This uses seeds just outside the outer sphere surface and traces in both directions, as requested.

The settings are deliberately separated from the total-flow settings so you can optimize them later without affecting the total-flow figures.

In [ ]:
# ============================================================
# EXPERIMENTAL DELTA POYNTING FLOW
#
# XZ plane only.
#
# Seeds:
#   distributed just outside the sphere surface.
#
# Integration:
#   both directions.
# ============================================================

DELTA_POYNTING_SETTINGS = {
    "background_floor_fraction": 1e-8,
    "background_reference_percentile": 75.0,
    "background_cmap": "plasma",

    "streamline_color": "#00E5FF",
    "streamline_linewidth": 1.2,
    "streamline_density": 3.0,
    "streamline_arrowsize": 2.0,

    "integration_direction": "both",
    "minlength": 0.025,
    "maxlength": 12.5,
    "broken_streamlines": False,

    "normalize_vectors": False,
    "vector_dynamic_range_decades": 3.0,
    "minimum_speed_percentile": 0.5,

    "seed_count": 64,
}

SAVE_DELTA_POYNTING_FIGURES = True


cm_to_inches = 1.0 / 2.54


with plt.rc_context(
    {
        "figure.dpi": 150,
        "font.size": 7.5,
        "axes.titlesize": 8.5,
        "axes.labelsize": 7.5,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
):

    for wavelength_nm in (
        clean_wavelengths_nm
    ):

        fig, axes = plt.subplots(
            nrows=1,
            ncols=3,
            figsize=(
                POYNTING_FIGURE_WIDTH_CM
                * cm_to_inches,
                POYNTING_FIGURE_HEIGHT_CM
                * cm_to_inches,
            ),
            squeeze=False,
            constrained_layout=True,
        )

        fig.set_constrained_layout_pads(
            w_pad=0.025,
            h_pad=0.025,
            wspace=0.025,
            hspace=0.025,
        )

        diagnostic_records = []

        for column_index, case_key in enumerate(
            CLEAN_CASE_KEYS
        ):

            ax = axes[
                0,
                column_index,
            ]

            field_map = clean_field_maps[
                (
                    case_key,
                    float(
                        wavelength_nm
                    ),
                    "xz",
                )
            ]

            seed_points_nm = (
                make_delta_surface_seeds(
                    field_map,
                    n_seeds=(
                        DELTA_POYNTING_SETTINGS[
                            "seed_count"
                        ]
                    ),
                    radial_offset_grid_cells=1.5,
                )
            )

            diagnostics = (
                plot_clean_poynting_panel(
                    fig,
                    ax,
                    field_map,
                    flow_kind="delta",
                    seed_points_nm=(
                        seed_points_nm
                    ),
                    settings=(
                        DELTA_POYNTING_SETTINGS
                    ),
                )
            )

            diagnostics.update(
                {
                    "case": case_key,
                    "wavelength_nm": (
                        wavelength_nm
                    ),
                }
            )

            diagnostic_records.append(
                diagnostics
            )

            ax.set_title(
                CLEAN_CASE_LABELS[
                    case_key
                ],
                pad=4,
                linespacing=0.9,
            )

            ax.xaxis.set_major_locator(
                MaxNLocator(
                    nbins=3
                )
            )

            ax.yaxis.set_major_locator(
                MaxNLocator(
                    nbins=3
                )
            )

            if column_index > 0:

                ax.tick_params(
                    axis="y",
                    labelleft=False,
                )

                ax.set_ylabel("")

        fig.suptitle(
            (
                rf"$\lambda="
                f"{wavelength_nm:.0f}"
                r"\,\mathrm{nm}$, "
                r"incident-subtracted Poynting flow, XZ plane"
            ),
            fontsize=10,
        )

        display(
            pd.DataFrame(
                diagnostic_records
            )
        )

        if SAVE_DELTA_POYNTING_FIGURES:

            basename = (
                "poynting_delta_"
                f"{wavelength_nm:.0f}nm_"
                "xz"
            )

            png_path = (
                CLEAN_OUTPUT_DIRECTORY
                / f"{basename}.png"
            )

            pdf_path = (
                CLEAN_OUTPUT_DIRECTORY
                / f"{basename}.pdf"
            )

            fig.savefig(
                png_path,
                dpi=CLEAN_PNG_DPI,
                facecolor="white",
                transparent=False,
                bbox_inches="tight",
                pad_inches=0.03,
            )

            fig.savefig(
                pdf_path,
                facecolor="white",
                transparent=False,
                bbox_inches="tight",
                pad_inches=0.03,
            )

            print("Saved:", png_path)
            print("Saved:", pdf_path)

        plt.show()

        plt.close(fig)

In [ ]:
# ============================================================
# SCATTERED-LIKE POYNTING FLOW TEST
#
# Uses:
#   s_sca_like_mag
#   plane_u_sca_like
#   plane_v_sca_like
#
# Seeds:
#   complete outer sphere surface
#
# Integration:
#   both directions
# ============================================================

SCATTERED_TEST_WAVELENGTH_INDEX = 1

SCATTERED_TEST_WAVELENGTH_NM = float(
    clean_wavelengths_nm[
        SCATTERED_TEST_WAVELENGTH_INDEX
    ]
)

SCATTERED_BACKGROUND_CMAP = "plasma"
SCATTERED_REFERENCE_PERCENTILE = 75.0
SCATTERED_FLOOR_FRACTION = 1e-8

SCATTERED_STREAMLINE_COLOR = "#00E5FF"
SCATTERED_STREAMLINE_WIDTH = 0.75
SCATTERED_STREAMLINE_DENSITY = 1.0
SCATTERED_ARROW_SIZE = 0.9

SCATTERED_SURFACE_SEED_COUNT = 44

SCATTERED_MINIMUM_SPEED_PERCENTILE = 0.5
SCATTERED_NORMALIZE_VECTORS = False
SCATTERED_DYNAMIC_RANGE_DECADES = 3.0

SCATTERED_MAXLENGTH = 8.0

SAVE_SCATTERED_TEST = True


field_maps_scattered = {
    case_key: clean_field_maps[
        (
            case_key,
            SCATTERED_TEST_WAVELENGTH_NM,
            "xz",
        )
    ]
    for case_key in CLEAN_CASE_KEYS
}


# Common scattered-like reference across the three cases.
pooled_scattered_values = []

for field_map in (
    field_maps_scattered.values()
):

    magnitude = get_poynting_magnitude(
        field_map,
        flow_kind="scattered",
    )

    valid = (
        np.isfinite(magnitude)
        & (magnitude > 0.0)
    )

    pooled_scattered_values.append(
        magnitude[
            valid
        ]
    )


pooled_scattered_values = np.concatenate(
    pooled_scattered_values
)

common_scattered_reference = float(
    np.nanpercentile(
        pooled_scattered_values,
        SCATTERED_REFERENCE_PERCENTILE,
    )
)


cm_to_inches = 1.0 / 2.54


fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(
        18.0
        * cm_to_inches,
        6.3
        * cm_to_inches,
    ),
    squeeze=False,
    constrained_layout=True,
)

fig.set_constrained_layout_pads(
    w_pad=0.03,
    h_pad=0.025,
    wspace=0.04,
    hspace=0.02,
)

shared_image = None


for column_index, case_key in enumerate(
    CLEAN_CASE_KEYS
):

    ax = axes[
        0,
        column_index,
    ]

    field_map = (
        field_maps_scattered[
            case_key
        ]
    )

    background_db = stable_relative_db(
        field_map.s_sca_like_mag,
        common_scattered_reference,
        floor_fraction=(
            SCATTERED_FLOOR_FRACTION
        ),
        saturate_above_reference=True,
    )

    extent_nm = [
        float(
            field_map.coord1_m[0]
            * 1e9
        ),
        float(
            field_map.coord1_m[-1]
            * 1e9
        ),
        float(
            field_map.coord2_m[0]
            * 1e9
        ),
        float(
            field_map.coord2_m[-1]
            * 1e9
        ),
    ]

    shared_image = ax.imshow(
        background_db,
        origin="lower",
        extent=extent_nm,
        cmap=SCATTERED_BACKGROUND_CMAP,
        vmin=-80.0,
        vmax=0.0,
        interpolation="nearest",
        aspect="equal",
        rasterized=True,
        zorder=0,
    )

    (
        U_plot,
        V_plot,
        valid_vectors,
    ) = prepare_poynting_vectors(
        field_map,
        flow_kind="scattered",
        minimum_speed_percentile=(
            SCATTERED_MINIMUM_SPEED_PERCENTILE
        ),
        normalize_vectors=(
            SCATTERED_NORMALIZE_VECTORS
        ),
        vector_dynamic_range_decades=(
            SCATTERED_DYNAMIC_RANGE_DECADES
        ),
    )

    surface_seeds = make_full_surface_seeds(
        field_map,
        n_seeds=(
            SCATTERED_SURFACE_SEED_COUNT
        ),
        radial_offset_grid_cells=1.5,
    )

    filtered_seeds = filter_streamline_seeds(
        field_map,
        surface_seeds,
        valid_vectors,
    )

    if filtered_seeds.size > 0:

        try:

            ax.streamplot(
                field_map.coord1_m
                * 1e9,
                field_map.coord2_m
                * 1e9,
                U_plot,
                V_plot,
                start_points=filtered_seeds,
                color=(
                    SCATTERED_STREAMLINE_COLOR
                ),
                linewidth=(
                    SCATTERED_STREAMLINE_WIDTH
                ),
                density=(
                    SCATTERED_STREAMLINE_DENSITY
                ),
                arrowsize=(
                    SCATTERED_ARROW_SIZE
                ),
                arrowstyle="->",
                integration_direction="both",
                minlength=0.025,
                maxlength=(
                    SCATTERED_MAXLENGTH
                ),
                broken_streamlines=True,
                zorder=5,
            )

        except IndexError:

            warnings.warn(
                f"Zero-length streamline encountered for {case_key}. "
                "The panel is shown without streamlines."
            )

    add_shell_and_sphere_boundaries(
        ax,
        field_map,
        show_shells=True,
        shell_linewidth=0.16,
        shell_alpha=0.18,
        sphere_linewidth=0.95,
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    ax.set_title(
        CLEAN_CASE_LABELS[
            case_key
        ],
        pad=4,
        linespacing=0.9,
    )

    ax.set_xlabel(
        "x [nm]"
    )

    if column_index == 0:

        ax.set_ylabel(
            "z [nm]"
        )

    else:

        ax.tick_params(
            axis="y",
            labelleft=False,
        )

    ax.xaxis.set_major_locator(
        MaxNLocator(
            nbins=3
        )
    )

    ax.yaxis.set_major_locator(
        MaxNLocator(
            nbins=3
        )
    )


add_shared_colorbar(
    fig,
    axes,
    shared_image,
    label=(
        r"$10\log_{10}"
        r"(|\mathbf{S}_{\mathrm{sca}}|/"
        r"S_{\mathrm{sca},75,\mathrm{common}})$ [dB]"
    ),
    tick_count=5,
    pad=0.025,
    fraction=0.028,
    shrink=0.98,
)


fig.suptitle(
    (
        rf"$\lambda="
        f"{SCATTERED_TEST_WAVELENGTH_NM:.0f}"
        r"\,\mathrm{nm}$, "
        r"scattered-like Poynting flow, XZ plane"
    ),
    fontsize=10,
)


if SAVE_SCATTERED_TEST:

    basename = (
        "poynting_scattered_like_test_"
        f"{SCATTERED_TEST_WAVELENGTH_NM:.0f}nm_"
        "xz"
    )

    png_path = (
        CLEAN_OUTPUT_DIRECTORY
        / f"{basename}.png"
    )

    pdf_path = (
        CLEAN_OUTPUT_DIRECTORY
        / f"{basename}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=CLEAN_PNG_DPI,
        facecolor="white",
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.04,
    )

    fig.savefig(
        pdf_path,
        facecolor="white",
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.04,
    )

    print("Saved:", png_path)
    print("Saved:", pdf_path)


plt.show()

plt.close(fig)